In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("ProyectoEmergenciasLaLibertad")
    .master("yarn")
    .getOrCreate()
)

print("Spark master:", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

In [ ]:
df_prueba_yarn = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/dataset_maestro_2003_2025"
)

print("Filas:", df_prueba_yarn.count())
print("Particiones:", df_prueba_yarn.rdd.getNumPartitions())

In [ ]:
df_2025_yarn = (
    df_prueba_yarn
    .filter(F.col("anio") == 2025)
)

print("Filas 2025:", df_2025_yarn.count())

df_2025_yarn.explain("formatted")

In [ ]:
print(
    "Particiones originales:",
    df_prueba_yarn.rdd.getNumPartitions()
)

In [ ]:
df_repartitionado = (
    df_prueba_yarn
    .repartition(8, "anio")
)

print(
    "Después de repartition:",
    df_repartitionado.rdd.getNumPartitions()
)

In [ ]:
(
    df_repartitionado
    .withColumn(
        "particion_spark",
        F.spark_partition_id()
    )
    .groupBy("particion_spark")
    .count()
    .orderBy("particion_spark")
    .show()
)

In [ ]:
df_coalesce = df_repartitionado.coalesce(2)

print(
    "Después de coalesce:",
    df_coalesce.rdd.getNumPartitions()
)

In [ ]:
(
    df_coalesce
    .withColumn(
        "particion_spark",
        F.spark_partition_id()
    )
    .groupBy("particion_spark")
    .count()
    .orderBy("particion_spark")
    .show()
)

In [ ]:
df_repartition_balanceado = (
    df_prueba_yarn
    .repartition(8)
)

print(
    "Particiones:",
    df_repartition_balanceado.rdd.getNumPartitions()
)

(
    df_repartition_balanceado
    .withColumn(
        "particion_spark",
        F.spark_partition_id()
    )
    .groupBy("particion_spark")
    .count()
    .orderBy("particion_spark")
    .show()
)

In [ ]:
def escalar_dataset(df, factor):
    replicas = spark.range(factor).select(
        F.col("id").alias("replica")
    )

    return df.crossJoin(replicas)

In [ ]:
df_1x = df_prueba_yarn
df_10x = escalar_dataset(df_prueba_yarn, 10)
df_100x = escalar_dataset(df_prueba_yarn, 100)

print("1x:", df_1x.count())
print("10x:", df_10x.count())
print("100x:", df_100x.count())

In [ ]:
import time

def medir_proceso(df, nombre, particiones):
    
    df_test = df.repartition(particiones)

    inicio = time.perf_counter()

    resultado = (
        df_test
        .groupBy("anio", "provincia")
        .agg(
            F.count("*").alias("emergencias"),
            F.sum("afectados").alias("afectados")
        )
        .count()
    )

    fin = time.perf_counter()

    tiempo = fin - inicio

    print(
        f"{nombre} | "
        f"particiones={particiones} | "
        f"tiempo={tiempo:.3f}s | "
        f"filas resultado={resultado}"
    )

    return tiempo

In [ ]:
medir_proceso(df_1x, "1x", 2)
medir_proceso(df_10x, "10x", 2)
medir_proceso(df_100x, "100x", 2)

In [ ]:
import time
import statistics

def benchmark(df, nombre, particiones, repeticiones=6):

    tiempos = []

    for i in range(repeticiones):

        df_test = df.repartition(particiones)

        inicio = time.perf_counter()

        resultado = (
            df_test
            .groupBy("anio", "provincia")
            .agg(
                F.count("*").alias("emergencias"),
                F.sum("afectados").alias("afectados")
            )
            .count()
        )

        fin = time.perf_counter()

        tiempo = fin - inicio
        tiempos.append(tiempo)

        print(
            f"{nombre} | ejecución {i+1} | "
            f"{tiempo:.3f}s"
        )

    # descartamos primera ejecución
    tiempos_validos = tiempos[1:]

    mediana = statistics.median(tiempos_validos)
    promedio = statistics.mean(tiempos_validos)

    print(
        f"\n{nombre} | "
        f"mediana={mediana:.3f}s | "
        f"promedio={promedio:.3f}s\n"
    )

    return mediana

In [ ]:
t_1x = benchmark(df_1x, "1x", 2)
t_10x = benchmark(df_10x, "10x", 2)
t_100x = benchmark(df_100x, "100x", 2)

In [ ]:
t100_p1 = benchmark(df_100x, "100x - 1 particion", 1)
t100_p2 = benchmark(df_100x, "100x - 2 particiones", 2)
t100_p4 = benchmark(df_100x, "100x - 4 particiones", 4)
t100_p8 = benchmark(df_100x, "100x - 8 particiones", 8)

In [ ]:
print("100x - 1 partición :", round(t100_p1, 3), "s")
print("100x - 2 particiones:", round(t100_p2, 3), "s")
print("100x - 4 particiones:", round(t100_p4, 3), "s")
print("100x - 8 particiones:", round(t100_p8, 3), "s")

In [ ]:
import time
import statistics

ruta_csv_nacional = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/raw/sinpad/"
    "BD_2003-2025_EMERGENCIAS.csv"
)

ruta_parquet_nacional = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/"
    "emergencias_nacional"
)

In [ ]:
def benchmark_lectura(tipo, repeticiones=6):

    tiempos = []

    for i in range(repeticiones):

        inicio = time.perf_counter()

        if tipo == "CSV":
            df_test = (
                spark.read
                .option("header", True)
                .option("inferSchema", True)
                .option("sep", ";")
                .option("encoding", "ISO-8859-1")
                .csv(ruta_csv_nacional)
            )

        elif tipo == "PARQUET":
            df_test = spark.read.parquet(
                ruta_parquet_nacional
            )

        filas = df_test.count()

        fin = time.perf_counter()

        tiempo = fin - inicio
        tiempos.append(tiempo)

        print(
            f"{tipo} | ejecución {i+1} | "
            f"{tiempo:.3f}s | filas={filas}"
        )

    validos = tiempos[1:]

    mediana = statistics.median(validos)
    promedio = statistics.mean(validos)

    print(
        f"\n{tipo} | "
        f"mediana={mediana:.3f}s | "
        f"promedio={promedio:.3f}s\n"
    )

    return mediana

In [ ]:
t_csv = benchmark_lectura("CSV")
t_parquet = benchmark_lectura("PARQUET")

print("CSV:", round(t_csv, 3), "s")
print("Parquet:", round(t_parquet, 3), "s")

In [ ]:
df_csv_temp = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .csv(ruta_csv_nacional)
)

print(df_csv_temp.columns)

In [ ]:
import time
import statistics

def benchmark_2025(tipo, repeticiones=6):

    tiempos = []
    filas_finales = None

    for i in range(repeticiones):

        inicio = time.perf_counter()

        if tipo == "CSV":
            df_test = (
                spark.read
                .option("header", True)
                .option("inferSchema", True)
                .option("sep", ";")
                .option("encoding", "ISO-8859-1")
                .csv(ruta_csv_nacional)
                .filter(F.col("AÑO") == 2025)
            )

        elif tipo == "PARQUET":
            df_test = (
                spark.read
                .parquet(ruta_parquet_nacional)
                .filter(F.col("anio") == 2025)
            )

        filas_finales = df_test.count()

        fin = time.perf_counter()

        tiempo = fin - inicio
        tiempos.append(tiempo)

        print(
            f"{tipo} 2025 | ejecución {i+1} | "
            f"{tiempo:.3f}s | filas={filas_finales}"
        )

    # descartamos calentamiento
    validos = tiempos[1:]

    mediana = statistics.median(validos)
    promedio = statistics.mean(validos)

    print(
        f"\n{tipo} 2025 | "
        f"mediana={mediana:.3f}s | "
        f"promedio={promedio:.3f}s | "
        f"filas={filas_finales}\n"
    )

    return mediana

In [ ]:
t_csv_2025 = benchmark_2025("CSV")
t_parquet_2025 = benchmark_2025("PARQUET")

print("CSV 2025:", round(t_csv_2025, 3), "s")
print("Parquet 2025:", round(t_parquet_2025, 3), "s")

print(
    "Mejora aproximada:",
    round(t_csv_2025 / t_parquet_2025, 2),
    "x"
)

In [ ]:
df_csv_plan = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .csv(ruta_csv_nacional)
    .filter(F.col("AÑO") == 2025)
)

df_csv_plan.explain("formatted")

In [ ]:
df_parquet_plan = (
    spark.read
    .parquet(ruta_parquet_nacional)
    .filter(F.col("anio") == 2025)
)

df_parquet_plan.explain("formatted")

In [ ]:
rdd_peligros = (
    df_prueba_yarn
    .select("peligro")
    .rdd
    .filter(lambda fila: fila["peligro"] is not None)
    .map(lambda fila: (fila["peligro"], 1))
    .reduceByKey(lambda a, b: a + b)
)

top_rdd = (
    rdd_peligros
    .takeOrdered(
        10,
        key=lambda x: -x[1]
    )
)

for peligro, cantidad in top_rdd:
    print(peligro, cantidad)

In [ ]:
(
    df_prueba_yarn
    .groupBy("peligro")
    .count()
    .orderBy(F.desc("count"))
    .show(10, truncate=False)
)

In [ ]:
import time
import statistics

def benchmark_rdd(df, repeticiones=6):
    tiempos = []

    for i in range(repeticiones):
        inicio = time.perf_counter()

        resultado = (
            df.select("peligro")
            .rdd
            .filter(lambda x: x["peligro"] is not None)
            .map(lambda x: (x["peligro"], 1))
            .reduceByKey(lambda a, b: a + b)
            .collect()
        )

        tiempo = time.perf_counter() - inicio
        tiempos.append(tiempo)

        print(f"RDD {i+1}: {tiempo:.3f}s")

    validos = tiempos[1:]
    mediana = statistics.median(validos)

    print("Mediana RDD:", round(mediana, 3), "s")
    return mediana

In [ ]:
def benchmark_dataframe(df, repeticiones=6):
    tiempos = []

    for i in range(repeticiones):
        inicio = time.perf_counter()

        resultado = (
            df
            .filter(F.col("peligro").isNotNull())
            .groupBy("peligro")
            .count()
            .collect()
        )

        tiempo = time.perf_counter() - inicio
        tiempos.append(tiempo)

        print(f"DataFrame {i+1}: {tiempo:.3f}s")

    validos = tiempos[1:]
    mediana = statistics.median(validos)

    print("Mediana DataFrame:", round(mediana, 3), "s")
    return mediana

In [ ]:
t_rdd = benchmark_rdd(df_100x)
t_df = benchmark_dataframe(df_100x)

print("RDD:", round(t_rdd, 3), "s")
print("DataFrame:", round(t_df, 3), "s")

In [ ]:
import time
from pyspark.sql import functions as F

ruta_maestro = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/"
    "dataset_maestro_2003_2025"
)

def pipeline_escalabilidad(factor):

    ruta_salida = (
        "hdfs://localhost:9000/"
        f"proyecto_emergencias/benchmark/pipeline_{factor}x"
    )

    inicio = time.perf_counter()

    # 1. Lectura desde HDFS
    df_base = spark.read.parquet(ruta_maestro)

    # 2. Replicación solo para prueba de escalabilidad
    replicas = (
        spark.range(factor)
        .select(F.col("id").alias("replica"))
    )

    df_escalado = df_base.crossJoin(replicas)

    # 3. Transformaciones
    df_transformado = (
        df_escalado
        .withColumn(
            "mes_num",
            F.month("fecha_emergencia")
        )
        .withColumn(
            "personas_impactadas",
            F.coalesce(F.col("afectados"), F.lit(0))
            + F.coalesce(F.col("damnificados"), F.lit(0))
        )
        .withColumn(
            "viviendas_impactadas",
            F.coalesce(F.col("viviendas_afectadas"), F.lit(0))
            + F.coalesce(F.col("viviendas_destruidas"), F.lit(0))
        )
    )

    # 4. Agregación analítica
    df_resultado = (
        df_transformado
        .groupBy(
            "anio",
            "provincia",
            "distrito",
            "mes_num",
            "peligro"
        )
        .agg(
            F.count("*").alias("cantidad_emergencias"),
            F.sum("personas_impactadas").alias("personas_impactadas"),
            F.sum("viviendas_impactadas").alias("viviendas_impactadas"),
            F.avg("precipitacion_total_mm").alias("precipitacion_promedio_mm")
        )
    )

    # 5. Escritura procesada en HDFS / Parquet
    (
        df_resultado
        .write
        .mode("overwrite")
        .partitionBy("anio")
        .parquet(ruta_salida)
    )

    fin = time.perf_counter()

    tiempo = fin - inicio

    # Verificación fuera del cronómetro
    filas_salida = (
        spark.read.parquet(ruta_salida).count()
    )

    print(
        f"{factor}x | "
        f"tiempo={tiempo:.3f}s | "
        f"filas_salida={filas_salida}"
    )

    return tiempo

In [ ]:
t_pipeline_1x = pipeline_escalabilidad(1)

In [ ]:
t_pipeline_10x = pipeline_escalabilidad(10)
t_pipeline_100x = pipeline_escalabilidad(100)

print("\nRESUMEN")
print("1x   :", round(t_pipeline_1x, 3), "s")
print("10x  :", round(t_pipeline_10x, 3), "s")
print("100x :", round(t_pipeline_100x, 3), "s")

In [ ]:
ruta_sinpad = "hdfs:///proyecto_emergencias/raw/sinpad/BD_2003-2025_EMERGENCIAS.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "ISO-8859-1")
    .csv(ruta_sinpad)
)

print("Filas:", df.count())
print("Columnas:", len(df.columns))

In [ ]:
print(repr(ruta_sinpad))

In [ ]:
ruta_sinpad = "hdfs://localhost:9000/proyecto_emergencias/raw/sinpad/BD_2003-2025_EMERGENCIAS.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .csv(ruta_sinpad)
)

print("Filas:", df.count())
print("Columnas:", len(df.columns))

In [ ]:
df.printSchema()

In [ ]:
df.show(5, truncate=False)

In [ ]:
from pyspark.sql import functions as F

total = df.count()

la_libertad = df.filter(
    F.upper(F.trim(F.col("`DPTO.`"))) == "LA LIBERTAD"
).count()

sinpad_nulos = df.filter(
    F.col("CODIGO DE EMERGENCIA-SINPAD").isNull()
).count()

sinpad_unicos = (
    df.select("CODIGO DE EMERGENCIA-SINPAD")
    .distinct()
    .count()
)

print("Total nacional:", total)
print("Registros de La Libertad:", la_libertad)
print("SINPAD nulos:", sinpad_nulos)
print("SINPAD únicos:", sinpad_unicos)

In [ ]:
duplicados_sinpad = (
    df.groupBy("CODIGO DE EMERGENCIA-SINPAD")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

duplicados_sinpad.show(truncate=False)

In [ ]:
duplicados_completos = df.count() - df.dropDuplicates().count()

print("Filas completamente duplicadas:", duplicados_completos)

In [ ]:
columnas_clave = [
    "FECHA  DE LA EMER",
    "`DPTO.`",
    "`PROV.`",
    "`DIST.`",
    "PELIGRO",
    "FALLECIDOS",
    "LESIONADOS",
    "DAMNIFICADOS",
    "AFECTADOS"
]

for col_name in columnas_clave:
    cantidad = df.filter(F.col(col_name).isNull()).count()
    print(f"{col_name}: {cantidad} nulos")

In [ ]:
codigos_repetidos = [
    97558,
    97553,
    97549,
    97561,
    153378,
    155509,
    154908
]

df.filter(
    F.col("CODIGO DE EMERGENCIA-SINPAD").isin(codigos_repetidos)
).select(
    "CODIGO DE EMERGENCIA-SINPAD",
    "FECHA  DE LA EMER",
    "AÑO",
    "MES",
    F.col("`DPTO.`").alias("DPTO"),
    F.col("`PROV.`").alias("PROV"),
    F.col("`DIST.`").alias("DIST"),
    "PELIGRO",
    "FALLECIDOS",
    "LESIONADOS",
    "DAMNIFICADOS",
    "AFECTADOS"
).orderBy(
    "CODIGO DE EMERGENCIA-SINPAD"
).show(20, truncate=False)

In [ ]:
columnas_texto = [
    "FECHA  DE LA EMER",
    "`DPTO.`",
    "`PROV.`",
    "`DIST.`",
    "PELIGRO",
    "TIPO DE PELIGRO"
]

for columna in columnas_texto:
    vacios = df.filter(
        F.trim(F.col(columna)) == ""
    ).count()

    print(f"{columna}: {vacios} vacíos")

In [ ]:
df_ll = df.filter(
    F.upper(F.trim(F.col("`DPTO.`"))) == "LA LIBERTAD"
)

total_ll = df_ll.count()

sinpad_unicos_ll = (
    df_ll
    .select("CODIGO DE EMERGENCIA-SINPAD")
    .distinct()
    .count()
)

duplicados_ll = (
    df_ll
    .groupBy("CODIGO DE EMERGENCIA-SINPAD")
    .count()
    .filter(F.col("count") > 1)
)

print("Registros La Libertad:", total_ll)
print("SINPAD únicos La Libertad:", sinpad_unicos_ll)
print("SINPAD repetidos La Libertad:", duplicados_ll.count())

duplicados_ll.show(truncate=False)

In [ ]:
import re
import unicodedata

def limpiar_nombre_columna(nombre):
    nombre = unicodedata.normalize("NFKD", nombre)
    nombre = "".join(
        c for c in nombre
        if not unicodedata.combining(c)
    )
    nombre = nombre.lower().strip()
    nombre = re.sub(r"[^a-z0-9]+", "_", nombre)
    nombre = re.sub(r"_+", "_", nombre)
    return nombre.strip("_")


df_clean = df

for columna in df.columns:
    df_clean = df_clean.withColumnRenamed(
        columna,
        limpiar_nombre_columna(columna)
    )

print("Columnas:", len(df_clean.columns))
print(df_clean.columns)

In [ ]:
df_clean = (
    df_clean
    .withColumnRenamed(
        "codigo_de_emergencia_sinpad",
        "codigo_sinpad"
    )
    .withColumnRenamed(
        "fecha_de_la_emer",
        "fecha_emergencia"
    )
    .withColumnRenamed(
        "dpto",
        "departamento"
    )
    .withColumnRenamed(
        "prov",
        "provincia"
    )
    .withColumnRenamed(
        "dist",
        "distrito"
    )
)

In [ ]:
from pyspark.sql import functions as F

df_clean = df_clean.withColumn(
    "fecha_emergencia",
    F.to_date(
        F.col("fecha_emergencia"),
        "dd/MM/yyyy"
    )
)

In [ ]:
columnas_texto = [
    "departamento",
    "provincia",
    "distrito",
    "peligro",
    "tipo_de_peligro",
    "region_natural",
    "mes"
]

for columna in columnas_texto:
    df_clean = df_clean.withColumn(
        columna,
        F.upper(
            F.trim(
                F.col(columna)
            )
        )
    )

In [ ]:
print("Filas después de transformación:", df_clean.count())
print("Columnas:", len(df_clean.columns))

df_clean.select(
    "codigo_sinpad",
    "fecha_emergencia",
    "ano",
    "mes",
    "departamento",
    "provincia",
    "distrito",
    "peligro"
).show(10, truncate=False)

In [ ]:
fechas_invalidas = df_clean.filter(
    F.col("fecha_emergencia").isNull()
).count()

anio_inconsistente = df_clean.filter(
    F.year(F.col("fecha_emergencia")) != F.col("ano")
).count()

print("Fechas inválidas:", fechas_invalidas)
print("Fecha y año inconsistentes:", anio_inconsistente)

In [ ]:
import re
import unicodedata

from pyspark.sql import functions as F


# ==========================================
# 1. NORMALIZAR NOMBRES DE COLUMNAS
# ==========================================

def limpiar_nombre_columna(nombre):
    nombre = unicodedata.normalize("NFKD", nombre)

    nombre = "".join(
        c for c in nombre
        if not unicodedata.combining(c)
    )

    nombre = nombre.lower().strip()

    nombre = re.sub(
        r"[^a-z0-9]+",
        "_",
        nombre
    )

    nombre = re.sub(
        r"_+",
        "_",
        nombre
    )

    return nombre.strip("_")


df_clean = df

for columna in df.columns:

    df_clean = df_clean.withColumnRenamed(
        columna,
        limpiar_nombre_columna(columna)
    )


# ==========================================
# 2. NOMBRES MÁS CLAROS
# ==========================================

df_clean = (
    df_clean
    .withColumnRenamed(
        "codigo_de_emergencia_sinpad",
        "codigo_sinpad"
    )
    .withColumnRenamed(
        "fecha_de_la_emer",
        "fecha_emergencia_original"
    )
    .withColumnRenamed(
        "ano",
        "anio"
    )
    .withColumnRenamed(
        "dpto",
        "departamento"
    )
    .withColumnRenamed(
        "prov",
        "provincia"
    )
    .withColumnRenamed(
        "dist",
        "distrito"
    )
)


# ==========================================
# 3. CONVERTIR FECHA DE FORMA SEGURA
# ==========================================

df_clean = df_clean.withColumn(
    "fecha_emergencia",

    F.when(
        F.col("fecha_emergencia_original").rlike(
            r"^\d{1,2}/\d{1,2}/\d{4}$"
        ),

        F.to_date(
            F.col("fecha_emergencia_original"),
            "dd/MM/yyyy"
        )
    )

    .when(
        F.col("fecha_emergencia_original").rlike(
            r"^\d{1,2}/\d{1,2}/\d{2}$"
        ),

        F.to_date(
            F.concat(
                F.substring(
                    F.col("fecha_emergencia_original"),
                    1,
                    6
                ),
                F.lit("20"),
                F.substring(
                    F.col("fecha_emergencia_original"),
                    7,
                    2
                )
            ),
            "dd/MM/yyyy"
        )
    )

    .otherwise(
        F.lit(None).cast("date")
    )
)


# ==========================================
# 4. NORMALIZAR TEXTO
# ==========================================

columnas_texto = [
    "departamento",
    "provincia",
    "distrito",
    "peligro",
    "tipo_de_peligro",
    "region_natural",
    "mes"
]

for columna in columnas_texto:

    df_clean = df_clean.withColumn(
        columna,
        F.upper(
            F.trim(
                F.col(columna)
            )
        )
    )


print(
    "Filas:",
    df_clean.count()
)

print(
    "Columnas:",
    len(df_clean.columns)
)

In [ ]:
fechas_invalidas = (
    df_clean
    .filter(
        F.col("fecha_emergencia").isNull()
    )
    .count()
)

anio_inconsistente = (
    df_clean
    .filter(
        F.col("fecha_emergencia").isNotNull()
        &
        (
            F.year(
                F.col("fecha_emergencia")
            )
            !=
            F.col("anio")
        )
    )
    .count()
)

print(
    "Fechas inválidas:",
    fechas_invalidas
)

print(
    "Fecha y año inconsistentes:",
    anio_inconsistente
)

In [ ]:
df_clean.filter(
    F.col("fecha_emergencia_original").rlike(
        r"^\d{1,2}/\d{1,2}/\d{2}$"
    )
).select(
    "codigo_sinpad",
    "fecha_emergencia_original",
    "fecha_emergencia",
    "anio",
    "departamento"
).show(30, truncate=False)

In [ ]:
df_fechas_2dig = df_clean.filter(
    F.col("fecha_emergencia_original").rlike(
        r"^\d{1,2}/\d{1,2}/\d{2}$"
    )
)

total_2dig = df_fechas_2dig.count()

segundo_mayor_12 = df_fechas_2dig.filter(
    F.split(
        F.col("fecha_emergencia_original"),
        "/"
    )[1].cast("int") > 12
).count()

primero_mayor_12 = df_fechas_2dig.filter(
    F.split(
        F.col("fecha_emergencia_original"),
        "/"
    )[0].cast("int") > 12
).count()

print("Fechas con año de 2 dígitos:", total_2dig)
print("Segundo valor > 12:", segundo_mayor_12)
print("Primer valor > 12:", primero_mayor_12)

In [ ]:
# Separar los componentes de la fecha original
partes = F.split(
    F.col("fecha_emergencia_original"),
    "/"
)

df_clean = df_clean.withColumn(
    "fecha_emergencia",

    # FORMATO ANTIGUO:
    # DD/MM/YYYY
    F.when(
        F.col("fecha_emergencia_original").rlike(
            r"^\d{1,2}/\d{1,2}/\d{4}$"
        ),
        F.make_date(
            partes[2].cast("int"),  # año
            partes[1].cast("int"),  # mes
            partes[0].cast("int")   # día
        )
    )

    # FORMATO 2025:
    # MM/DD/YY
    .when(
        F.col("fecha_emergencia_original").rlike(
            r"^\d{1,2}/\d{1,2}/\d{2}$"
        ),
        F.make_date(
            (partes[2].cast("int") + 2000),  # año
            partes[0].cast("int"),           # mes
            partes[1].cast("int")            # día
        )
    )

    .otherwise(
        F.lit(None).cast("date")
    )
)

In [ ]:
fechas_invalidas = df_clean.filter(
    F.col("fecha_emergencia").isNull()
).count()

anio_inconsistente = df_clean.filter(
    F.col("fecha_emergencia").isNotNull()
    &
    (
        F.year(F.col("fecha_emergencia"))
        != F.col("anio")
    )
).count()

print("Fechas inválidas:", fechas_invalidas)
print("Fecha y año inconsistentes:", anio_inconsistente)

In [ ]:
df_clean.filter(
    F.col("fecha_emergencia_original").rlike(
        r"^\d{1,2}/\d{1,2}/\d{2}$"
    )
).select(
    "codigo_sinpad",
    "fecha_emergencia_original",
    "fecha_emergencia",
    "anio",
    "departamento"
).show(30, truncate=False)

In [ ]:
fechas_invalidas = (
    df_clean
    .filter(F.col("fecha_emergencia").isNull())
    .count()
)

anio_inconsistente = (
    df_clean
    .filter(
        F.col("fecha_emergencia").isNotNull()
        &
        (
            F.year(F.col("fecha_emergencia"))
            != F.col("anio")
        )
    )
    .count()
)

print("Fechas inválidas:", fechas_invalidas)
print("Fecha y año inconsistentes:", anio_inconsistente)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

total_registros = df_clean.count()

expr_nulos = [
    F.sum(
        F.when(
            F.col(c).isNull(),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_clean.columns
]

fila_nulos = df_clean.agg(
    *expr_nulos
).collect()[0]

resumen_nulos = []

for columna in df_clean.columns:
    cantidad = fila_nulos[columna]

    resumen_nulos.append(
        (
            columna,
            cantidad,
            round(
                cantidad / total_registros * 100,
                2
            )
        )
    )

resumen_nulos = sorted(
    resumen_nulos,
    key=lambda x: x[1],
    reverse=True
)

print("Total de registros:", total_registros)

for columna, nulos, porcentaje in resumen_nulos:
    print(
        f"{columna:35} "
        f"{nulos:8} nulos "
        f"({porcentaje:6.2f}%)"
    )

In [ ]:
columnas_string = [
    campo.name
    for campo in df_clean.schema.fields
    if isinstance(
        campo.dataType,
        StringType
    )
]

expr_vacios = [
    F.sum(
        F.when(
            F.trim(F.col(c)) == "",
            1
        ).otherwise(0)
    ).alias(c)
    for c in columnas_string
]

fila_vacios = df_clean.agg(
    *expr_vacios
).collect()[0]

print("TEXTOS VACÍOS")
print("-" * 55)

for columna in columnas_string:
    vacios = fila_vacios[columna]

    if vacios > 0:
        print(
            f"{columna:35} "
            f"{vacios:8} vacíos"
        )

In [ ]:
columnas_con_nulos = [
    "aula_des_inha",
    "scant_ccee",
    "safec_aula",
    "sdestru_aula",
    "sinhab_aula",
    "centros_educativos_destruidos",
    "centros_educativos_afectados"
]

expr_por_anio = [
    F.sum(
        F.when(
            F.col(c).isNull(),
            1
        ).otherwise(0)
    ).alias(c)
    for c in columnas_con_nulos
]

nulos_por_anio = (
    df_clean
    .groupBy("anio")
    .agg(
        F.count("*").alias("registros"),
        *expr_por_anio
    )
    .orderBy("anio")
)

nulos_por_anio.show(30, truncate=False)

In [ ]:
df_ll_clean = df_clean.filter(
    F.col("departamento") == "LA LIBERTAD"
)

expr_ll = [
    F.sum(
        F.when(
            F.col(c).isNull(),
            1
        ).otherwise(0)
    ).alias(c)
    for c in columnas_con_nulos
]

df_ll_clean.agg(*expr_ll).show(truncate=False)

print("Registros La Libertad:", df_ll_clean.count())

In [ ]:
df_clean = df_clean.drop("fecha_emergencia_original")

In [ ]:
print("Filas:", df_clean.count())
print("Columnas:", len(df_clean.columns))

In [ ]:
df_ll_clean = df_clean.filter(
    F.col("departamento") == "LA LIBERTAD"
)

print("Dataset nacional limpio:", df_clean.count())
print("Dataset La Libertad:", df_ll_clean.count())

In [ ]:
(
    df_clean
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(
        "hdfs://localhost:9000/proyecto_emergencias/processed/emergencias_nacional"
    )
)

In [ ]:
(
    df_ll_clean
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(
        "hdfs://localhost:9000/proyecto_emergencias/processed/emergencias_la_libertad"
    )
)

In [ ]:
df_test_nacional = spark.read.parquet(
    "hdfs://localhost:9000/proyecto_emergencias/processed/emergencias_nacional"
)

df_test_ll = spark.read.parquet(
    "hdfs://localhost:9000/proyecto_emergencias/processed/emergencias_la_libertad"
)

print("Nacional:", df_test_nacional.count())
print("La Libertad:", df_test_ll.count())

In [ ]:
ruta_coen = "hdfs://localhost:9000/proyecto_emergencias/raw/coen/metadata_reportes_linkage.csv"

df_coen = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(ruta_coen)
)

print("Filas COEN:", df_coen.count())
print("Columnas COEN:", len(df_coen.columns))

df_coen.printSchema()

In [ ]:
df_coen.select(
    "anio",
    "titulo",
    "sinpad_original",
    "sinpad_inferido",
    "sinpad_final",
    "metodo_match",
    "confianza_match"
).show(20, truncate=False)

In [ ]:
from pyspark.sql import functions as F

total_coen = df_coen.count()

con_sinpad = df_coen.filter(
    F.col("sinpad_final").isNotNull()
).count()

sin_sinpad = df_coen.filter(
    F.col("sinpad_final").isNull()
).count()

sinpad_unicos_coen = (
    df_coen
    .filter(F.col("sinpad_final").isNotNull())
    .select("sinpad_final")
    .distinct()
    .count()
)

print("Total PDFs:", total_coen)
print("PDF con SINPAD final:", con_sinpad)
print("PDF sin SINPAD:", sin_sinpad)
print("SINPAD únicos en COEN:", sinpad_unicos_coen)

In [ ]:
reportes_por_sinpad = (
    df_coen
    .filter(F.col("sinpad_final").isNotNull())
    .groupBy("sinpad_final")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

print(
    "SINPAD con más de un reporte:",
    reportes_por_sinpad.count()
)

reportes_por_sinpad.show(30, truncate=False)

In [ ]:
df_coen_valid = (
    df_coen
    .filter(F.col("sinpad_final").isNotNull())
    .withColumn(
        "codigo_sinpad",
        F.col("sinpad_final").cast("int")
    )
)

In [ ]:
df_coen_valid.select(
    "codigo_sinpad",
    "anio",
    "titulo",
    "metodo_match",
    "confianza_match"
).show(10, truncate=False)

In [ ]:
df_coen_agrupado = (
    df_coen_valid
    .groupBy("codigo_sinpad")
    .agg(
        F.count("*").alias("cantidad_reportes_coen"),

        F.collect_set(
            "titulo"
        ).alias("titulos_coen"),

        F.collect_set(
            "url_pdf"
        ).alias("urls_pdf_coen"),

        F.collect_set(
            "anio"
        ).alias("anios_reportes_coen"),

        F.collect_set(
            "metodo_match"
        ).alias("metodos_match_coen"),

        F.collect_set(
            "confianza_match"
        ).alias("confianzas_match_coen")
    )
)

In [ ]:
print(
    "Emergencias únicas con COEN:",
    df_coen_agrupado.count()
)

df_coen_agrupado.orderBy(
    F.desc("cantidad_reportes_coen")
).show(20, truncate=False)

In [ ]:
df_integrado = (
    df_ll_clean
    .join(
        df_coen_agrupado,
        on="codigo_sinpad",
        how="left"
    )
)

antes = df_ll_clean.count()
despues = df_integrado.count()

print("Antes del JOIN:", antes)
print("Después del JOIN:", despues)

In [ ]:
con_coen = (
    df_integrado
    .filter(
        F.col("cantidad_reportes_coen").isNotNull()
    )
    .count()
)

sin_coen = (
    df_integrado
    .filter(
        F.col("cantidad_reportes_coen").isNull()
    )
    .count()
)

porcentaje_match = round(
    con_coen / df_integrado.count() * 100,
    2
)

print("Emergencias con reporte COEN:", con_coen)
print("Emergencias sin reporte COEN:", sin_coen)
print("Cobertura del JOIN:", porcentaje_match, "%")

In [ ]:
coen_sin_match_csv = (
    df_coen_agrupado
    .join(
        df_ll_clean.select("codigo_sinpad"),
        on="codigo_sinpad",
        how="left_anti"
    )
)

print(
    "SINPAD COEN sin correspondencia en CSV:",
    coen_sin_match_csv.count()
)

coen_sin_match_csv.select(
    "codigo_sinpad",
    "anios_reportes_coen",
    "cantidad_reportes_coen"
).show(50, truncate=False)

In [ ]:
df_integrado = (
    df_integrado
    .withColumn(
        "tiene_reporte_coen",
        F.when(
            F.col("cantidad_reportes_coen").isNotNull(),
            1
        ).otherwise(0)
    )
    .withColumn(
        "cantidad_reportes_coen",
        F.coalesce(
            F.col("cantidad_reportes_coen"),
            F.lit(0)
        )
    )
)

In [ ]:
(
    df_integrado
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(
        "hdfs://localhost:9000/proyecto_emergencias/processed/emergencias_integradas"
    )
)

In [ ]:
# Códigos COEN que SÍ aparecen en la base de La Libertad
coen_match = (
    df_coen_valid
    .join(
        df_ll_clean.select("codigo_sinpad"),
        on="codigo_sinpad",
        how="inner"
    )
)

# Códigos COEN que NO aparecen
coen_no_match = (
    df_coen_valid
    .join(
        df_ll_clean.select("codigo_sinpad"),
        on="codigo_sinpad",
        how="left_anti"
    )
)

print("PDF con match:", coen_match.count())
print("PDF sin match:", coen_no_match.count())

print("\n--- MATCH POR CONFIANZA ---")
coen_match.groupBy(
    "confianza_match"
).count().show()

print("\n--- NO MATCH POR CONFIANZA ---")
coen_no_match.groupBy(
    "confianza_match"
).count().show()

In [ ]:
print("NO MATCH POR AÑO")

(
    coen_no_match
    .groupBy("anio")
    .count()
    .orderBy("anio")
    .show(20, truncate=False)
)

In [ ]:
print("MATCH POR AÑO")

(
    coen_match
    .groupBy("anio")
    .count()
    .orderBy("anio")
    .show(20, truncate=False)
)

In [ ]:
# SINPAD oficiales de COEN que no encontraron
# correspondencia dentro de La Libertad

coen_no_match_codigos = (
    coen_no_match
    .select("codigo_sinpad")
    .distinct()
)

# Buscar esos códigos en TODA la base nacional
encontrados_nacional = (
    coen_no_match_codigos
    .join(
        df_clean.select(
            "codigo_sinpad",
            "anio",
            "departamento",
            "provincia",
            "distrito",
            "peligro"
        ),
        on="codigo_sinpad",
        how="inner"
    )
)

# Los que tampoco existen en ninguna parte
ausentes_nacional = (
    coen_no_match_codigos
    .join(
        df_clean.select("codigo_sinpad"),
        on="codigo_sinpad",
        how="left_anti"
    )
)

print(
    "SINPAD encontrados en base nacional pero no en La Libertad:",
    encontrados_nacional.select("codigo_sinpad").distinct().count()
)

print(
    "SINPAD ausentes de toda la base estructurada:",
    ausentes_nacional.count()
)

In [ ]:
encontrados_nacional.show(
    50,
    truncate=False
)

In [ ]:
df_integrado = (
    df_integrado
    .withColumn(
        "tiene_reporte_coen",
        F.when(
            F.col("cantidad_reportes_coen").isNotNull(),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "cantidad_reportes_coen",
        F.coalesce(
            F.col("cantidad_reportes_coen"),
            F.lit(0)
        )
    )
)

In [ ]:
df_integrado.groupBy(
    "tiene_reporte_coen"
).count().show()

In [ ]:
print("Filas finales:", df_integrado.count())

print(
    "SINPAD únicos:",
    df_integrado.select("codigo_sinpad").distinct().count()
)

In [ ]:
(
    df_integrado
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(
        "hdfs://localhost:9000/proyecto_emergencias/processed/emergencias_integradas"
    )
)

In [ ]:
df_integrado = df_integrado.withColumn(
    "tiene_reporte_coen",
    F.when(
        F.col("cantidad_reportes_coen") > 0,
        F.lit(1)
    ).otherwise(F.lit(0))
)

In [ ]:
df_integrado.groupBy(
    "tiene_reporte_coen"
).count().orderBy(
    "tiene_reporte_coen"
).show()

In [ ]:
(
    df_integrado
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(
        "hdfs://localhost:9000/proyecto_emergencias/processed/emergencias_integradas"
    )
)

In [ ]:
(
    coen_no_match
    .write
    .mode("overwrite")
    .parquet(
        "hdfs://localhost:9000/proyecto_emergencias/processed/coen_sin_correspondencia"
    )
)

In [ ]:
ruta_inei = "hdfs://localhost:9000/proyecto_emergencias/raw/inei/inei_la_libertad_poblacion_distrital_2021_2025.csv"

df_inei = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_inei)
)

print("Filas INEI:", df_inei.count())
print("Columnas INEI:", len(df_inei.columns))

df_inei.printSchema()
df_inei.show(10, truncate=False)

In [ ]:
from pyspark.sql import functions as F

print(
    "Distritos únicos:",
    df_inei.select("distrito").distinct().count()
)

print("\nRegistros por año:")
df_inei.groupBy("anio") \
    .count() \
    .orderBy("anio") \
    .show()

print("Nulos:")
df_inei.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_inei.columns
]).show()

print("Duplicados provincia+distrito+anio:")
(
    df_inei
    .groupBy("provincia", "distrito", "anio")
    .count()
    .filter(F.col("count") > 1)
    .show(truncate=False)
)

print("Alto Trujillo:")
(
    df_inei
    .filter(F.col("distrito") == "ALTO TRUJILLO")
    .orderBy("anio")
    .show(truncate=False)
)

In [ ]:
from pyspark.sql import functions as F

def normalizar_clave(columna):
    return F.translate(
        F.upper(F.trim(columna)),
        "ÁÉÍÓÚÜÑ",
        "AEIOUUN"
    )


# SINPAD La Libertad 2021-2025
df_ll_2021_2025 = (
    df_ll_clean
    .filter(F.col("anio").between(2021, 2025))
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
)


# INEI normalizado
df_inei_norm = (
    df_inei
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .select(
        "provincia_key",
        "distrito_key",
        "anio",
        "poblacion"
    )
)

In [ ]:
df_sinpad_inei = (
    df_ll_2021_2025
    .join(
        df_inei_norm,
        on=[
            "provincia_key",
            "distrito_key",
            "anio"
        ],
        how="left"
    )
)

In [ ]:
total_antes = df_ll_2021_2025.count()
total_despues = df_sinpad_inei.count()

con_poblacion = (
    df_sinpad_inei
    .filter(F.col("poblacion").isNotNull())
    .count()
)

sin_poblacion = (
    df_sinpad_inei
    .filter(F.col("poblacion").isNull())
    .count()
)

print("Emergencias 2021-2025:", total_antes)
print("Filas después del JOIN:", total_despues)

print("Con población:", con_poblacion)
print("Sin población:", sin_poblacion)

print(
    "Cobertura INEI:",
    round(con_poblacion / total_antes * 100, 2),
    "%"
)

In [ ]:
(
    df_sinpad_inei
    .filter(F.col("poblacion").isNull())
    .select(
        "anio",
        "provincia",
        "distrito"
    )
    .distinct()
    .orderBy(
        "anio",
        "provincia",
        "distrito"
    )
    .show(100, truncate=False)
)

In [ ]:
ruta_chirps = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/raw/chirps/historico/*.csv"
)

df_chirps = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_chirps)
)

print("Filas CHIRPS:", df_chirps.count())
print("Columnas CHIRPS:", len(df_chirps.columns))

df_chirps.printSchema()
df_chirps.show(10, truncate=False)

In [ ]:
from pyspark.sql import functions as F

print(
    "Distritos únicos:",
    df_chirps.select("ubigeo").distinct().count()
)

print(
    "Años:",
    df_chirps.select("anio").distinct().count()
)

print("Nulos:")
df_chirps.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_chirps.columns
]).show()

print("Duplicados ubigeo + año + mes:")
(
    df_chirps
    .groupBy("ubigeo", "anio", "mes")
    .count()
    .filter(F.col("count") > 1)
    .show()
)

print("Registros por año:")
(
    df_chirps
    .groupBy("anio")
    .count()
    .orderBy("anio")
    .show(30)
)

In [ ]:
from pyspark.sql import functions as F

df_chirps_norm = (
    df_chirps
    .withColumn("mes_num", F.col("mes").cast("int"))
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .select(
        "provincia_key",
        "distrito_key",
        "anio",
        "mes_num",
        "precipitacion_total_mm",
        "precipitacion_promedio_diaria_mm",
        "precipitacion_maxima_diaria_mm"
    )
)

df_chirps_norm.printSchema()

In [ ]:
df_base = (
    df_integrado
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .withColumn(
        "mes_num",
        F.month(F.col("fecha_emergencia"))
    )
)

In [ ]:
df_base_chirps = (
    df_base
    .join(
        df_chirps_norm,
        on=[
            "provincia_key",
            "distrito_key",
            "anio",
            "mes_num"
        ],
        how="left"
    )
)

In [ ]:
total_antes = df_base.count()
total_despues = df_base_chirps.count()

con_chirps = (
    df_base_chirps
    .filter(
        F.col("precipitacion_total_mm").isNotNull()
    )
    .count()
)

sin_chirps = (
    df_base_chirps
    .filter(
        F.col("precipitacion_total_mm").isNull()
    )
    .count()
)

print("Emergencias antes:", total_antes)
print("Filas después JOIN:", total_despues)
print("Con CHIRPS:", con_chirps)
print("Sin CHIRPS:", sin_chirps)

print(
    "Cobertura CHIRPS:",
    round(con_chirps / total_antes * 100, 2),
    "%"
)

In [ ]:
(
    df_base_chirps
    .filter(
        F.col("precipitacion_total_mm").isNull()
    )
    .select(
        "anio",
        "provincia",
        "distrito",
        "mes_num"
    )
    .distinct()
    .orderBy(
        "anio",
        "provincia",
        "distrito",
        "mes_num"
    )
    .show(100, truncate=False)
)

In [ ]:
df_maestro = (
    df_base_chirps
    .join(
        df_inei_norm,
        on=[
            "provincia_key",
            "distrito_key",
            "anio"
        ],
        how="left"
    )
)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ======================================================
# 1. Spark
# ======================================================

spark = (
    SparkSession.builder
    .appName("ProyectoEmergenciasLaLibertad")
    .getOrCreate()
)


# ======================================================
# 2. Función para normalizar claves
# ======================================================

def normalizar_clave(columna):
    return F.translate(
        F.upper(F.trim(columna)),
        "ÁÉÍÓÚÜÑ",
        "AEIOUUN"
    )


# ======================================================
# 3. Recuperar SINPAD + COEN ya procesado
# ======================================================

df_integrado = spark.read.parquet(
    "hdfs://localhost:9000/proyecto_emergencias/processed/emergencias_integradas"
)

print("SINPAD + COEN:", df_integrado.count())


# ======================================================
# 4. Recuperar INEI
# ======================================================

df_inei = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "hdfs://localhost:9000/proyecto_emergencias/raw/inei/"
        "inei_la_libertad_poblacion_distrital_2021_2025.csv"
    )
)

df_inei_norm = (
    df_inei
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .select(
        "provincia_key",
        "distrito_key",
        "anio",
        "poblacion"
    )
)


# ======================================================
# 5. Recuperar CHIRPS 2003-2025
# ======================================================

df_chirps = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "hdfs://localhost:9000/"
        "proyecto_emergencias/raw/chirps/historico/*.csv"
    )
)

df_chirps_norm = (
    df_chirps
    .withColumn(
        "mes_num",
        F.col("mes").cast("int")
    )
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .select(
        "provincia_key",
        "distrito_key",
        "anio",
        "mes_num",
        "precipitacion_total_mm",
        "precipitacion_promedio_diaria_mm",
        "precipitacion_maxima_diaria_mm"
    )
)


# ======================================================
# 6. Preparar dataset base
# ======================================================

df_base = (
    df_integrado
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .withColumn(
        "mes_num",
        F.month(F.col("fecha_emergencia"))
    )
)


# ======================================================
# 7. JOIN con CHIRPS
# ======================================================

df_base_chirps = (
    df_base
    .join(
        df_chirps_norm,
        on=[
            "provincia_key",
            "distrito_key",
            "anio",
            "mes_num"
        ],
        how="left"
    )
)


# ======================================================
# 8. JOIN con INEI
# ======================================================

df_maestro = (
    df_base_chirps
    .join(
        df_inei_norm,
        on=[
            "provincia_key",
            "distrito_key",
            "anio"
        ],
        how="left"
    )
)


# ======================================================
# 9. Validación rápida
# ======================================================

print("Filas dataset maestro:", df_maestro.count())

print(
    "Con CHIRPS:",
    df_maestro.filter(
        F.col("precipitacion_total_mm").isNotNull()
    ).count()
)

print(
    "Con población INEI:",
    df_maestro.filter(
        F.col("poblacion").isNotNull()
    ).count()
)

In [ ]:
df_maestro_final = (
    df_maestro
    .drop(
        "provincia_key",
        "distrito_key"
    )
)

(
    df_maestro_final
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(
        "hdfs://localhost:9000/"
        "proyecto_emergencias/processed/dataset_maestro_2003_2025"
    )
)

In [ ]:
df_verificacion = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/dataset_maestro_2003_2025"
)

print("Filas:", df_verificacion.count())

print(
    "SINPAD únicos:",
    df_verificacion
    .select("codigo_sinpad")
    .distinct()
    .count()
)

print(
    "Con CHIRPS:",
    df_verificacion
    .filter(F.col("precipitacion_total_mm").isNotNull())
    .count()
)

print(
    "Con población:",
    df_verificacion
    .filter(F.col("poblacion").isNotNull())
    .count()
)

In [ ]:
from pyspark.sql import functions as F

df_clima_anual = (
    df_chirps
    .withColumn("mes", F.col("mes").cast("int"))
    .groupBy(
        "ubigeo",
        "provincia",
        "distrito",
        "anio"
    )
    .agg(
        F.sum("precipitacion_total_mm")
            .alias("precipitacion_total_anual_mm"),

        F.max("precipitacion_maxima_diaria_mm")
            .alias("precipitacion_maxima_diaria_anual_mm"),

        F.avg("precipitacion_promedio_diaria_mm")
            .alias("precipitacion_promedio_diaria_anual_mm")
    )
)

print("Filas clima anual:", df_clima_anual.count())

df_clima_anual.show(10, truncate=False)

In [ ]:
df_emergencias_anual = (
    df_maestro_final
    .groupBy(
        "provincia",
        "distrito",
        "anio"
    )
    .agg(
        F.count("*").alias("cantidad_emergencias"),

        F.countDistinct("peligro")
            .alias("tipos_peligro_distintos")
    )
)

print(
    "Combinaciones distrito-año con al menos una emergencia:",
    df_emergencias_anual.count()
)

df_emergencias_anual.show(10, truncate=False)

In [ ]:
def normalizar_clave(columna):
    return F.translate(
        F.upper(F.trim(columna)),
        "ÁÉÍÓÚÜÑ",
        "AEIOUUN"
    )


df_clima_panel = (
    df_clima_anual
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
)


df_emergencias_panel = (
    df_emergencias_anual
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .select(
        "provincia_key",
        "distrito_key",
        "anio",
        "cantidad_emergencias",
        "tipos_peligro_distintos"
    )
)


df_panel = (
    df_clima_panel
    .join(
        df_emergencias_panel,
        on=[
            "provincia_key",
            "distrito_key",
            "anio"
        ],
        how="left"
    )
    .fillna({
        "cantidad_emergencias": 0,
        "tipos_peligro_distintos": 0
    })
)

In [ ]:
print("Filas panel:", df_panel.count())

print(
    "Distritos:",
    df_panel.select("ubigeo").distinct().count()
)

print(
    "Años:",
    df_panel.select("anio").distinct().count()
)

df_panel.groupBy("anio") \
    .count() \
    .orderBy("anio") \
    .show(30)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Ventana temporal por distrito
w = (
    Window
    .partitionBy("ubigeo")
    .orderBy("anio")
)

# Ventanas históricas excluyendo el año actual
w_3 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("anio")
    .rowsBetween(-3, -1)
)

w_5 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("anio")
    .rowsBetween(-5, -1)
)


df_features = (
    df_panel

    # Emergencias previas
    .withColumn(
        "emergencias_anio_anterior",
        F.lag("cantidad_emergencias", 1).over(w)
    )

    .withColumn(
        "emergencias_ultimos_3_anios",
        F.sum("cantidad_emergencias").over(w_3)
    )

    .withColumn(
        "emergencias_ultimos_5_anios",
        F.sum("cantidad_emergencias").over(w_5)
    )

    # Promedio histórico reciente
    .withColumn(
        "promedio_emergencias_3_anios",
        F.avg("cantidad_emergencias").over(w_3)
    )

    # Clima previo
    .withColumn(
        "precipitacion_total_anio_anterior",
        F.lag(
            "precipitacion_total_anual_mm",
            1
        ).over(w)
    )

    .withColumn(
        "precipitacion_maxima_anio_anterior",
        F.lag(
            "precipitacion_maxima_diaria_anual_mm",
            1
        ).over(w)
    )

    # TARGET:
    # cantidad de emergencias del distrito el año siguiente
    .withColumn(
        "emergencias_siguiente_anio",
        F.lead("cantidad_emergencias", 1).over(w)
    )
)

In [ ]:
(
    df_features
    .filter(F.col("distrito") == "TRUJILLO")
    .select(
        "anio",
        "cantidad_emergencias",
        "emergencias_anio_anterior",
        "emergencias_ultimos_3_anios",
        "emergencias_ultimos_5_anios",
        "precipitacion_total_anual_mm",
        "precipitacion_total_anio_anterior",
        "emergencias_siguiente_anio"
    )
    .orderBy("anio")
    .show(30, truncate=False)
)

In [ ]:
palabras = [
    "afect",
    "damn",
    "falle",
    "herid",
    "desap",
    "vivi",
    "destr",
    "inhab"
]

columnas_dano = [
    c for c in df_maestro_final.columns
    if any(p in c.lower() for p in palabras)
]

for c in columnas_dano:
    print(c)

In [ ]:
from pyspark.sql import functions as F

df_danos_anual = (
    df_maestro_final
    .groupBy(
        "provincia",
        "distrito",
        "anio"
    )
    .agg(
        F.sum("fallecidos").alias("fallecidos_anio"),
        F.sum("desaparecidos").alias("desaparecidos_anio"),
        F.sum("damnificados").alias("damnificados_anio"),
        F.sum("afectados").alias("afectados_anio"),
        F.sum("viviendas_destruidas").alias("viviendas_destruidas_anio"),
        F.sum("viviendas_afectadas").alias("viviendas_afectadas_anio")
    )
)

print(
    "Combinaciones distrito-año con daños:",
    df_danos_anual.count()
)

df_danos_anual.show(10, truncate=False)

In [ ]:
df_danos_panel = (
    df_danos_anual
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .select(
        "provincia_key",
        "distrito_key",
        "anio",
        "fallecidos_anio",
        "desaparecidos_anio",
        "damnificados_anio",
        "afectados_anio",
        "viviendas_destruidas_anio",
        "viviendas_afectadas_anio"
    )
)

In [ ]:
df_panel_danos = (
    df_panel
    .join(
        df_danos_panel,
        on=[
            "provincia_key",
            "distrito_key",
            "anio"
        ],
        how="left"
    )
    .fillna({
        "fallecidos_anio": 0,
        "desaparecidos_anio": 0,
        "damnificados_anio": 0,
        "afectados_anio": 0,
        "viviendas_destruidas_anio": 0,
        "viviendas_afectadas_anio": 0
    })
)

In [ ]:
print("Filas panel con daños:", df_panel_danos.count())

print(
    "Distritos:",
    df_panel_danos.select("ubigeo").distinct().count()
)

print(
    "Años:",
    df_panel_danos.select("anio").distinct().count()
)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Orden temporal por distrito
w = (
    Window
    .partitionBy("ubigeo")
    .orderBy("anio")
)

# Solo años ANTERIORES al año de la fila
w3 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("anio")
    .rowsBetween(-3, -1)
)

w5 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("anio")
    .rowsBetween(-5, -1)
)

df_features = (
    df_panel_danos

    # ==================================================
    # FRECUENCIA HISTÓRICA
    # ==================================================
    .withColumn(
        "emergencias_anio_anterior",
        F.lag("cantidad_emergencias", 1).over(w)
    )
    .withColumn(
        "emergencias_ultimos_3_anios",
        F.sum("cantidad_emergencias").over(w3)
    )
    .withColumn(
        "emergencias_ultimos_5_anios",
        F.sum("cantidad_emergencias").over(w5)
    )

    # ==================================================
    # DAÑOS HISTÓRICOS
    # ==================================================
    .withColumn(
        "afectados_ultimos_3_anios",
        F.sum("afectados_anio").over(w3)
    )
    .withColumn(
        "damnificados_ultimos_3_anios",
        F.sum("damnificados_anio").over(w3)
    )
    .withColumn(
        "fallecidos_ultimos_5_anios",
        F.sum("fallecidos_anio").over(w5)
    )
    .withColumn(
        "desaparecidos_ultimos_5_anios",
        F.sum("desaparecidos_anio").over(w5)
    )
    .withColumn(
        "viviendas_destruidas_ultimos_3_anios",
        F.sum("viviendas_destruidas_anio").over(w3)
    )
    .withColumn(
        "viviendas_afectadas_ultimos_3_anios",
        F.sum("viviendas_afectadas_anio").over(w3)
    )

    # ==================================================
    # CLIMA PREVIO
    # ==================================================
    .withColumn(
        "precipitacion_total_anio_anterior",
        F.lag("precipitacion_total_anual_mm", 1).over(w)
    )
    .withColumn(
        "precipitacion_maxima_anio_anterior",
        F.lag("precipitacion_maxima_diaria_anual_mm", 1).over(w)
    )

    # ==================================================
    # TARGET: EMERGENCIAS DEL AÑO SIGUIENTE
    # ==================================================
    .withColumn(
        "emergencias_siguiente_anio",
        F.lead("cantidad_emergencias", 1).over(w)
    )
)

In [ ]:
df_modelo = (
    df_features
    .filter(
        (F.col("anio") >= 2008) &
        (F.col("anio") <= 2024)
    )
)

print("Filas para modelado:", df_modelo.count())
print(
    "Targets nulos:",
    df_modelo.filter(
        F.col("emergencias_siguiente_anio").isNull()
    ).count()
)

In [ ]:
df_prediccion_2026 = (
    df_features
    .filter(F.col("anio") == 2025)
)

print(
    "Distritos a predecir para 2026:",
    df_prediccion_2026.count()
)

print(
    "Target desconocido 2026:",
    df_prediccion_2026.filter(
        F.col("emergencias_siguiente_anio").isNull()
    ).count()
)

In [ ]:
# Validación del conjunto de modelado
print("Filas para modelado:", df_modelo.count())

print(
    "Targets nulos:",
    df_modelo.filter(
        F.col("emergencias_siguiente_anio").isNull()
    ).count()
)

print("\nDistribución del número de emergencias del año siguiente:")

(
    df_modelo
    .groupBy("emergencias_siguiente_anio")
    .count()
    .orderBy("emergencias_siguiente_anio")
    .show(50)
)

# Estadísticas generales
df_modelo.select(
    F.min("emergencias_siguiente_anio").alias("min"),
    F.avg("emergencias_siguiente_anio").alias("promedio"),
    F.max("emergencias_siguiente_anio").alias("maximo")
).show()

# Percentiles
percentiles = df_modelo.approxQuantile(
    "emergencias_siguiente_anio",
    [0.25, 0.50, 0.75, 0.90],
    0.01
)

print("P25:", percentiles[0])
print("Mediana:", percentiles[1])
print("P75:", percentiles[2])
print("P90:", percentiles[3])

# Cuántos tienen cero y cuántos al menos una emergencia
df_modelo.select(
    F.sum(
        F.when(F.col("emergencias_siguiente_anio") == 0, 1).otherwise(0)
    ).alias("sin_emergencias"),

    F.sum(
        F.when(F.col("emergencias_siguiente_anio") > 0, 1).otherwise(0)
    ).alias("con_emergencias")
).show()

In [ ]:
df_modelo_riesgo = (
    df_modelo
    .withColumn(
        "nivel_riesgo",
        F.when(
            F.col("emergencias_siguiente_anio") == 0,
            F.lit("BAJO")
        )
        .when(
            F.col("emergencias_siguiente_anio").between(1, 3),
            F.lit("MEDIO")
        )
        .otherwise(
            F.lit("ALTO")
        )
    )
)

In [ ]:
(
    df_modelo_riesgo
    .groupBy("nivel_riesgo")
    .count()
    .orderBy(F.desc("count"))
    .show()
)

In [ ]:
# ======================================================
# DIVISIÓN TEMPORAL
# ======================================================

df_train = (
    df_modelo_riesgo
    .filter(F.col("anio").between(2008, 2021))
)

df_val = (
    df_modelo_riesgo
    .filter(F.col("anio").between(2022, 2023))
)

df_test = (
    df_modelo_riesgo
    .filter(F.col("anio") == 2024)
)

print("Train:", df_train.count())
print("Validación:", df_val.count())
print("Test:", df_test.count())

In [ ]:
print("TRAIN")
df_train.groupBy("nivel_riesgo").count().show()

print("VALIDACIÓN")
df_val.groupBy("nivel_riesgo").count().show()

print("TEST")
df_test.groupBy("nivel_riesgo").count().show()

In [ ]:
df_distribucion_anual = (
    df_modelo_riesgo
    .withColumn(
        "anio_objetivo",
        F.col("anio") + 1
    )
    .groupBy(
        "anio_objetivo",
        "nivel_riesgo"
    )
    .count()
)

(
    df_distribucion_anual
    .groupBy("anio_objetivo")
    .pivot(
        "nivel_riesgo",
        ["BAJO", "MEDIO", "ALTO"]
    )
    .sum("count")
    .fillna(0)
    .orderBy("anio_objetivo")
    .show(30)
)

In [ ]:
(
    df_modelo
    .withColumn(
        "anio_objetivo",
        F.col("anio") + 1
    )
    .groupBy("anio_objetivo")
    .agg(
        F.avg("emergencias_siguiente_anio")
            .alias("promedio_emergencias"),

        F.max("emergencias_siguiente_anio")
            .alias("max_emergencias"),

        F.sum("emergencias_siguiente_anio")
            .alias("total_emergencias")
    )
    .orderBy("anio_objetivo")
    .show(30, truncate=False)
)

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Etiqueta numérica fija
def agregar_label(df):
    return (
        df.withColumn(
            "label",
            F.when(F.col("nivel_riesgo") == "BAJO", 0.0)
             .when(F.col("nivel_riesgo") == "MEDIO", 1.0)
             .when(F.col("nivel_riesgo") == "ALTO", 2.0)
        )
    )

df_train_dt = agregar_label(df_train)
df_val_dt = agregar_label(df_val)
df_test_dt = agregar_label(df_test)

# Variables conocidas ANTES del año objetivo
features_cols = [
    "cantidad_emergencias",
    "tipos_peligro_distintos",

    "emergencias_anio_anterior",
    "emergencias_ultimos_3_anios",
    "emergencias_ultimos_5_anios",

    "afectados_ultimos_3_anios",
    "damnificados_ultimos_3_anios",
    "fallecidos_ultimos_5_anios",
    "desaparecidos_ultimos_5_anios",

    "viviendas_destruidas_ultimos_3_anios",
    "viviendas_afectadas_ultimos_3_anios",

    "precipitacion_total_anual_mm",
    "precipitacion_maxima_diaria_anual_mm",

    "precipitacion_total_anio_anterior",
    "precipitacion_maxima_anio_anterior"
]

assembler = VectorAssembler(
    inputCols=features_cols,
    outputCol="features",
    handleInvalid="keep"
)

train_vec = assembler.transform(df_train_dt)
val_vec = assembler.transform(df_val_dt)
test_vec = assembler.transform(df_test_dt)

In [ ]:
dt = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=5,
    minInstancesPerNode=10,
    seed=42
)

modelo_dt = dt.fit(train_vec)

pred_val = modelo_dt.transform(val_vec)

In [ ]:
evaluador_accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluador_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

accuracy = evaluador_accuracy.evaluate(pred_val)
f1_weighted = evaluador_f1.evaluate(pred_val)

print("Accuracy validación:", round(accuracy, 4))
print("F1 weighted:", round(f1_weighted, 4))

In [ ]:
tp_alto = pred_val.filter(
    (F.col("label") == 2) &
    (F.col("prediction") == 2)
).count()

fn_alto = pred_val.filter(
    (F.col("label") == 2) &
    (F.col("prediction") != 2)
).count()

fp_alto = pred_val.filter(
    (F.col("label") != 2) &
    (F.col("prediction") == 2)
).count()

recall_alto = (
    tp_alto / (tp_alto + fn_alto)
    if (tp_alto + fn_alto) > 0 else 0
)

precision_alto = (
    tp_alto / (tp_alto + fp_alto)
    if (tp_alto + fp_alto) > 0 else 0
)

f1_alto = (
    2 * precision_alto * recall_alto /
    (precision_alto + recall_alto)
    if (precision_alto + recall_alto) > 0 else 0
)

print("Precision ALTO:", round(precision_alto, 4))
print("Recall ALTO:", round(recall_alto, 4))
print("F1 ALTO:", round(f1_alto, 4))

In [ ]:
(
    pred_val
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
    .show()
)

In [ ]:
# Conteos del TRAIN
conteos = {
    row["nivel_riesgo"]: row["count"]
    for row in df_train.groupBy("nivel_riesgo").count().collect()
}

n_total = sum(conteos.values())
n_clases = len(conteos)

peso_bajo = n_total / (n_clases * conteos["BAJO"])
peso_medio = n_total / (n_clases * conteos["MEDIO"])
peso_alto = n_total / (n_clases * conteos["ALTO"])

print("Peso BAJO:", peso_bajo)
print("Peso MEDIO:", peso_medio)
print("Peso ALTO:", peso_alto)

In [ ]:
df_train_pesos = (
    df_train_dt
    .withColumn(
        "peso_clase",
        F.when(F.col("label") == 0, F.lit(peso_bajo))
         .when(F.col("label") == 1, F.lit(peso_medio))
         .otherwise(F.lit(peso_alto))
    )
)

train_vec_pesos = assembler.transform(df_train_pesos)

In [ ]:
dt_pesos = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    weightCol="peso_clase",
    maxDepth=5,
    minInstancesPerNode=10,
    seed=42
)

modelo_dt_pesos = dt_pesos.fit(train_vec_pesos)

pred_val_pesos = modelo_dt_pesos.transform(val_vec)

In [ ]:
tp = pred_val_pesos.filter(
    (F.col("label") == 2) &
    (F.col("prediction") == 2)
).count()

fn = pred_val_pesos.filter(
    (F.col("label") == 2) &
    (F.col("prediction") != 2)
).count()

fp = pred_val_pesos.filter(
    (F.col("label") != 2) &
    (F.col("prediction") == 2)
).count()

recall_alto_pesos = tp / (tp + fn) if (tp + fn) else 0
precision_alto_pesos = tp / (tp + fp) if (tp + fp) else 0

f1_alto_pesos = (
    2 * precision_alto_pesos * recall_alto_pesos /
    (precision_alto_pesos + recall_alto_pesos)
    if (precision_alto_pesos + recall_alto_pesos) else 0
)

print("Precision ALTO:", round(precision_alto_pesos, 4))
print("Recall ALTO:", round(recall_alto_pesos, 4))
print("F1 ALTO:", round(f1_alto_pesos, 4))

In [ ]:
(
    pred_val_pesos
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
    .show()
)

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf_pesos = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    weightCol="peso_clase",
    numTrees=100,
    maxDepth=8,
    minInstancesPerNode=5,
    seed=42
)

modelo_rf_pesos = rf_pesos.fit(train_vec_pesos)

pred_val_rf = modelo_rf_pesos.transform(val_vec)

In [ ]:
accuracy_rf = evaluador_accuracy.evaluate(pred_val_rf)
f1_weighted_rf = evaluador_f1.evaluate(pred_val_rf)

print("Accuracy RF:", round(accuracy_rf, 4))
print("F1 weighted RF:", round(f1_weighted_rf, 4))

In [ ]:
tp_rf = pred_val_rf.filter(
    (F.col("label") == 2) &
    (F.col("prediction") == 2)
).count()

fn_rf = pred_val_rf.filter(
    (F.col("label") == 2) &
    (F.col("prediction") != 2)
).count()

fp_rf = pred_val_rf.filter(
    (F.col("label") != 2) &
    (F.col("prediction") == 2)
).count()

recall_alto_rf = (
    tp_rf / (tp_rf + fn_rf)
    if (tp_rf + fn_rf) else 0
)

precision_alto_rf = (
    tp_rf / (tp_rf + fp_rf)
    if (tp_rf + fp_rf) else 0
)

f1_alto_rf = (
    2 * precision_alto_rf * recall_alto_rf /
    (precision_alto_rf + recall_alto_rf)
    if (precision_alto_rf + recall_alto_rf) else 0
)

print("Precision ALTO RF:", round(precision_alto_rf, 4))
print("Recall ALTO RF:", round(recall_alto_rf, 4))
print("F1 ALTO RF:", round(f1_alto_rf, 4))

In [ ]:
(
    pred_val_rf
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
    .show()
)

In [ ]:
importancias = modelo_rf_pesos.featureImportances

ranking_features = sorted(
    zip(features_cols, importancias),
    key=lambda x: x[1],
    reverse=True
)

for nombre, importancia in ranking_features:
    print(f"{nombre:45s} {importancia:.4f}")

In [ ]:
df_train_final = (
    df_modelo_riesgo
    .filter(F.col("anio").between(2008, 2023))
)

df_train_final = agregar_label(df_train_final)

print("Filas entrenamiento final:", df_train_final.count())

df_train_final.groupBy("nivel_riesgo").count().show()

In [ ]:
conteos_final = {
    row["nivel_riesgo"]: row["count"]
    for row in df_train_final.groupBy("nivel_riesgo").count().collect()
}

n_total_final = sum(conteos_final.values())
n_clases = 3

peso_bajo_final = (
    n_total_final /
    (n_clases * conteos_final["BAJO"])
)

peso_medio_final = (
    n_total_final /
    (n_clases * conteos_final["MEDIO"])
)

peso_alto_final = (
    n_total_final /
    (n_clases * conteos_final["ALTO"])
)

print("BAJO:", peso_bajo_final)
print("MEDIO:", peso_medio_final)
print("ALTO:", peso_alto_final)

In [ ]:
df_train_final = (
    df_train_final
    .withColumn(
        "peso_clase",
        F.when(
            F.col("label") == 0,
            F.lit(peso_bajo_final)
        )
        .when(
            F.col("label") == 1,
            F.lit(peso_medio_final)
        )
        .otherwise(
            F.lit(peso_alto_final)
        )
    )
)

train_final_vec = assembler.transform(df_train_final)

In [ ]:
dt_final = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    weightCol="peso_clase",
    maxDepth=5,
    minInstancesPerNode=10,
    seed=42
)

modelo_dt_final = dt_final.fit(train_final_vec)

In [ ]:
pred_test = modelo_dt_final.transform(test_vec)

accuracy_test = evaluador_accuracy.evaluate(pred_test)
f1_test = evaluador_f1.evaluate(pred_test)

print("Accuracy TEST:", round(accuracy_test, 4))
print("F1 weighted TEST:", round(f1_test, 4))

In [ ]:
tp = pred_test.filter(
    (F.col("label") == 2) &
    (F.col("prediction") == 2)
).count()

fn = pred_test.filter(
    (F.col("label") == 2) &
    (F.col("prediction") != 2)
).count()

fp = pred_test.filter(
    (F.col("label") != 2) &
    (F.col("prediction") == 2)
).count()

recall_alto_test = tp / (tp + fn) if (tp + fn) else 0

precision_alto_test = (
    tp / (tp + fp)
    if (tp + fp) else 0
)

f1_alto_test = (
    2 * precision_alto_test * recall_alto_test /
    (precision_alto_test + recall_alto_test)
    if (precision_alto_test + recall_alto_test)
    else 0
)

print("Precision ALTO TEST:", round(precision_alto_test, 4))
print("Recall ALTO TEST:", round(recall_alto_test, 4))
print("F1 ALTO TEST:", round(f1_alto_test, 4))

In [ ]:
(
    pred_test
    .groupBy("label", "prediction")
    .count()
    .orderBy("label", "prediction")
    .show()
)

In [ ]:
df_train_2026 = (
    df_modelo_riesgo
    .filter(F.col("anio").between(2008, 2024))
)

df_train_2026 = agregar_label(df_train_2026)

print("Filas entrenamiento 2026:", df_train_2026.count())

df_train_2026.groupBy("nivel_riesgo").count().show()

In [ ]:
conteos_2026 = {
    row["nivel_riesgo"]: row["count"]
    for row in df_train_2026.groupBy("nivel_riesgo").count().collect()
}

n_total_2026 = sum(conteos_2026.values())

peso_bajo_2026 = n_total_2026 / (3 * conteos_2026["BAJO"])
peso_medio_2026 = n_total_2026 / (3 * conteos_2026["MEDIO"])
peso_alto_2026 = n_total_2026 / (3 * conteos_2026["ALTO"])

print("Peso BAJO:", peso_bajo_2026)
print("Peso MEDIO:", peso_medio_2026)
print("Peso ALTO:", peso_alto_2026)

In [ ]:
df_train_2026 = (
    df_train_2026
    .withColumn(
        "peso_clase",
        F.when(
            F.col("label") == 0,
            F.lit(peso_bajo_2026)
        )
        .when(
            F.col("label") == 1,
            F.lit(peso_medio_2026)
        )
        .otherwise(
            F.lit(peso_alto_2026)
        )
    )
)

train_2026_vec = assembler.transform(df_train_2026)

In [ ]:
dt_2026 = DecisionTreeClassifier(
    labelCol="label",
    featuresCol="features",
    weightCol="peso_clase",
    maxDepth=5,
    minInstancesPerNode=10,
    seed=42
)

modelo_2026 = dt_2026.fit(train_2026_vec)

In [ ]:
prediccion_2026_vec = assembler.transform(
    df_prediccion_2026
)

resultado_2026 = modelo_2026.transform(
    prediccion_2026_vec
)

In [ ]:
resultado_2026 = (
    resultado_2026
    .withColumn(
        "riesgo_2026",
        F.when(F.col("prediction") == 0, "BAJO")
         .when(F.col("prediction") == 1, "MEDIO")
         .otherwise("ALTO")
    )
)

In [ ]:
(
    resultado_2026
    .groupBy("riesgo_2026")
    .count()
    .show()
)

In [ ]:
(
    resultado_2026
    .select(
        "provincia",
        "distrito",
        "cantidad_emergencias",
        "emergencias_ultimos_3_anios",
        "emergencias_ultimos_5_anios",
        "riesgo_2026"
    )
    .orderBy(
        F.desc("prediction"),
        F.desc("emergencias_ultimos_3_anios")
    )
    .show(84, truncate=False)
)

In [ ]:
from pyspark.ml.functions import vector_to_array

resultado_2026_scores = (
    resultado_2026
    .withColumn(
        "probabilidades",
        vector_to_array("probability")
    )
    .withColumn(
        "score_bajo",
        F.col("probabilidades")[0]
    )
    .withColumn(
        "score_medio",
        F.col("probabilidades")[1]
    )
    .withColumn(
        "score_alto",
        F.col("probabilidades")[2]
    )
    .drop("probabilidades")
)

In [ ]:
(
    resultado_2026_scores
    .select(
        "provincia",
        "distrito",
        "cantidad_emergencias",
        "emergencias_ultimos_3_anios",
        "riesgo_2026",
        F.round("score_alto", 4).alias("score_alto")
    )
    .orderBy(
        F.desc("score_alto")
    )
    .show(25, truncate=False)
)

In [ ]:
print(
    "Distritos:",
    resultado_2026_scores.select("ubigeo").distinct().count()
)

In [ ]:
df_poblacion_2025 = (
    df_inei_norm
    .filter(F.col("anio") == 2025)
    .select(
        "provincia_key",
        "distrito_key",
        F.col("poblacion").alias("poblacion_2025")
    )
)

print("Distritos INEI 2025:", df_poblacion_2025.count())

In [ ]:
resultado_2026_poblacion = (
    resultado_2026_scores
    .join(
        df_poblacion_2025,
        on=[
            "provincia_key",
            "distrito_key"
        ],
        how="left"
    )
)

print("Distritos totales:", resultado_2026_poblacion.count())

print(
    "Con población:",
    resultado_2026_poblacion
    .filter(F.col("poblacion_2025").isNotNull())
    .count()
)

print(
    "Sin población:",
    resultado_2026_poblacion
    .filter(F.col("poblacion_2025").isNull())
    .count()
)

In [ ]:
(
    resultado_2026_poblacion
    .filter(F.col("riesgo_2026") == "ALTO")
    .select(
        "provincia",
        "distrito",
        "riesgo_2026",
        F.round("score_alto", 4).alias("score_alto"),
        "poblacion_2025",
        "cantidad_emergencias",
        "emergencias_ultimos_3_anios"
    )
    .orderBy(
        F.desc("score_alto"),
        F.desc("poblacion_2025")
    )
    .show(44, truncate=False)
)

In [ ]:
df_chirps_2026 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "hdfs://localhost:9000/proyecto_emergencias/raw/chirps/2026/*.csv"
    )
)

print("Filas CHIRPS 2026:", df_chirps_2026.count())
print(
    "Distritos:",
    df_chirps_2026.select("ubigeo").distinct().count()
)
print(
    "Meses:",
    df_chirps_2026.select("mes").distinct().count()
)

In [ ]:
df_lluvia_2026 = (
    df_chirps_2026
    .withColumn("ubigeo", F.col("ubigeo").cast("string"))
    .groupBy(
        "ubigeo",
        "provincia",
        "distrito"
    )
    .agg(
        F.sum("precipitacion_total_mm")
            .alias("precipitacion_ene_ago_2026_mm"),

        F.max("precipitacion_maxima_diaria_mm")
            .alias("maxima_diaria_ene_ago_2026_mm")
    )
)

print("Distritos 2026:", df_lluvia_2026.count())

In [ ]:
df_hist_ene_ago_anual = (
    df_chirps
    .withColumn("mes_num", F.col("mes").cast("int"))
    .withColumn("ubigeo", F.col("ubigeo").cast("string"))
    .filter(F.col("mes_num").between(1, 8))
    .groupBy(
        "ubigeo",
        "anio"
    )
    .agg(
        F.sum("precipitacion_total_mm")
            .alias("precipitacion_ene_ago_mm")
    )
)

In [ ]:
df_clima_historico_ref = (
    df_hist_ene_ago_anual
    .groupBy("ubigeo")
    .agg(
        F.avg("precipitacion_ene_ago_mm")
            .alias("promedio_historico_ene_ago_mm"),

        F.stddev("precipitacion_ene_ago_mm")
            .alias("desv_historica_ene_ago_mm")
    )
)

In [ ]:
df_clima_2026_contexto = (
    df_lluvia_2026
    .join(
        df_clima_historico_ref,
        on="ubigeo",
        how="left"
    )
    .withColumn(
        "anomalia_precipitacion_pct",
        (
            (
                F.col("precipitacion_ene_ago_2026_mm")
                - F.col("promedio_historico_ene_ago_mm")
            )
            / F.col("promedio_historico_ene_ago_mm")
        ) * 100
    )
    .withColumn(
        "z_precipitacion_2026",
        (
            F.col("precipitacion_ene_ago_2026_mm")
            - F.col("promedio_historico_ene_ago_mm")
        ) / F.col("desv_historica_ene_ago_mm")
    )
)

In [ ]:
(
    df_clima_2026_contexto
    .select(
        "provincia",
        "distrito",
        F.round(
            "precipitacion_ene_ago_2026_mm", 2
        ).alias("lluvia_2026_mm"),
        F.round(
            "promedio_historico_ene_ago_mm", 2
        ).alias("promedio_historico_mm"),
        F.round(
            "anomalia_precipitacion_pct", 2
        ).alias("anomalia_pct"),
        F.round(
            "z_precipitacion_2026", 2
        ).alias("z_score")
    )
    .orderBy(
        F.desc("anomalia_precipitacion_pct")
    )
    .show(25, truncate=False)
)

In [ ]:
resultado_final_2026 = (
    resultado_2026_poblacion
    .join(
        df_clima_2026_contexto.select(
            "ubigeo",
            F.col("precipitacion_ene_ago_2026_mm")
                .alias("lluvia_ene_ago_2026_mm"),
            "promedio_historico_ene_ago_mm",
            "anomalia_precipitacion_pct",
            "z_precipitacion_2026"
        ),
        on="ubigeo",
        how="left"
    )
)

print("Filas finales:", resultado_final_2026.count())

print(
    "Con clima 2026:",
    resultado_final_2026
    .filter(F.col("lluvia_ene_ago_2026_mm").isNotNull())
    .count()
)

In [ ]:
resultado_final_2026 = (
    resultado_final_2026
    .withColumn(
        "contexto_lluvia_2026",
        F.when(
            F.col("z_precipitacion_2026") >= 2,
            "MUY_SUPERIOR_A_LO_NORMAL"
        )
        .when(
            F.col("z_precipitacion_2026") >= 1,
            "SUPERIOR_A_LO_NORMAL"
        )
        .when(
            F.col("z_precipitacion_2026") <= -1,
            "INFERIOR_A_LO_NORMAL"
        )
        .otherwise("DENTRO_DE_LO_NORMAL")
    )
)

In [ ]:
tabla_priorizacion_2026 = (
    resultado_final_2026
    .select(
        "ubigeo",
        "provincia",
        "distrito",
        "riesgo_2026",
        F.round("score_alto", 4).alias("score_modelo_alto"),
        "poblacion_2025",
        "cantidad_emergencias",
        "emergencias_ultimos_3_anios",
        "emergencias_ultimos_5_anios",
        F.round(
            "lluvia_ene_ago_2026_mm", 2
        ).alias("lluvia_ene_ago_2026_mm"),
        F.round(
            "promedio_historico_ene_ago_mm", 2
        ).alias("promedio_hist_ene_ago_mm"),
        F.round(
            "anomalia_precipitacion_pct", 2
        ).alias("anomalia_lluvia_pct"),
        F.round(
            "z_precipitacion_2026", 2
        ).alias("z_lluvia_2026"),
        "contexto_lluvia_2026"
    )
)

In [ ]:
(
    tabla_priorizacion_2026
    .filter(F.col("riesgo_2026") == "ALTO")
    .orderBy(
        F.desc("score_modelo_alto"),
        F.desc("poblacion_2025")
    )
    .show(44, truncate=False)
)

In [ ]:
ruta_resultado_2026 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/priorizacion_distrital_2026"
)

(
    tabla_priorizacion_2026
    .write
    .mode("overwrite")
    .parquet(ruta_resultado_2026)
)

print("Tabla final guardada.")

In [ ]:
ruta_modelo_2026 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/models/decision_tree_riesgo_2026"
)

(
    modelo_2026
    .write()
    .overwrite()
    .save(ruta_modelo_2026)
)

print("Modelo guardado.")

In [ ]:
df_resultado_guardado = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/priorizacion_distrital_2026"
)

print("Filas guardadas:", df_resultado_guardado.count())
print(
    "Distritos únicos:",
    df_resultado_guardado.select("ubigeo").distinct().count()
)

df_resultado_guardado.groupBy("riesgo_2026").count().show()

In [ ]:
from pyspark.sql import functions as F

df_emergencias_mensual = (
    df_maestro_final
    .withColumn(
        "mes_num",
        F.month("fecha_emergencia")
    )
    .groupBy(
        "provincia",
        "distrito",
        "anio",
        "mes_num"
    )
    .agg(
        F.count("*")
            .alias("cantidad_emergencias_mes"),

        F.countDistinct("peligro")
            .alias("tipos_peligro_mes")
    )
)

print(
    "Combinaciones distrito-mes con emergencias:",
    df_emergencias_mensual.count()
)

In [ ]:
df_emergencias_mensual = (
    df_emergencias_mensual
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .select(
        "provincia_key",
        "distrito_key",
        "anio",
        "mes_num",
        "cantidad_emergencias_mes",
        "tipos_peligro_mes"
    )
)

In [ ]:
df_chirps_mensual = (
    df_chirps
    .withColumn(
        "mes_num",
        F.col("mes").cast("int")
    )
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
)

In [ ]:
df_panel_mensual = (
    df_chirps_mensual
    .join(
        df_emergencias_mensual,
        on=[
            "provincia_key",
            "distrito_key",
            "anio",
            "mes_num"
        ],
        how="left"
    )
    .fillna({
        "cantidad_emergencias_mes": 0,
        "tipos_peligro_mes": 0
    })
)

In [ ]:
print(
    "Filas panel mensual:",
    df_panel_mensual.count()
)

print(
    "Distritos:",
    df_panel_mensual
    .select("ubigeo")
    .distinct()
    .count()
)

print(
    "Años:",
    df_panel_mensual
    .select("anio")
    .distinct()
    .count()
)

print(
    "Meses:",
    df_panel_mensual
    .select("mes_num")
    .distinct()
    .count()
)

In [ ]:
(
    df_panel_mensual
    .groupBy("anio")
    .count()
    .orderBy("anio")
    .show(30)
)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Índice cronológico continuo
df_panel_mensual = (
    df_panel_mensual
    .withColumn(
        "periodo_idx",
        F.col("anio") * 12 + F.col("mes_num")
    )
)

# Ventana cronológica por distrito
w_mes = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
)

w_3m = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
    .rowsBetween(-3, -1)
)

w_6m = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
    .rowsBetween(-6, -1)
)

w_12m = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
    .rowsBetween(-12, -1)
)

df_features_mensual = (
    df_panel_mensual

    # Emergencias recientes
    .withColumn(
        "emergencias_mes_anterior",
        F.lag("cantidad_emergencias_mes", 1).over(w_mes)
    )

    .withColumn(
        "emergencias_3_meses",
        F.sum("cantidad_emergencias_mes").over(w_3m)
    )

    .withColumn(
        "emergencias_6_meses",
        F.sum("cantidad_emergencias_mes").over(w_6m)
    )

    .withColumn(
        "emergencias_12_meses",
        F.sum("cantidad_emergencias_mes").over(w_12m)
    )

    # Mismo mes del año anterior
    .withColumn(
        "emergencias_mismo_mes_anio_anterior",
        F.lag("cantidad_emergencias_mes", 12).over(w_mes)
    )

    # Clima previo
    .withColumn(
        "lluvia_mes_anterior",
        F.lag("precipitacion_total_mm", 1).over(w_mes)
    )

    .withColumn(
        "lluvia_3_meses",
        F.sum("precipitacion_total_mm").over(w_3m)
    )

    # TARGET:
    # mismo mes del año siguiente
    .withColumn(
        "emergencias_mismo_mes_anio_siguiente",
        F.lead("cantidad_emergencias_mes", 12).over(w_mes)
    )
)

In [ ]:
(
    df_features_mensual
    .filter(F.col("distrito") == "TRUJILLO")
    .select(
        "anio",
        "mes_num",
        "cantidad_emergencias_mes",
        "emergencias_mes_anterior",
        "emergencias_3_meses",
        "emergencias_12_meses",
        "emergencias_mismo_mes_anio_anterior",
        "precipitacion_total_mm",
        "emergencias_mismo_mes_anio_siguiente"
    )
    .orderBy("anio", "mes_num")
    .show(30, truncate=False)
)

In [ ]:
# Solo filas cuyo año siguiente sí conocemos
df_mensual_modelable = (
    df_features_mensual
    .filter(
        F.col("emergencias_mismo_mes_anio_siguiente").isNotNull()
    )
)

print(
    "Filas con target conocido:",
    df_mensual_modelable.count()
)

print("\nDistribución del target mensual:")

(
    df_mensual_modelable
    .groupBy("emergencias_mismo_mes_anio_siguiente")
    .count()
    .orderBy("emergencias_mismo_mes_anio_siguiente")
    .show(50)
)

In [ ]:
# Ceros vs al menos una emergencia
df_mensual_modelable.select(
    F.sum(
        F.when(
            F.col("emergencias_mismo_mes_anio_siguiente") == 0,
            1
        ).otherwise(0)
    ).alias("meses_sin_emergencias"),

    F.sum(
        F.when(
            F.col("emergencias_mismo_mes_anio_siguiente") > 0,
            1
        ).otherwise(0)
    ).alias("meses_con_emergencias")
).show()

In [ ]:
df_mensual_modelable.select(
    F.min("emergencias_mismo_mes_anio_siguiente").alias("min"),
    F.avg("emergencias_mismo_mes_anio_siguiente").alias("promedio"),
    F.max("emergencias_mismo_mes_anio_siguiente").alias("max")
).show()

In [ ]:
df_prediccion_mensual_2026 = (
    df_features_mensual
    .filter(F.col("anio") == 2025)
)

print(
    "Filas para predecir 2026:",
    df_prediccion_mensual_2026.count()
)

print(
    "Targets nulos 2026:",
    df_prediccion_mensual_2026
    .filter(
        F.col("emergencias_mismo_mes_anio_siguiente").isNull()
    )
    .count()
)

In [ ]:
df_mensual_binario = (
    df_features_mensual
    .filter(
        F.col("emergencias_mismo_mes_anio_siguiente").isNotNull()
    )
    .withColumn(
        "ocurre_emergencia",
        F.when(
            F.col("emergencias_mismo_mes_anio_siguiente") > 0,
            F.lit(1.0)
        ).otherwise(F.lit(0.0))
    )
)

In [ ]:
(
    df_mensual_binario
    .groupBy("ocurre_emergencia")
    .count()
    .orderBy("ocurre_emergencia")
    .show()
)

In [ ]:
df_train_mensual = (
    df_mensual_binario
    .filter(F.col("anio").between(2004, 2021))
)

df_val_mensual = (
    df_mensual_binario
    .filter(F.col("anio").between(2022, 2023))
)

df_test_mensual = (
    df_mensual_binario
    .filter(F.col("anio") == 2024)
)

print("TRAIN:", df_train_mensual.count())
print("VALIDACIÓN:", df_val_mensual.count())
print("TEST:", df_test_mensual.count())

In [ ]:
print("TRAIN")
df_train_mensual.groupBy("ocurre_emergencia").count().show()

print("VALIDACIÓN")
df_val_mensual.groupBy("ocurre_emergencia").count().show()

print("TEST")
df_test_mensual.groupBy("ocurre_emergencia").count().show()

In [ ]:
df_tasa_anual = (
    df_mensual_binario
    .groupBy("anio")
    .agg(
        F.count("*").alias("total_meses_distrito"),
        F.sum("ocurre_emergencia").alias("meses_con_emergencia")
    )
    .withColumn(
        "porcentaje_con_emergencia",
        F.round(
            F.col("meses_con_emergencia")
            / F.col("total_meses_distrito") * 100,
            2
        )
    )
    .orderBy("anio")
)

df_tasa_anual.show(30, truncate=False)

In [ ]:
import math

df_mensual_binario = (
    df_mensual_binario

    # Estacionalidad cíclica
    .withColumn(
        "mes_sin",
        F.sin(
            2 * F.lit(math.pi) * F.col("mes_num") / 12
        )
    )

    .withColumn(
        "mes_cos",
        F.cos(
            2 * F.lit(math.pi) * F.col("mes_num") / 12
        )
    )

    # Cambio reciente de frecuencia
    .withColumn(
        "cambio_emergencias_3m_vs_12m",
        F.col("emergencias_3_meses") -
        (F.col("emergencias_12_meses") / 4)
    )
)

In [ ]:
df_train_mensual = (
    df_mensual_binario
    .filter(F.col("anio").between(2004, 2021))
)

df_val_mensual = (
    df_mensual_binario
    .filter(F.col("anio").between(2022, 2023))
)

df_test_mensual = (
    df_mensual_binario
    .filter(F.col("anio") == 2024)
)

In [ ]:
conteos = {
    row["ocurre_emergencia"]: row["count"]
    for row in df_train_mensual
        .groupBy("ocurre_emergencia")
        .count()
        .collect()
}

n_train = sum(conteos.values())

peso_no = (
    n_train /
    (2 * conteos[0.0])
)

peso_si = (
    n_train /
    (2 * conteos[1.0])
)

print("Peso NO:", peso_no)
print("Peso SI:", peso_si)

In [ ]:
df_train_mensual = (
    df_train_mensual
    .withColumn(
        "peso_clase",
        F.when(
            F.col("ocurre_emergencia") == 1,
            F.lit(peso_si)
        ).otherwise(
            F.lit(peso_no)
        )
    )
)

In [ ]:
# Total real de emergencias de 2017
df_2017 = df_maestro_final.filter(F.col("anio") == 2017)

print("Emergencias 2017:", df_2017.count())

print(
    "SINPAD únicos 2017:",
    df_2017.select("codigo_sinpad").distinct().count()
)

# Emergencias por mes
(
    df_2017
    .withColumn("mes", F.month("fecha_emergencia"))
    .groupBy("mes")
    .count()
    .orderBy("mes")
    .show(12)
)

# Distritos con más emergencias
(
    df_2017
    .groupBy("provincia", "distrito")
    .count()
    .orderBy(F.desc("count"))
    .show(30, truncate=False)
)

In [ ]:
(
    df_2017
    .filter(F.col("distrito") == "TRUJILLO")
    .withColumn("mes", F.month("fecha_emergencia"))
    .groupBy("mes")
    .count()
    .orderBy("mes")
    .show()
)

In [ ]:
df_panel_mensual.filter(F.col("anio") == 2017).select(
    F.sum("cantidad_emergencias_mes").alias("total_emergencias"),
    F.sum(
        F.when(F.col("cantidad_emergencias_mes") > 0, 1).otherwise(0)
    ).alias("distrito_meses_con_emergencia")
).show()

In [ ]:
df_tasa_anual_correcta = (
    df_mensual_binario
    .withColumn(
        "anio_objetivo",
        F.col("anio") + 1
    )
    .groupBy("anio_objetivo")
    .agg(
        F.count("*").alias("total_meses_distrito"),
        F.sum("ocurre_emergencia").alias("meses_con_emergencia")
    )
    .withColumn(
        "porcentaje_con_emergencia",
        F.round(
            F.col("meses_con_emergencia")
            / F.col("total_meses_distrito") * 100,
            2
        )
    )
    .orderBy("anio_objetivo")
)

df_tasa_anual_correcta.show(30, truncate=False)

In [ ]:
(
    df_maestro_final
    .groupBy("peligro")
    .count()
    .orderBy(F.desc("count"))
    .show(100, truncate=False)
)

In [ ]:
(
    df_maestro_final
    .filter(F.col("anio").between(2019, 2025))
    .groupBy("anio", "peligro")
    .count()
    .orderBy("anio", F.desc("count"))
    .show(200, truncate=False)
)

In [ ]:
(
    df_maestro_final
    .filter(F.col("anio").between(2015, 2025))
    .groupBy("anio")
    .pivot("peligro")
    .count()
    .fillna(0)
    .orderBy("anio")
    .show(truncate=False)
)

In [ ]:
(
    df_maestro_final
    .groupBy("anio")
    .agg(
        F.count("*").alias("emergencias"),
        F.sum("afectados").alias("afectados"),
        F.sum("damnificados").alias("damnificados"),
        F.sum("fallecidos").alias("fallecidos"),
        F.sum("viviendas_afectadas").alias("viviendas_afectadas"),
        F.sum("viviendas_destruidas").alias("viviendas_destruidas")
    )
    .orderBy("anio")
    .show(30, truncate=False)
)

In [ ]:
df_calidad_registro = (
    df_maestro_final
    .groupBy("anio")
    .agg(
        F.count("*").alias("emergencias"),
        F.sum("afectados").alias("afectados"),
        F.sum("damnificados").alias("damnificados"),
        F.sum("fallecidos").alias("fallecidos"),
        F.sum("viviendas_afectadas").alias("viviendas_afectadas"),
        F.sum("viviendas_destruidas").alias("viviendas_destruidas")
    )
    .withColumn(
        "afectados_por_emergencia",
        F.round(F.col("afectados") / F.col("emergencias"), 2)
    )
    .withColumn(
        "damnificados_por_emergencia",
        F.round(F.col("damnificados") / F.col("emergencias"), 2)
    )
    .withColumn(
        "viviendas_afectadas_por_emergencia",
        F.round(F.col("viviendas_afectadas") / F.col("emergencias"), 2)
    )
    .orderBy("anio")
)

df_calidad_registro.show(30, truncate=False)

In [ ]:
df_danos_mensual = (
    df_maestro_final
    .withColumn(
        "mes_num",
        F.month("fecha_emergencia")
    )
    .groupBy(
        "provincia",
        "distrito",
        "anio",
        "mes_num"
    )
    .agg(
        F.sum("afectados").alias("afectados_mes"),
        F.sum("damnificados").alias("damnificados_mes"),
        F.sum("fallecidos").alias("fallecidos_mes"),
        F.sum("desaparecidos").alias("desaparecidos_mes"),
        F.sum("viviendas_afectadas").alias("viviendas_afectadas_mes"),
        F.sum("viviendas_destruidas").alias("viviendas_destruidas_mes")
    )
)

In [ ]:
df_danos_mensual = (
    df_danos_mensual
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .select(
        "provincia_key",
        "distrito_key",
        "anio",
        "mes_num",
        "afectados_mes",
        "damnificados_mes",
        "fallecidos_mes",
        "desaparecidos_mes",
        "viviendas_afectadas_mes",
        "viviendas_destruidas_mes"
    )
)

df_panel_mensual_danos = (
    df_panel_mensual
    .join(
        df_danos_mensual,
        on=[
            "provincia_key",
            "distrito_key",
            "anio",
            "mes_num"
        ],
        how="left"
    )
    .fillna({
        "afectados_mes": 0,
        "damnificados_mes": 0,
        "fallecidos_mes": 0,
        "desaparecidos_mes": 0,
        "viviendas_afectadas_mes": 0,
        "viviendas_destruidas_mes": 0
    })
)

print("Filas:", df_panel_mensual_danos.count())

In [ ]:
df_danos_mensual = (
    df_maestro_final
    .withColumn("mes_num", F.month("fecha_emergencia"))
    .groupBy(
        "provincia",
        "distrito",
        "anio",
        "mes_num"
    )
    .agg(
        F.sum("afectados").alias("afectados_mes"),
        F.sum("damnificados").alias("damnificados_mes"),
        F.sum("fallecidos").alias("fallecidos_mes"),
        F.sum("desaparecidos").alias("desaparecidos_mes"),
        F.sum("viviendas_afectadas").alias("viviendas_afectadas_mes"),
        F.sum("viviendas_destruidas").alias("viviendas_destruidas_mes")
    )
)

df_danos_mensual = (
    df_danos_mensual
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .select(
        "provincia_key",
        "distrito_key",
        "anio",
        "mes_num",
        "afectados_mes",
        "damnificados_mes",
        "fallecidos_mes",
        "desaparecidos_mes",
        "viviendas_afectadas_mes",
        "viviendas_destruidas_mes"
    )
)

df_panel_mensual_danos = (
    df_panel_mensual
    .join(
        df_danos_mensual,
        on=[
            "provincia_key",
            "distrito_key",
            "anio",
            "mes_num"
        ],
        how="left"
    )
    .fillna({
        "afectados_mes": 0,
        "damnificados_mes": 0,
        "fallecidos_mes": 0,
        "desaparecidos_mes": 0,
        "viviendas_afectadas_mes": 0,
        "viviendas_destruidas_mes": 0
    })
)

print("Filas:", df_panel_mensual_danos.count())
print(
    "Distritos:",
    df_panel_mensual_danos.select("ubigeo").distinct().count()
)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_meses_con_evento = (
    df_panel_mensual_danos
    .filter(F.col("cantidad_emergencias_mes") > 0)
)

print("Meses-distrito con emergencias:", df_meses_con_evento.count())

In [ ]:
w_afectados = Window.orderBy("afectados_mes")
w_damnificados = Window.orderBy("damnificados_mes")
w_fallecidos = Window.orderBy("fallecidos_mes")
w_desaparecidos = Window.orderBy("desaparecidos_mes")
w_viv_afectadas = Window.orderBy("viviendas_afectadas_mes")
w_viv_destruidas = Window.orderBy("viviendas_destruidas_mes")

df_impacto_eventos = (
    df_meses_con_evento

    .withColumn(
        "p_afectados",
        F.percent_rank().over(w_afectados)
    )
    .withColumn(
        "p_damnificados",
        F.percent_rank().over(w_damnificados)
    )
    .withColumn(
        "p_fallecidos",
        F.percent_rank().over(w_fallecidos)
    )
    .withColumn(
        "p_desaparecidos",
        F.percent_rank().over(w_desaparecidos)
    )
    .withColumn(
        "p_viviendas_afectadas",
        F.percent_rank().over(w_viv_afectadas)
    )
    .withColumn(
        "p_viviendas_destruidas",
        F.percent_rank().over(w_viv_destruidas)
    )
)

In [ ]:
df_impacto_eventos = (
    df_impacto_eventos
    .withColumn(
        "score_impacto",
        (
            F.col("p_afectados")
            + F.col("p_damnificados")
            + F.col("p_fallecidos")
            + F.col("p_desaparecidos")
            + F.col("p_viviendas_afectadas")
            + F.col("p_viviendas_destruidas")
        ) / 6
    )
)

In [ ]:
df_scores_impacto = (
    df_impacto_eventos
    .select(
        "ubigeo",
        "anio",
        "mes_num",
        "score_impacto"
    )
)

df_panel_impacto = (
    df_panel_mensual_danos
    .join(
        df_scores_impacto,
        on=["ubigeo", "anio", "mes_num"],
        how="left"
    )
    .fillna({
        "score_impacto": 0.0
    })
)

print("Filas:", df_panel_impacto.count())

In [ ]:
(
    df_panel_impacto
    .filter(F.col("cantidad_emergencias_mes") > 0)
    .select("score_impacto")
    .summary()
    .show()
)

In [ ]:
(
    df_panel_impacto
    .filter(F.col("cantidad_emergencias_mes") > 0)
    .select(
        "anio",
        "mes_num",
        "provincia",
        "distrito",
        "cantidad_emergencias_mes",
        "afectados_mes",
        "damnificados_mes",
        "fallecidos_mes",
        "viviendas_afectadas_mes",
        "viviendas_destruidas_mes",
        F.round("score_impacto", 4).alias("score_impacto")
    )
    .orderBy(F.desc("score_impacto"))
    .show(20, truncate=False)
)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import math

df_modelo_mensual_base = (
    df_panel_mensual_danos
    .withColumn(
        "periodo_idx",
        F.col("anio") * 12 + F.col("mes_num")
    )
)

w = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
)

w3 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
    .rowsBetween(-3, -1)
)

w6 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
    .rowsBetween(-6, -1)
)

w12 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
    .rowsBetween(-12, -1)
)

In [ ]:
df_features_mensual_v2 = (
    df_modelo_mensual_base

    # ===============================
    # FRECUENCIA
    # ===============================
    .withColumn(
        "emergencias_mes_anterior",
        F.lag("cantidad_emergencias_mes", 1).over(w)
    )
    .withColumn(
        "emergencias_3_meses",
        F.sum("cantidad_emergencias_mes").over(w3)
    )
    .withColumn(
        "emergencias_6_meses",
        F.sum("cantidad_emergencias_mes").over(w6)
    )
    .withColumn(
        "emergencias_12_meses",
        F.sum("cantidad_emergencias_mes").over(w12)
    )
    .withColumn(
        "emergencias_mismo_mes_anio_anterior",
        F.lag("cantidad_emergencias_mes", 12).over(w)
    )

    # ===============================
    # IMPACTO HISTÓRICO
    # ===============================
    .withColumn(
        "afectados_12_meses",
        F.sum("afectados_mes").over(w12)
    )
    .withColumn(
        "damnificados_12_meses",
        F.sum("damnificados_mes").over(w12)
    )
    .withColumn(
        "fallecidos_12_meses",
        F.sum("fallecidos_mes").over(w12)
    )
    .withColumn(
        "viviendas_afectadas_12_meses",
        F.sum("viviendas_afectadas_mes").over(w12)
    )
    .withColumn(
        "viviendas_destruidas_12_meses",
        F.sum("viviendas_destruidas_mes").over(w12)
    )

    # ===============================
    # CLIMA
    # ===============================
    .withColumn(
        "lluvia_mes_anterior",
        F.lag("precipitacion_total_mm", 1).over(w)
    )
    .withColumn(
        "lluvia_3_meses",
        F.sum("precipitacion_total_mm").over(w3)
    )

    # ===============================
    # ESTACIONALIDAD
    # ===============================
    .withColumn(
        "mes_sin",
        F.sin(
            2 * F.lit(math.pi) * F.col("mes_num") / 12
        )
    )
    .withColumn(
        "mes_cos",
        F.cos(
            2 * F.lit(math.pi) * F.col("mes_num") / 12
        )
    )

    # ===============================
    # TARGET: MISMO MES AÑO SIGUIENTE
    # ===============================
    .withColumn(
        "emergencias_mismo_mes_anio_siguiente",
        F.lead("cantidad_emergencias_mes", 12).over(w)
    )

    .withColumn(
        "ocurre_emergencia",
        F.when(
            F.col("emergencias_mismo_mes_anio_siguiente") > 0,
            F.lit(1.0)
        )
        .when(
            F.col("emergencias_mismo_mes_anio_siguiente") == 0,
            F.lit(0.0)
        )
    )
)

In [ ]:
df_modelable_v2 = (
    df_features_mensual_v2
    .filter(
        (F.col("anio") >= 2004) &
        (F.col("anio") <= 2024) &
        F.col("ocurre_emergencia").isNotNull()
    )
)

print("Filas modelables:", df_modelable_v2.count())

df_modelable_v2.groupBy("ocurre_emergencia").count().show()

In [ ]:
df_pred_2026_v2 = (
    df_features_mensual_v2
    .filter(F.col("anio") == 2025)
)

print("Filas para 2026:", df_pred_2026_v2.count())

print(
    "Targets nulos:",
    df_pred_2026_v2
    .filter(
        F.col("emergencias_mismo_mes_anio_siguiente").isNull()
    )
    .count()
)

In [ ]:
df_train_v2 = (
    df_modelable_v2
    .filter(F.col("anio").between(2004, 2021))
)

df_val_v2 = (
    df_modelable_v2
    .filter(F.col("anio").between(2022, 2023))
)

df_test_v2 = (
    df_modelable_v2
    .filter(F.col("anio") == 2024)
)

print("TRAIN:", df_train_v2.count())
print("VALIDACIÓN:", df_val_v2.count())
print("TEST:", df_test_v2.count())

In [ ]:
conteos_v2 = {
    row["ocurre_emergencia"]: row["count"]
    for row in (
        df_train_v2
        .groupBy("ocurre_emergencia")
        .count()
        .collect()
    )
}

n_train_v2 = sum(conteos_v2.values())

peso_no_v2 = (
    n_train_v2 /
    (2 * conteos_v2[0.0])
)

peso_si_v2 = (
    n_train_v2 /
    (2 * conteos_v2[1.0])
)

print("NO:", conteos_v2[0.0], "| peso:", peso_no_v2)
print("SI:", conteos_v2[1.0], "| peso:", peso_si_v2)

In [ ]:
df_train_v2 = (
    df_train_v2
    .withColumn(
        "peso_clase",
        F.when(
            F.col("ocurre_emergencia") == 1.0,
            F.lit(peso_si_v2)
        ).otherwise(
            F.lit(peso_no_v2)
        )
    )
)

In [ ]:
features_mensuales = [
    "cantidad_emergencias_mes",
    "tipos_peligro_mes",

    "emergencias_mes_anterior",
    "emergencias_3_meses",
    "emergencias_6_meses",
    "emergencias_12_meses",
    "emergencias_mismo_mes_anio_anterior",

    "afectados_12_meses",
    "damnificados_12_meses",
    "fallecidos_12_meses",
    "viviendas_afectadas_12_meses",
    "viviendas_destruidas_12_meses",

    "precipitacion_total_mm",
    "precipitacion_promedio_diaria_mm",
    "precipitacion_maxima_diaria_mm",
    "lluvia_mes_anterior",
    "lluvia_3_meses",

    "mes_sin",
    "mes_cos"
]

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler_mensual = VectorAssembler(
    inputCols=features_mensuales,
    outputCol="features",
    handleInvalid="keep"
)

train_vec_v2 = assembler_mensual.transform(df_train_v2)
val_vec_v2 = assembler_mensual.transform(df_val_v2)
test_vec_v2 = assembler_mensual.transform(df_test_v2)

In [ ]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    labelCol="ocurre_emergencia",
    featuresCol="features",
    weightCol="peso_clase",
    maxIter=80,
    maxDepth=5,
    minInstancesPerNode=20,
    stepSize=0.05,
    seed=42
)

modelo_gbt = gbt.fit(train_vec_v2)

pred_val_gbt = modelo_gbt.transform(val_vec_v2)

In [ ]:
tp = pred_val_gbt.filter(
    (F.col("ocurre_emergencia") == 1) &
    (F.col("prediction") == 1)
).count()

fn = pred_val_gbt.filter(
    (F.col("ocurre_emergencia") == 1) &
    (F.col("prediction") == 0)
).count()

fp = pred_val_gbt.filter(
    (F.col("ocurre_emergencia") == 0) &
    (F.col("prediction") == 1)
).count()

tn = pred_val_gbt.filter(
    (F.col("ocurre_emergencia") == 0) &
    (F.col("prediction") == 0)
).count()

precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) else 0
)

accuracy = (
    (tp + tn) / (tp + tn + fp + fn)
)

print("TP:", tp)
print("FN:", fn)
print("FP:", fp)
print("TN:", tn)

print("Accuracy:", round(accuracy, 4))
print("Precision emergencia:", round(precision, 4))
print("Recall emergencia:", round(recall, 4))
print("F1 emergencia:", round(f1, 4))

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluador_pr = BinaryClassificationEvaluator(
    labelCol="ocurre_emergencia",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

pr_auc = evaluador_pr.evaluate(pred_val_gbt)

print("PR-AUC:", round(pr_auc, 4))

In [ ]:
(
    pred_val_gbt
    .groupBy(
        "ocurre_emergencia",
        "prediction"
    )
    .count()
    .orderBy(
        "ocurre_emergencia",
        "prediction"
    )
    .show()
)

In [ ]:
from pyspark.ml.functions import vector_to_array

pred_val_scores = (
    pred_val_gbt
    .withColumn(
        "prob_array",
        vector_to_array("probability")
    )
    .withColumn(
        "prob_emergencia",
        F.col("prob_array")[1]
    )
)

In [ ]:
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

resultados_threshold = []

for t in thresholds:

    df_t = pred_val_scores.withColumn(
        "pred_t",
        F.when(
            F.col("prob_emergencia") >= t,
            1.0
        ).otherwise(0.0)
    )

    tp = df_t.filter(
        (F.col("ocurre_emergencia") == 1) &
        (F.col("pred_t") == 1)
    ).count()

    fn = df_t.filter(
        (F.col("ocurre_emergencia") == 1) &
        (F.col("pred_t") == 0)
    ).count()

    fp = df_t.filter(
        (F.col("ocurre_emergencia") == 0) &
        (F.col("pred_t") == 1)
    ).count()

    tn = df_t.filter(
        (F.col("ocurre_emergencia") == 0) &
        (F.col("pred_t") == 0)
    ).count()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    resultados_threshold.append(
        (t, tp, fn, fp, tn, precision, recall, f1)
    )

for r in resultados_threshold:
    print(
        f"Threshold={r[0]:.2f} | "
        f"TP={r[1]} FN={r[2]} FP={r[3]} TN={r[4]} | "
        f"Precision={r[5]:.4f} "
        f"Recall={r[6]:.4f} "
        f"F1={r[7]:.4f}"
    )

In [ ]:
for r in resultados_threshold:
    t = r[0]
    precision = r[5]
    recall = r[6]

    f2 = (
        5 * precision * recall /
        (4 * precision + recall)
        if (4 * precision + recall) > 0
        else 0
    )

    print(
        f"Threshold={t:.2f} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f} | "
        f"F1={r[7]:.4f} | "
        f"F2={f2:.4f}"
    )

In [ ]:
import math

anio_mas_reciente_train = 2021
vida_media = 5.0

df_train_temporal = (
    df_train_v2
    .withColumn(
        "peso_temporal",
        F.pow(
            F.lit(0.5),
            (F.lit(anio_mas_reciente_train) - F.col("anio"))
            / F.lit(vida_media)
        )
    )
    .withColumn(
        "peso_clase_temp",
        F.col("peso_clase") * F.col("peso_temporal")
    )
)

In [ ]:
train_vec_temporal = assembler_mensual.transform(
    df_train_temporal
)

gbt_temporal = GBTClassifier(
    labelCol="ocurre_emergencia",
    featuresCol="features",
    weightCol="peso_clase_temp",
    maxIter=80,
    maxDepth=5,
    minInstancesPerNode=20,
    stepSize=0.05,
    seed=42
)

modelo_gbt_temporal = gbt_temporal.fit(
    train_vec_temporal
)

pred_val_temporal = modelo_gbt_temporal.transform(
    val_vec_v2
)

In [ ]:
from pyspark.ml.functions import vector_to_array

pred_val_temporal = (
    pred_val_temporal
    .withColumn(
        "prob_array",
        vector_to_array("probability")
    )
    .withColumn(
        "prob_emergencia",
        F.col("prob_array")[1]
    )
)

In [ ]:
pred_val_temporal_030 = (
    pred_val_temporal
    .withColumn(
        "pred_030",
        F.when(
            F.col("prob_emergencia") >= 0.30,
            1.0
        ).otherwise(0.0)
    )
)

In [ ]:
tp_t = pred_val_temporal_030.filter(
    (F.col("ocurre_emergencia") == 1) &
    (F.col("pred_030") == 1)
).count()

fn_t = pred_val_temporal_030.filter(
    (F.col("ocurre_emergencia") == 1) &
    (F.col("pred_030") == 0)
).count()

fp_t = pred_val_temporal_030.filter(
    (F.col("ocurre_emergencia") == 0) &
    (F.col("pred_030") == 1)
).count()

tn_t = pred_val_temporal_030.filter(
    (F.col("ocurre_emergencia") == 0) &
    (F.col("pred_030") == 0)
).count()

precision_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) else 0
recall_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) else 0

f1_t = (
    2 * precision_t * recall_t /
    (precision_t + recall_t)
    if precision_t + recall_t else 0
)

f2_t = (
    5 * precision_t * recall_t /
    (4 * precision_t + recall_t)
    if 4 * precision_t + recall_t else 0
)

print("TP:", tp_t)
print("FN:", fn_t)
print("FP:", fp_t)
print("TN:", tn_t)

print("Precision:", round(precision_t, 4))
print("Recall:", round(recall_t, 4))
print("F1:", round(f1_t, 4))
print("F2:", round(f2_t, 4))

In [ ]:
pr_auc_temporal = evaluador_pr.evaluate(
    pred_val_temporal
)

print("PR-AUC:", round(pr_auc_temporal, 4))

In [ ]:
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

resultados_temp = []

for t in thresholds:

    df_t = pred_val_temporal.withColumn(
        "pred_t",
        F.when(
            F.col("prob_emergencia") >= t,
            1.0
        ).otherwise(0.0)
    )

    tp = df_t.filter(
        (F.col("ocurre_emergencia") == 1) &
        (F.col("pred_t") == 1)
    ).count()

    fn = df_t.filter(
        (F.col("ocurre_emergencia") == 1) &
        (F.col("pred_t") == 0)
    ).count()

    fp = df_t.filter(
        (F.col("ocurre_emergencia") == 0) &
        (F.col("pred_t") == 1)
    ).count()

    tn = df_t.filter(
        (F.col("ocurre_emergencia") == 0) &
        (F.col("pred_t") == 0)
    ).count()

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if precision + recall else 0
    )

    f2 = (
        5 * precision * recall /
        (4 * precision + recall)
        if 4 * precision + recall else 0
    )

    resultados_temp.append(
        (t, tp, fn, fp, tn, precision, recall, f1, f2)
    )

for r in resultados_temp:
    print(
        f"T={r[0]:.2f} | "
        f"TP={r[1]} FN={r[2]} FP={r[3]} TN={r[4]} | "
        f"P={r[5]:.4f} R={r[6]:.4f} "
        f"F1={r[7]:.4f} F2={r[8]:.4f}"
    )

In [ ]:
df_train_final_mensual = (
    df_modelable_v2
    .filter(F.col("anio").between(2004, 2023))
)

df_test_final_mensual = (
    df_modelable_v2
    .filter(F.col("anio") == 2024)
)

print("Train final:", df_train_final_mensual.count())
print("Test final:", df_test_final_mensual.count())

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    labelCol="ocurre_emergencia",
    featuresCol="features",
    weightCol="peso_clase",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

modelo_lr = lr.fit(train_vec_v2)

pred_val_lr = modelo_lr.transform(val_vec_v2)

In [ ]:
from pyspark.ml.functions import vector_to_array

pred_val_lr = (
    pred_val_lr
    .withColumn(
        "prob_array",
        vector_to_array("probability")
    )
    .withColumn(
        "prob_emergencia",
        F.col("prob_array")[1]
    )
)

In [ ]:
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

resultados_lr = []

for t in thresholds:

    df_t = pred_val_lr.withColumn(
        "pred_t",
        F.when(
            F.col("prob_emergencia") >= t,
            1.0
        ).otherwise(0.0)
    )

    tp = df_t.filter(
        (F.col("ocurre_emergencia") == 1) &
        (F.col("pred_t") == 1)
    ).count()

    fn = df_t.filter(
        (F.col("ocurre_emergencia") == 1) &
        (F.col("pred_t") == 0)
    ).count()

    fp = df_t.filter(
        (F.col("ocurre_emergencia") == 0) &
        (F.col("pred_t") == 1)
    ).count()

    tn = df_t.filter(
        (F.col("ocurre_emergencia") == 0) &
        (F.col("pred_t") == 0)
    ).count()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    f2 = (
        5 * precision * recall / (4 * precision + recall)
        if 4 * precision + recall else 0
    )

    resultados_lr.append(
        (t, tp, fn, fp, tn, precision, recall, f1, f2)
    )

for r in resultados_lr:
    print(
        f"T={r[0]:.2f} | "
        f"TP={r[1]} FN={r[2]} FP={r[3]} TN={r[4]} | "
        f"P={r[5]:.4f} R={r[6]:.4f} "
        f"F1={r[7]:.4f} F2={r[8]:.4f}"
    )

In [ ]:
pr_auc_lr = evaluador_pr.evaluate(pred_val_lr)

print("PR-AUC LR:", round(pr_auc_lr, 4))

In [ ]:
# Ya contiene:
# peso_clase
# peso_temporal
# peso_clase_temp

train_vec_lr_temporal = assembler_mensual.transform(
    df_train_temporal
)

lr_temporal = LogisticRegression(
    labelCol="ocurre_emergencia",
    featuresCol="features",
    weightCol="peso_clase_temp",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

modelo_lr_temporal = lr_temporal.fit(
    train_vec_lr_temporal
)

pred_val_lr_temporal = modelo_lr_temporal.transform(
    val_vec_v2
)

In [ ]:
pred_val_lr_temporal = (
    pred_val_lr_temporal
    .withColumn(
        "prob_array",
        vector_to_array("probability")
    )
    .withColumn(
        "prob_emergencia",
        F.col("prob_array")[1]
    )
)

In [ ]:
thresholds = [0.40, 0.45, 0.50]

In [ ]:
pr_auc_lr_temporal = evaluador_pr.evaluate(
    pred_val_lr_temporal
)

print(
    "PR-AUC LR temporal:",
    round(pr_auc_lr_temporal, 4)
)

In [ ]:
thresholds = [0.40, 0.45, 0.50]

resultados_lr_temporal = []

for t in thresholds:

    df_t = pred_val_lr_temporal.withColumn(
        "pred_t",
        F.when(
            F.col("prob_emergencia") >= t,
            1.0
        ).otherwise(0.0)
    )

    tp = df_t.filter(
        (F.col("ocurre_emergencia") == 1) &
        (F.col("pred_t") == 1)
    ).count()

    fn = df_t.filter(
        (F.col("ocurre_emergencia") == 1) &
        (F.col("pred_t") == 0)
    ).count()

    fp = df_t.filter(
        (F.col("ocurre_emergencia") == 0) &
        (F.col("pred_t") == 1)
    ).count()

    tn = df_t.filter(
        (F.col("ocurre_emergencia") == 0) &
        (F.col("pred_t") == 0)
    ).count()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if precision + recall else 0
    )

    f2 = (
        5 * precision * recall /
        (4 * precision + recall)
        if 4 * precision + recall else 0
    )

    resultados_lr_temporal.append(
        (t, tp, fn, fp, tn, precision, recall, f1, f2)
    )

for r in resultados_lr_temporal:
    print(
        f"T={r[0]:.2f} | "
        f"TP={r[1]} FN={r[2]} FP={r[3]} TN={r[4]} | "
        f"P={r[5]:.4f} R={r[6]:.4f} "
        f"F1={r[7]:.4f} F2={r[8]:.4f}"
    )

In [ ]:
conteos_final_lr = {
    row["ocurre_emergencia"]: row["count"]
    for row in (
        df_train_final_mensual
        .groupBy("ocurre_emergencia")
        .count()
        .collect()
    )
}

n_final_lr = sum(conteos_final_lr.values())

peso_no_final_lr = (
    n_final_lr /
    (2 * conteos_final_lr[0.0])
)

peso_si_final_lr = (
    n_final_lr /
    (2 * conteos_final_lr[1.0])
)

print("Peso NO:", peso_no_final_lr)
print("Peso SI:", peso_si_final_lr)

In [ ]:
df_train_final_lr = (
    df_train_final_mensual
    .withColumn(
        "peso_clase",
        F.when(
            F.col("ocurre_emergencia") == 1,
            F.lit(peso_si_final_lr)
        ).otherwise(
            F.lit(peso_no_final_lr)
        )
    )
)

In [ ]:
train_final_lr_vec = assembler_mensual.transform(
    df_train_final_lr
)

test_final_lr_vec = assembler_mensual.transform(
    df_test_final_mensual
)

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr_final = LogisticRegression(
    labelCol="ocurre_emergencia",
    featuresCol="features",
    weightCol="peso_clase",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

modelo_lr_final = lr_final.fit(
    train_final_lr_vec
)

pred_test_lr = modelo_lr_final.transform(
    test_final_lr_vec
)

In [ ]:
pred_test_lr = (
    pred_test_lr
    .withColumn(
        "prob_array",
        vector_to_array("probability")
    )
    .withColumn(
        "prob_emergencia",
        F.col("prob_array")[1]
    )
    .withColumn(
        "pred_final",
        F.when(
            F.col("prob_emergencia") >= 0.50,
            1.0
        ).otherwise(0.0)
    )
)

In [ ]:
tp = pred_test_lr.filter(
    (F.col("ocurre_emergencia") == 1) &
    (F.col("pred_final") == 1)
).count()

fn = pred_test_lr.filter(
    (F.col("ocurre_emergencia") == 1) &
    (F.col("pred_final") == 0)
).count()

fp = pred_test_lr.filter(
    (F.col("ocurre_emergencia") == 0) &
    (F.col("pred_final") == 1)
).count()

tn = pred_test_lr.filter(
    (F.col("ocurre_emergencia") == 0) &
    (F.col("pred_final") == 0)
).count()

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0

f1 = (
    2 * precision * recall /
    (precision + recall)
    if precision + recall else 0
)

f2 = (
    5 * precision * recall /
    (4 * precision + recall)
    if 4 * precision + recall else 0
)

print("TP:", tp)
print("FN:", fn)
print("FP:", fp)
print("TN:", tn)

print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1:", round(f1, 4))
print("F2:", round(f2, 4))

In [ ]:
pr_auc_test_lr = evaluador_pr.evaluate(
    pred_test_lr
)

print(
    "PR-AUC TEST:",
    round(pr_auc_test_lr, 4)
)

In [ ]:
df_train_2026_mensual = (
    df_features_mensual_v2
    .filter(
        (F.col("anio").between(2004, 2024)) &
        F.col("ocurre_emergencia").isNotNull()
    )
)

print("Filas entrenamiento 2026:", df_train_2026_mensual.count())

In [ ]:
conteos_2026_lr = {
    row["ocurre_emergencia"]: row["count"]
    for row in (
        df_train_2026_mensual
        .groupBy("ocurre_emergencia")
        .count()
        .collect()
    )
}

n_2026_lr = sum(conteos_2026_lr.values())

peso_no_2026_lr = (
    n_2026_lr /
    (2 * conteos_2026_lr[0.0])
)

peso_si_2026_lr = (
    n_2026_lr /
    (2 * conteos_2026_lr[1.0])
)

print("Peso NO:", peso_no_2026_lr)
print("Peso SI:", peso_si_2026_lr)

In [ ]:
df_train_2026_mensual = (
    df_train_2026_mensual
    .withColumn(
        "peso_clase",
        F.when(
            F.col("ocurre_emergencia") == 1,
            F.lit(peso_si_2026_lr)
        ).otherwise(
            F.lit(peso_no_2026_lr)
        )
    )
)

In [ ]:
df_train_2026_mensual = (
    df_train_2026_mensual
    .withColumn(
        "peso_clase",
        F.when(
            F.col("ocurre_emergencia") == 1.0,
            F.lit(peso_si_2026_lr)
        ).otherwise(
            F.lit(peso_no_2026_lr)
        )
    )
)

In [ ]:
train_2026_lr_vec = assembler_mensual.transform(
    df_train_2026_mensual
)

pred_2026_lr_vec = assembler_mensual.transform(
    df_pred_2026_v2
)

print("Train:", train_2026_lr_vec.count())
print("A predecir:", pred_2026_lr_vec.count())

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr_2026 = LogisticRegression(
    labelCol="ocurre_emergencia",
    featuresCol="features",
    weightCol="peso_clase",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

modelo_lr_2026 = lr_2026.fit(
    train_2026_lr_vec
)

In [ ]:
pred_2026 = modelo_lr_2026.transform(
    pred_2026_lr_vec
)

pred_2026 = (
    pred_2026
    .withColumn(
        "prob_array",
        vector_to_array("probability")
    )
    .withColumn(
        "prob_emergencia_2026",
        F.col("prob_array")[1]
    )
    .withColumn(
        "pred_emergencia_2026",
        F.when(
            F.col("prob_emergencia_2026") >= 0.50,
            1
        ).otherwise(0)
    )
)

In [ ]:
print("Predicciones:", pred_2026.count())

pred_2026.groupBy(
    "pred_emergencia_2026"
).count().show()

In [ ]:
(
    pred_2026
    .groupBy("mes_num")
    .agg(
        F.sum("pred_emergencia_2026")
            .alias("distritos_con_alerta"),

        F.round(
            F.avg("prob_emergencia_2026"),
            4
        ).alias("probabilidad_media")
    )
    .orderBy("mes_num")
    .show(12)
)

In [ ]:
pred_2026 = pred_2026.withColumnRenamed(
    "prob_emergencia_2026",
    "score_ocurrencia_2026"
)

priorizacion_distrital_2026 = (
    pred_2026
    .groupBy(
        "ubigeo",
        "provincia",
        "distrito"
    )
    .agg(
        # Cuántos meses de 2026 presentan alerta
        F.sum("pred_emergencia_2026")
            .alias("meses_con_alerta_2026"),

        # Score promedio de los 12 meses
        F.avg("score_ocurrencia_2026")
            .alias("score_promedio_2026"),

        # Mes de mayor señal
        F.max("score_ocurrencia_2026")
            .alias("score_maximo_2026")
    )
)

In [ ]:
print(
    "Distritos:",
    priorizacion_distrital_2026.count()
)

In [ ]:
(
    priorizacion_distrital_2026
    .select(
        "provincia",
        "distrito",
        "meses_con_alerta_2026",
        F.round(
            "score_promedio_2026", 4
        ).alias("score_promedio_2026"),
        F.round(
            "score_maximo_2026", 4
        ).alias("score_maximo_2026")
    )
    .orderBy(
        F.desc("meses_con_alerta_2026"),
        F.desc("score_promedio_2026")
    )
    .show(84, truncate=False)
)

In [ ]:
impacto_reciente_distrito = (
    df_impacto_eventos
    .filter(F.col("anio").between(2021, 2025))
    .groupBy("ubigeo")
    .agg(
        F.avg("score_impacto")
            .alias("impacto_promedio_reciente"),

        F.max("score_impacto")
            .alias("impacto_maximo_reciente"),

        F.count("*")
            .alias("meses_historicos_con_emergencia")
    )
)

print(
    "Distritos con antecedentes recientes:",
    impacto_reciente_distrito.count()
)

In [ ]:
priorizacion_v2 = (
    priorizacion_distrital_2026
    .join(
        impacto_reciente_distrito,
        on="ubigeo",
        how="left"
    )
    .fillna({
        "impacto_promedio_reciente": 0.0,
        "impacto_maximo_reciente": 0.0,
        "meses_historicos_con_emergencia": 0
    })
)

In [ ]:
df_pob_2025_ubigeo = (
    df_inei
    .filter(F.col("anio") == 2025)
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
)

In [ ]:
priorizacion_v2 = (
    priorizacion_v2
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .join(
        df_inei_norm
        .filter(F.col("anio") == 2025)
        .select(
            "provincia_key",
            "distrito_key",
            F.col("poblacion").alias("poblacion_2025")
        ),
        on=["provincia_key", "distrito_key"],
        how="left"
    )
)

In [ ]:
(
    priorizacion_v2
    .select(
        "provincia",
        "distrito",
        "meses_con_alerta_2026",
        F.round("score_promedio_2026", 4)
            .alias("score_ocurrencia"),
        F.round("impacto_promedio_reciente", 4)
            .alias("impacto_promedio"),
        F.round("impacto_maximo_reciente", 4)
            .alias("impacto_maximo"),
        "poblacion_2025"
    )
    .orderBy(
        F.desc("meses_con_alerta_2026"),
        F.desc("score_promedio_2026")
    )
    .show(30, truncate=False)
)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

w_ranking = Window.orderBy(
    F.desc("meses_con_alerta_2026"),
    F.desc("score_promedio_2026"),
    F.desc("impacto_promedio_reciente"),
    F.desc("poblacion_2025")
)

ranking_2026 = (
    priorizacion_v2
    .withColumn(
        "ranking_prioridad_2026",
        F.row_number().over(w_ranking)
    )
)

In [ ]:
(
    ranking_2026
    .select(
        "ranking_prioridad_2026",
        "provincia",
        "distrito",
        "meses_con_alerta_2026",
        F.round("score_promedio_2026", 4)
            .alias("score_ocurrencia"),
        F.round("impacto_promedio_reciente", 4)
            .alias("impacto_historico"),
        "poblacion_2025"
    )
    .orderBy("ranking_prioridad_2026")
    .show(25, truncate=False)
)

In [ ]:
w_ocurrencia = Window.orderBy(
    F.desc("meses_con_alerta_2026"),
    F.desc("score_promedio_2026")
)

w_impacto = Window.orderBy(
    F.desc("impacto_promedio_reciente")
)

w_poblacion = Window.orderBy(
    F.desc("poblacion_2025")
)

ranking_multicriterio = (
    priorizacion_v2
    .withColumn(
        "ranking_ocurrencia",
        F.row_number().over(w_ocurrencia)
    )
    .withColumn(
        "ranking_impacto",
        F.row_number().over(w_impacto)
    )
    .withColumn(
        "ranking_poblacion",
        F.row_number().over(w_poblacion)
    )
)

In [ ]:
(
    ranking_multicriterio
    .select(
        "ranking_ocurrencia",
        "ranking_impacto",
        "ranking_poblacion",
        "provincia",
        "distrito",
        "meses_con_alerta_2026",
        F.round("score_promedio_2026", 4)
            .alias("score_ocurrencia"),
        F.round("impacto_promedio_reciente", 4)
            .alias("impacto_historico"),
        "poblacion_2025"
    )
    .orderBy("ranking_ocurrencia")
    .show(30, truncate=False)
)

In [ ]:
df_clima_ranking = (
    df_clima_2026_contexto
    .select(
        "ubigeo",
        "precipitacion_ene_ago_2026_mm",
        "promedio_historico_ene_ago_mm",
        "anomalia_precipitacion_pct",
        "z_precipitacion_2026"
    )
)

tabla_final_2026 = (
    ranking_multicriterio
    .join(
        df_clima_ranking,
        on="ubigeo",
        how="left"
    )
)

In [ ]:
tabla_final_2026 = (
    tabla_final_2026
    .withColumn(
        "contexto_climatico_2026",
        F.when(
            F.col("z_precipitacion_2026") >= 2,
            "MUY_SUPERIOR_A_LO_NORMAL"
        )
        .when(
            F.col("z_precipitacion_2026") >= 1,
            "SUPERIOR_A_LO_NORMAL"
        )
        .when(
            F.col("z_precipitacion_2026") <= -1,
            "INFERIOR_A_LO_NORMAL"
        )
        .otherwise("DENTRO_DE_LO_NORMAL")
    )
)

In [ ]:
(
    tabla_final_2026
    .select(
        "ranking_ocurrencia",
        "ranking_impacto",
        "ranking_poblacion",
        "provincia",
        "distrito",
        "meses_con_alerta_2026",
        F.round("score_promedio_2026", 4).alias("score_ocurrencia"),
        F.round("impacto_promedio_reciente", 4).alias("impacto_historico"),
        "poblacion_2025",
        F.round("anomalia_precipitacion_pct", 2).alias("anomalia_lluvia_pct"),
        F.round("z_precipitacion_2026", 2).alias("z_lluvia_2026"),
        "contexto_climatico_2026"
    )
    .orderBy("ranking_ocurrencia")
    .show(30, truncate=False)
)

In [ ]:
print("Filas finales:", tabla_final_2026.count())

print(
    "Con clima 2026:",
    tabla_final_2026
    .filter(F.col("z_precipitacion_2026").isNotNull())
    .count()
)

In [ ]:
ruta_tabla_final = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/"
    "priorizacion_multicriterio_2026"
)

(
    tabla_final_2026
    .write
    .mode("overwrite")
    .parquet(ruta_tabla_final)
)

print("Tabla final guardada.")

In [ ]:
ruta_pred_mensual = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/"
    "predicciones_mensuales_2026"
)

(
    pred_2026
    .select(
        "ubigeo",
        "provincia",
        "distrito",
        "mes_num",
        "score_ocurrencia_2026",
        "pred_emergencia_2026"
    )
    .write
    .mode("overwrite")
    .parquet(ruta_pred_mensual)
)

print("Predicciones mensuales guardadas.")

In [ ]:
ruta_modelo_final = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/models/"
    "logistic_regression_mensual_2026"
)

(
    modelo_lr_2026
    .write()
    .overwrite()
    .save(ruta_modelo_final)
)

print("Modelo final guardado.")

In [ ]:
df_check = spark.read.parquet(ruta_tabla_final)

print("Filas:", df_check.count())
print(
    "Distritos únicos:",
    df_check.select("ubigeo").distinct().count()
)

In [ ]:
print("Spark master:", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

print(
    "Particiones df maestro:",
    df_maestro_final.rdd.getNumPartitions()
)

print(
    "Particiones panel mensual:",
    df_panel_mensual.rdd.getNumPartitions()
)

In [ ]:
spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/dataset_maestro_2003_2025"
).filter(
    F.col("anio") == 2025
).explain("formatted")

In [ ]:
app_id = spark.sparkContext.applicationId
print(app_id)

In [ ]:
import subprocess
import re
import time

app_id = spark.sparkContext.applicationId

def obtener_recursos_yarn():
    salida = subprocess.check_output(
        [
            "yarn",
            "application",
            "-status",
            app_id
        ],
        text=True,
        stderr=subprocess.STDOUT
    )

    patron = (
        r"Aggregate Resource Allocation\s*:\s*"
        r"(\d+)\s+MB-seconds,\s*"
        r"(\d+)\s+vcore-seconds"
    )

    match = re.search(patron, salida)

    if not match:
        raise RuntimeError(
            "No se pudo leer Aggregate Resource Allocation"
        )

    return {
        "mb_seconds": int(match.group(1)),
        "vcore_seconds": int(match.group(2))
    }

In [ ]:
print(obtener_recursos_yarn())

In [ ]:
def benchmark_pipeline_recursos(factor):

    recursos_antes = obtener_recursos_yarn()

    inicio = time.perf_counter()

    pipeline_escalabilidad(factor)

    fin = time.perf_counter()

    recursos_despues = obtener_recursos_yarn()

    tiempo = fin - inicio

    mb_seconds = (
        recursos_despues["mb_seconds"]
        - recursos_antes["mb_seconds"]
    )

    vcore_seconds = (
        recursos_despues["vcore_seconds"]
        - recursos_antes["vcore_seconds"]
    )

    resultado = {
        "factor": factor,
        "filas_entrada": 2853 * factor,
        "tiempo_s": tiempo,
        "mb_seconds": mb_seconds,
        "vcore_seconds": vcore_seconds
    }

    print(resultado)

    return resultado

In [ ]:
r1 = benchmark_pipeline_recursos(1)
r10 = benchmark_pipeline_recursos(10)
r100 = benchmark_pipeline_recursos(100)

In [ ]:
resultados_escalabilidad = [r1, r10, r100]

for r in resultados_escalabilidad:
    print(
        f"{r['factor']}x | "
        f"filas={r['filas_entrada']} | "
        f"tiempo={r['tiempo_s']:.3f}s | "
        f"MB-s={r['mb_seconds']} | "
        f"vCore-s={r['vcore_seconds']}"
    )

In [ ]:
df_eda_anual = (
    df_prueba_yarn
    .groupBy("anio")
    .agg(
        F.count("*").alias("emergencias"),
        F.sum("afectados").alias("afectados"),
        F.sum("damnificados").alias("damnificados")
    )
    .orderBy("anio")
)

df_eda_anual.show(30)

In [ ]:
pdf_eda_anual = df_eda_anual.toPandas()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    pdf_eda_anual["anio"],
    pdf_eda_anual["emergencias"],
    marker="o"
)

plt.title(
    "Evolución anual de emergencias en La Libertad (2003–2025)"
)

plt.xlabel("Año")
plt.ylabel("Número de emergencias")

plt.grid(True, alpha=0.3)
plt.xticks(
    pdf_eda_anual["anio"],
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
df_eda_peligros = (
    df_prueba_yarn
    .groupBy("peligro")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

df_eda_peligros.show(10, truncate=False)

In [ ]:
pdf_eda_peligros = df_eda_peligros.toPandas()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 6))

plt.barh(
    pdf_eda_peligros["peligro"],
    pdf_eda_peligros["count"]
)

plt.title(
    "Principales tipos de emergencia registrados en La Libertad (2003–2025)"
)

plt.xlabel("Número de emergencias")
plt.ylabel("Tipo de peligro")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
df_eda_meses = (
    df_prueba_yarn
    .withColumn(
        "mes_num",
        F.month("fecha_emergencia")
    )
    .groupBy("mes_num")
    .count()
    .orderBy("mes_num")
)

df_eda_meses.show(12)

In [ ]:
pdf_eda_meses = df_eda_meses.toPandas()

nombres_meses = {
    1: "Ene",
    2: "Feb",
    3: "Mar",
    4: "Abr",
    5: "May",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Sep",
    10: "Oct",
    11: "Nov",
    12: "Dic"
}

pdf_eda_meses["mes"] = (
    pdf_eda_meses["mes_num"]
    .map(nombres_meses)
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.bar(
    pdf_eda_meses["mes"],
    pdf_eda_meses["count"]
)

plt.title(
    "Distribución mensual histórica de emergencias en La Libertad (2003–2025)"
)

plt.xlabel("Mes")
plt.ylabel("Número de emergencias")

plt.tight_layout()
plt.show()

In [ ]:
import re
import unicodedata
from pyspark.sql import functions as F

# ---------------------------------------------------------
# 1. Función para normalizar provincia/distrito
# ---------------------------------------------------------

def normalizar_clave(columna):
    return F.translate(
        F.upper(F.trim(columna)),
        "ÁÉÍÓÚÜÑ",
        "AEIOUUN"
    )


# ---------------------------------------------------------
# 2. CHIRPS histórico = grilla completa distrito-mes
# ---------------------------------------------------------

ruta_chirps = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/raw/chirps/historico/*.csv"
)

df_chirps = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_chirps)
)

df_panel_base = (
    df_chirps
    .withColumn(
        "mes_num",
        F.col("mes").cast("int")
    )
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .select(
        "ubigeo",
        "provincia",
        "distrito",
        "provincia_key",
        "distrito_key",
        "anio",
        "mes_num",
        "precipitacion_total_mm",
        "precipitacion_promedio_diaria_mm",
        "precipitacion_maxima_diaria_mm"
    )
)


# ---------------------------------------------------------
# 3. Emergencias SINPAD agregadas por distrito-mes
# ---------------------------------------------------------

df_eventos_mensual = (
    df_prueba_yarn
    .withColumn(
        "mes_num",
        F.month("fecha_emergencia")
    )
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .groupBy(
        "provincia_key",
        "distrito_key",
        "anio",
        "mes_num"
    )
    .agg(
        F.count("*")
            .alias("cantidad_emergencias_mes"),

        F.countDistinct("peligro")
            .alias("tipos_peligro_mes"),

        F.sum("afectados")
            .alias("afectados_mes"),

        F.sum("damnificados")
            .alias("damnificados_mes"),

        F.sum("fallecidos")
            .alias("fallecidos_mes"),

        F.sum("desaparecidos")
            .alias("desaparecidos_mes"),

        F.sum("viviendas_afectadas")
            .alias("viviendas_afectadas_mes"),

        F.sum("viviendas_destruidas")
            .alias("viviendas_destruidas_mes")
    )
)


# ---------------------------------------------------------
# 4. Join: todos los distrito-mes, incluso sin emergencia
# ---------------------------------------------------------

df_panel_mensual = (
    df_panel_base
    .join(
        df_eventos_mensual,
        on=[
            "provincia_key",
            "distrito_key",
            "anio",
            "mes_num"
        ],
        how="left"
    )
    .fillna({
        "cantidad_emergencias_mes": 0,
        "tipos_peligro_mes": 0,
        "afectados_mes": 0,
        "damnificados_mes": 0,
        "fallecidos_mes": 0,
        "desaparecidos_mes": 0,
        "viviendas_afectadas_mes": 0,
        "viviendas_destruidas_mes": 0
    })
)

print("Filas panel:", df_panel_mensual.count())
print(
    "Distritos:",
    df_panel_mensual.select("ubigeo").distinct().count()
)
print(
    "Años:",
    df_panel_mensual.select("anio").distinct().count()
)

In [ ]:
ruta_panel = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/"
    "panel_distrito_mes_2003_2025"
)

(
    df_panel_mensual
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(ruta_panel)
)

print("Panel mensual guardado en HDFS.")

In [ ]:
import math
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Índice temporal continuo
df_modelo_mensual = (
    df_panel_mensual
    .withColumn(
        "periodo_idx",
        F.col("anio") * 12 + F.col("mes_num")
    )
)

# Ventana por distrito
w = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
)

w3 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
    .rowsBetween(-3, -1)
)

w6 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
    .rowsBetween(-6, -1)
)

w12 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
    .rowsBetween(-12, -1)
)

In [ ]:
df_features_mensual = (
    df_modelo_mensual

    # ---------------------------
    # FRECUENCIA HISTÓRICA
    # ---------------------------
    .withColumn(
        "emergencias_mes_anterior",
        F.lag("cantidad_emergencias_mes", 1).over(w)
    )
    .withColumn(
        "emergencias_3_meses",
        F.sum("cantidad_emergencias_mes").over(w3)
    )
    .withColumn(
        "emergencias_6_meses",
        F.sum("cantidad_emergencias_mes").over(w6)
    )
    .withColumn(
        "emergencias_12_meses",
        F.sum("cantidad_emergencias_mes").over(w12)
    )
    .withColumn(
        "emergencias_mismo_mes_anio_anterior",
        F.lag("cantidad_emergencias_mes", 12).over(w)
    )

    # ---------------------------
    # IMPACTO HISTÓRICO
    # ---------------------------
    .withColumn(
        "afectados_12_meses",
        F.sum("afectados_mes").over(w12)
    )
    .withColumn(
        "damnificados_12_meses",
        F.sum("damnificados_mes").over(w12)
    )
    .withColumn(
        "fallecidos_12_meses",
        F.sum("fallecidos_mes").over(w12)
    )
    .withColumn(
        "viviendas_afectadas_12_meses",
        F.sum("viviendas_afectadas_mes").over(w12)
    )
    .withColumn(
        "viviendas_destruidas_12_meses",
        F.sum("viviendas_destruidas_mes").over(w12)
    )

    # ---------------------------
    # CLIMA
    # ---------------------------
    .withColumn(
        "lluvia_mes_anterior",
        F.lag("precipitacion_total_mm", 1).over(w)
    )
    .withColumn(
        "lluvia_3_meses",
        F.sum("precipitacion_total_mm").over(w3)
    )

    # ---------------------------
    # ESTACIONALIDAD
    # ---------------------------
    .withColumn(
        "mes_sin",
        F.sin(
            2 * F.lit(math.pi) * F.col("mes_num") / 12
        )
    )
    .withColumn(
        "mes_cos",
        F.cos(
            2 * F.lit(math.pi) * F.col("mes_num") / 12
        )
    )

    # ---------------------------
    # TARGET: mismo mes del año siguiente
    # ---------------------------
    .withColumn(
        "emergencias_siguiente_anio",
        F.lead("cantidad_emergencias_mes", 12).over(w)
    )
    .withColumn(
        "ocurre_emergencia",
        F.when(
            F.col("emergencias_siguiente_anio") > 0,
            F.lit(1.0)
        )
        .when(
            F.col("emergencias_siguiente_anio") == 0,
            F.lit(0.0)
        )
    )
)

In [ ]:
df_modelable = (
    df_features_mensual
    .filter(
        (F.col("anio") >= 2004) &
        (F.col("anio") <= 2024) &
        F.col("ocurre_emergencia").isNotNull()
    )
)

print("Filas modelables:", df_modelable.count())

df_modelable.groupBy(
    "ocurre_emergencia"
).count().show()

In [ ]:
df_entrada_2026 = (
    df_features_mensual
    .filter(F.col("anio") == 2025)
)

print("Filas 2025:", df_entrada_2026.count())

print(
    "Targets desconocidos:",
    df_entrada_2026
    .filter(
        F.col("emergencias_siguiente_anio").isNull()
    )
    .count()
)

In [ ]:
df_train_final = (
    df_modelable
    .filter(F.col("anio").between(2004, 2023))
)

df_test_final = (
    df_modelable
    .filter(F.col("anio") == 2024)
)

print("TRAIN:", df_train_final.count())
print("TEST:", df_test_final.count())

print("TRAIN")
df_train_final.groupBy("ocurre_emergencia").count().show()

print("TEST")
df_test_final.groupBy("ocurre_emergencia").count().show()

In [ ]:
features_mensuales = [
    "cantidad_emergencias_mes",
    "tipos_peligro_mes",

    "emergencias_mes_anterior",
    "emergencias_3_meses",
    "emergencias_6_meses",
    "emergencias_12_meses",
    "emergencias_mismo_mes_anio_anterior",

    "afectados_12_meses",
    "damnificados_12_meses",
    "fallecidos_12_meses",
    "viviendas_afectadas_12_meses",
    "viviendas_destruidas_12_meses",

    "precipitacion_total_mm",
    "precipitacion_promedio_diaria_mm",
    "precipitacion_maxima_diaria_mm",
    "lluvia_mes_anterior",
    "lluvia_3_meses",

    "mes_sin",
    "mes_cos"
]

In [ ]:
conteos_train = {
    row["ocurre_emergencia"]: row["count"]
    for row in (
        df_train_final
        .groupBy("ocurre_emergencia")
        .count()
        .collect()
    )
}

n_train = sum(conteos_train.values())

peso_no = n_train / (2 * conteos_train[0.0])
peso_si = n_train / (2 * conteos_train[1.0])

print("Peso NO:", peso_no)
print("Peso SI:", peso_si)

In [ ]:
df_train_final = (
    df_train_final
    .withColumn(
        "peso_clase",
        F.when(
            F.col("ocurre_emergencia") == 1.0,
            F.lit(peso_si)
        ).otherwise(
            F.lit(peso_no)
        )
    )
)

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler_mensual = VectorAssembler(
    inputCols=features_mensuales,
    outputCol="features",
    handleInvalid="keep"
)

train_vec = assembler_mensual.transform(df_train_final)
test_vec = assembler_mensual.transform(df_test_final)

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr_final = LogisticRegression(
    labelCol="ocurre_emergencia",
    featuresCol="features",
    weightCol="peso_clase",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

modelo_lr_final = lr_final.fit(train_vec)

pred_test = modelo_lr_final.transform(test_vec)

In [ ]:
from pyspark.ml.functions import vector_to_array

pred_test = (
    pred_test
    .withColumn(
        "prob_array",
        vector_to_array("probability")
    )
    .withColumn(
        "score_emergencia",
        F.col("prob_array")[1]
    )
    .withColumn(
        "pred_final",
        F.when(
            F.col("score_emergencia") >= 0.50,
            1.0
        ).otherwise(0.0)
    )
)

In [ ]:
tp = pred_test.filter(
    (F.col("ocurre_emergencia") == 1) &
    (F.col("pred_final") == 1)
).count()

fn = pred_test.filter(
    (F.col("ocurre_emergencia") == 1) &
    (F.col("pred_final") == 0)
).count()

fp = pred_test.filter(
    (F.col("ocurre_emergencia") == 0) &
    (F.col("pred_final") == 1)
).count()

tn = pred_test.filter(
    (F.col("ocurre_emergencia") == 0) &
    (F.col("pred_final") == 0)
).count()

precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

f2 = (
    5 * precision * recall /
    (4 * precision + recall)
    if (4 * precision + recall) else 0
)

accuracy = (tp + tn) / (tp + tn + fp + fn)

print("TP:", tp)
print("FN:", fn)
print("FP:", fp)
print("TN:", tn)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1:", round(f1, 4))
print("F2:", round(f2, 4))

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluador_pr = BinaryClassificationEvaluator(
    labelCol="ocurre_emergencia",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

pr_auc = evaluador_pr.evaluate(pred_test)

print("PR-AUC:", round(pr_auc, 4))

In [ ]:
df_train_2026 = (
    df_modelable
    .filter(F.col("anio").between(2004, 2024))
)

print("TRAIN 2026:", df_train_2026.count())

df_train_2026.groupBy(
    "ocurre_emergencia"
).count().show()

In [ ]:
conteos_2026 = {
    row["ocurre_emergencia"]: row["count"]
    for row in (
        df_train_2026
        .groupBy("ocurre_emergencia")
        .count()
        .collect()
    )
}

n_2026 = sum(conteos_2026.values())

peso_no_2026 = (
    n_2026 /
    (2 * conteos_2026[0.0])
)

peso_si_2026 = (
    n_2026 /
    (2 * conteos_2026[1.0])
)

print("Peso NO:", peso_no_2026)
print("Peso SI:", peso_si_2026)

In [ ]:
df_train_2026 = (
    df_train_2026
    .withColumn(
        "peso_clase",
        F.when(
            F.col("ocurre_emergencia") == 1.0,
            F.lit(peso_si_2026)
        ).otherwise(
            F.lit(peso_no_2026)
        )
    )
)

In [ ]:
ruta_sinpad_2026 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/raw/sinpad/2026/"
    "sinpad_la_libertad_2026.geojson"
)

df_geo_2026 = (
    spark.read
    .option("multiline", True)
    .json(ruta_sinpad_2026)
)

df_geo_2026.printSchema()

In [ ]:
df_sinpad_2026_raw = (
    df_geo_2026
    .select(F.explode("features").alias("feature"))
    .select("feature.properties.*")
)

print("Filas:", df_sinpad_2026_raw.count())

print(
    "SINPAD únicos:",
    df_sinpad_2026_raw
    .select("IDE_SINPAD")
    .distinct()
    .count()
)

df_sinpad_2026_raw.select(
    "IDE_SINPAD",
    "FECHA",
    "ANHO",
    "MES",
    "COD_UBIGEO",
    "DEPARTAMENTO",
    "PROVINCIA",
    "DISTRITO",
    "FENOMENO"
).show(10, truncate=False)

In [ ]:
(
    df_sinpad_2026_raw
    .groupBy("MES")
    .count()
    .orderBy(F.desc("count"))
    .show(20, truncate=False)
)

In [ ]:
print(df_sinpad_2026_raw.columns)


In [ ]:
df_sinpad_2026_raw.printSchema()

In [ ]:
df_sinpad_2026 = (
    df_sinpad_2026_raw

    # Identificadores y fecha
    .withColumn(
        "codigo_sinpad",
        F.col("IDE_SINPAD").cast("long")
    )
    .withColumn(
        "fecha_emergencia",
        F.to_date(
            F.from_unixtime(
                (F.col("FECHA") / 1000).cast("long")
            )
        )
    )
    .withColumn(
        "anio",
        F.col("ANHO").cast("int")
    )
    .withColumn(
        "mes_num",
        F.month("fecha_emergencia")
    )
    .withColumn(
        "ubigeo",
        F.col("COD_UBIGEO").cast("string")
    )

    # Limpiar nombres geográficos
    .withColumn(
        "departamento",
        F.trim(
            F.regexp_replace(
                F.col("DEPARTAMENTO"),
                r"^DEPA\.\s*",
                ""
            )
        )
    )
    .withColumn(
        "provincia",
        F.trim(
            F.regexp_replace(
                F.col("PROVINCIA"),
                r"^PROV\.\s*",
                ""
            )
        )
    )
    .withColumn(
        "distrito",
        F.trim(
            F.regexp_replace(
                F.col("DISTRITO"),
                r"^DIST\.\s*",
                ""
            )
        )
    )

    # Peligro
    .withColumn(
        "peligro",
        F.upper(F.trim(F.col("FENOMENO")))
    )

    # Impacto
    .withColumn("afectados", F.col("SAFECTA").cast("long"))
    .withColumn("damnificados", F.col("SDAMNI").cast("long"))
    .withColumn("fallecidos", F.col("SFALLE").cast("long"))
    .withColumn("desaparecidos", F.col("SDESA").cast("long"))
    .withColumn("lesionados", F.col("SLESI").cast("long"))
    .withColumn(
        "viviendas_afectadas",
        F.col("SAFECTAVIVI").cast("long")
    )
    .withColumn(
        "viviendas_destruidas",
        F.col("SDESTRUVIVI").cast("long")
    )

    .select(
        "codigo_sinpad",
        "fecha_emergencia",
        "anio",
        "mes_num",
        "ubigeo",
        "departamento",
        "provincia",
        "distrito",
        "peligro",
        "afectados",
        "damnificados",
        "fallecidos",
        "desaparecidos",
        "lesionados",
        "viviendas_afectadas",
        "viviendas_destruidas"
    )
)

In [ ]:
print("Filas:", df_sinpad_2026.count())

print(
    "SINPAD únicos:",
    df_sinpad_2026
    .select("codigo_sinpad")
    .distinct()
    .count()
)

print(
    "Fechas nulas:",
    df_sinpad_2026
    .filter(F.col("fecha_emergencia").isNull())
    .count()
)

print(
    "UBIGEO nulos:",
    df_sinpad_2026
    .filter(F.col("ubigeo").isNull())
    .count()
)

print(
    "Distritos distintos:",
    df_sinpad_2026
    .select("ubigeo")
    .distinct()
    .count()
)

In [ ]:
df_sinpad_2026.orderBy(
    "fecha_emergencia"
).show(15, truncate=False)

In [ ]:
(
    df_sinpad_2026
    .groupBy("mes_num")
    .count()
    .orderBy("mes_num")
    .show()
)

In [ ]:
ruta_sinpad_2026_processed = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/"
    "sinpad_la_libertad_2026"
)

(
    df_sinpad_2026
    .write
    .mode("overwrite")
    .partitionBy("mes_num")
    .parquet(ruta_sinpad_2026_processed)
)

print("SINPAD 2026 procesado guardado.")

In [ ]:
meses_map = (
    F.when(F.col("MES") == "ENERO", 1)
     .when(F.col("MES") == "FEBRERO", 2)
     .when(F.col("MES") == "MARZO", 3)
     .when(F.col("MES") == "ABRIL", 4)
     .when(F.col("MES") == "MAYO", 5)
     .when(F.col("MES") == "JUNIO", 6)
     .when(F.col("MES") == "JULIO", 7)
     .when(F.col("MES") == "AGOSTO", 8)
     .when(F.col("MES") == "SETIEMBRE", 9)
     .when(F.col("MES") == "SEPTIEMBRE", 9)
     .when(F.col("MES") == "OCTUBRE", 10)
     .when(F.col("MES") == "NOVIEMBRE", 11)
     .when(F.col("MES") == "DICIEMBRE", 12)
)

df_control_mes = (
    df_sinpad_2026_raw
    .withColumn("mes_declarado", meses_map)
    .withColumn(
        "fecha_convertida",
        F.to_date(
            F.from_unixtime(
                (F.col("FECHA") / 1000).cast("long")
            )
        )
    )
    .withColumn(
        "mes_fecha",
        F.month("fecha_convertida")
    )
)

print(
    "Registros con MES distinto a FECHA:",
    df_control_mes
    .filter(F.col("mes_declarado") != F.col("mes_fecha"))
    .count()
)

(
    df_control_mes
    .filter(F.col("mes_declarado") != F.col("mes_fecha"))
    .select(
        "IDE_SINPAD",
        "fecha_convertida",
        "MES",
        "mes_declarado",
        "mes_fecha",
        "PROVINCIA",
        "DISTRITO",
        "FENOMENO"
    )
    .orderBy("fecha_convertida")
    .show(50, truncate=False)
)

In [ ]:
df_real_2026_agg = (
    df_sinpad_2026
    .filter(F.col("mes_num").between(1, 5))
    .groupBy(
        "ubigeo",
        "mes_num"
    )
    .agg(
        F.count("*")
            .alias("emergencias_reales_2026"),

        F.sum(
            F.coalesce(F.col("afectados"), F.lit(0))
        ).alias("afectados_reales_2026"),

        F.sum(
            F.coalesce(F.col("damnificados"), F.lit(0))
        ).alias("damnificados_reales_2026")
    )
)

In [ ]:
df_distritos_84 = (
    df_panel_base
    .select(
        "ubigeo",
        "provincia",
        "distrito"
    )
    .distinct()
)

df_meses_2026 = spark.createDataFrame(
    [(1,), (2,), (3,), (4,), (5,)],
    ["mes_num"]
)

df_real_2026_panel = (
    df_distritos_84
    .crossJoin(df_meses_2026)
    .join(
        df_real_2026_agg,
        on=["ubigeo", "mes_num"],
        how="left"
    )
    .fillna({
        "emergencias_reales_2026": 0,
        "afectados_reales_2026": 0,
        "damnificados_reales_2026": 0
    })
    .withColumn(
        "ocurre_emergencia_real",
        F.when(
            F.col("emergencias_reales_2026") > 0,
            1.0
        ).otherwise(0.0)
    )
)

In [ ]:
print(
    "Filas observadas:",
    df_real_2026_panel.count()
)

print(
    "Distritos:",
    df_real_2026_panel
    .select("ubigeo")
    .distinct()
    .count()
)

df_real_2026_panel.groupBy(
    "ocurre_emergencia_real"
).count().show()

In [ ]:
(
    df_real_2026_panel
    .groupBy("mes_num")
    .agg(
        F.sum("ocurre_emergencia_real")
            .alias("distritos_con_emergencia"),

        F.sum("emergencias_reales_2026")
            .alias("emergencias_registradas")
    )
    .orderBy("mes_num")
    .show()
)

In [ ]:
train_2026_vec = assembler_mensual.transform(
    df_train_2026
)

entrada_2026_vec = assembler_mensual.transform(
    df_entrada_2026
)

lr_2026 = LogisticRegression(
    labelCol="ocurre_emergencia",
    featuresCol="features",
    weightCol="peso_clase",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

modelo_lr_2026 = lr_2026.fit(train_2026_vec)

In [ ]:
pred_2026 = (
    modelo_lr_2026
    .transform(entrada_2026_vec)
    .withColumn(
        "prob_array",
        vector_to_array("probability")
    )
    .withColumn(
        "score_emergencia_2026",
        F.col("prob_array")[1]
    )
    .withColumn(
        "pred_emergencia_2026",
        F.when(
            F.col("score_emergencia_2026") >= 0.50,
            1.0
        ).otherwise(0.0)
    )
)

In [ ]:
pred_ene_may_2026 = (
    pred_2026
    .filter(F.col("mes_num").between(1, 5))
    .select(
        "ubigeo",
        "provincia",
        "distrito",
        "mes_num",
        "score_emergencia_2026",
        "pred_emergencia_2026"
    )
)

print(
    "Predicciones ene-may:",
    pred_ene_may_2026.count()
)

In [ ]:
validacion_real_2026 = (
    pred_ene_may_2026
    .join(
        df_real_2026_panel.select(
            "ubigeo",
            "mes_num",
            "emergencias_reales_2026",
            "ocurre_emergencia_real"
        ),
        on=["ubigeo", "mes_num"],
        how="inner"
    )
)

print(
    "Filas validación:",
    validacion_real_2026.count()
)

In [ ]:
tp_2026 = validacion_real_2026.filter(
    (F.col("ocurre_emergencia_real") == 1) &
    (F.col("pred_emergencia_2026") == 1)
).count()

fn_2026 = validacion_real_2026.filter(
    (F.col("ocurre_emergencia_real") == 1) &
    (F.col("pred_emergencia_2026") == 0)
).count()

fp_2026 = validacion_real_2026.filter(
    (F.col("ocurre_emergencia_real") == 0) &
    (F.col("pred_emergencia_2026") == 1)
).count()

tn_2026 = validacion_real_2026.filter(
    (F.col("ocurre_emergencia_real") == 0) &
    (F.col("pred_emergencia_2026") == 0)
).count()

precision_2026 = (
    tp_2026 / (tp_2026 + fp_2026)
    if tp_2026 + fp_2026 else 0
)

recall_2026 = (
    tp_2026 / (tp_2026 + fn_2026)
    if tp_2026 + fn_2026 else 0
)

f1_2026 = (
    2 * precision_2026 * recall_2026 /
    (precision_2026 + recall_2026)
    if precision_2026 + recall_2026 else 0
)

f2_2026 = (
    5 * precision_2026 * recall_2026 /
    (4 * precision_2026 + recall_2026)
    if 4 * precision_2026 + recall_2026 else 0
)

accuracy_2026 = (
    (tp_2026 + tn_2026) /
    (tp_2026 + tn_2026 + fp_2026 + fn_2026)
)

print("TP:", tp_2026)
print("FN:", fn_2026)
print("FP:", fp_2026)
print("TN:", tn_2026)

print("Accuracy:", round(accuracy_2026, 4))
print("Precision:", round(precision_2026, 4))
print("Recall:", round(recall_2026, 4))
print("F1:", round(f1_2026, 4))
print("F2:", round(f2_2026, 4))

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.linalg import Vectors

In [ ]:
from pyspark.ml.functions import array_to_vector

validacion_real_2026_eval = (
    validacion_real_2026
    .withColumn(
        "raw_eval",
        array_to_vector(
            F.array(
                1 - F.col("score_emergencia_2026"),
                F.col("score_emergencia_2026")
            )
        )
    )
)

In [ ]:
evaluador_pr_2026 = BinaryClassificationEvaluator(
    labelCol="ocurre_emergencia_real",
    rawPredictionCol="raw_eval",
    metricName="areaUnderPR"
)

pr_auc_2026 = evaluador_pr_2026.evaluate(
    validacion_real_2026_eval
)

print("PR-AUC real 2026:", round(pr_auc_2026, 4))

In [ ]:
(
    validacion_real_2026
    .groupBy("mes_num")
    .agg(
        F.sum(
            F.when(
                (F.col("ocurre_emergencia_real") == 1) &
                (F.col("pred_emergencia_2026") == 1),
                1
            ).otherwise(0)
        ).alias("TP"),

        F.sum(
            F.when(
                (F.col("ocurre_emergencia_real") == 1) &
                (F.col("pred_emergencia_2026") == 0),
                1
            ).otherwise(0)
        ).alias("FN"),

        F.sum(
            F.when(
                (F.col("ocurre_emergencia_real") == 0) &
                (F.col("pred_emergencia_2026") == 1),
                1
            ).otherwise(0)
        ).alias("FP"),

        F.sum(
            F.when(
                (F.col("ocurre_emergencia_real") == 0) &
                (F.col("pred_emergencia_2026") == 0),
                1
            ).otherwise(0)
        ).alias("TN")
    )
    .orderBy("mes_num")
    .show()
)

In [ ]:
ruta_validacion_2026 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/"
    "validacion_real_ene_may_2026"
)

(
    validacion_real_2026
    .write
    .mode("overwrite")
    .partitionBy("mes_num")
    .parquet(ruta_validacion_2026)
)

print("Validación real 2026 guardada.")

In [ ]:
ruta_modelo_2026 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/models/"
    "lr_distrito_mes_corte_2025"
)

(
    modelo_lr_2026
    .write()
    .overwrite()
    .save(ruta_modelo_2026)
)

print("Modelo guardado.")

In [ ]:
# Filas de 2025 cuyas etiquetas 2026 ahora conocemos
df_nuevas_etiquetas_2026 = (
    df_features_mensual
    .filter(
        (F.col("anio") == 2025) &
        (F.col("mes_num").between(1, 5))
    )
    .drop(
        "emergencias_siguiente_anio",
        "ocurre_emergencia"
    )
    .join(
        df_real_2026_panel.select(
            "ubigeo",
            "mes_num",
            F.col("emergencias_reales_2026")
                .alias("emergencias_siguiente_anio"),
            F.col("ocurre_emergencia_real")
                .alias("ocurre_emergencia")
        ),
        on=["ubigeo", "mes_num"],
        how="inner"
    )
)

print(
    "Nuevas filas etiquetadas:",
    df_nuevas_etiquetas_2026.count()
)

df_nuevas_etiquetas_2026.groupBy(
    "ocurre_emergencia"
).count().show()

In [ ]:
df_train_v2 = (
    df_train_2026
    .drop("peso_clase")
    .unionByName(
        df_nuevas_etiquetas_2026,
        allowMissingColumns=True
    )
)

print("TRAIN V2:", df_train_v2.count())

df_train_v2.groupBy(
    "ocurre_emergencia"
).count().show()

In [ ]:
conteos_v2 = {
    row["ocurre_emergencia"]: row["count"]
    for row in (
        df_train_v2
        .groupBy("ocurre_emergencia")
        .count()
        .collect()
    )
}

n_v2 = sum(conteos_v2.values())

peso_no_v2 = (
    n_v2 /
    (2 * conteos_v2[0.0])
)

peso_si_v2 = (
    n_v2 /
    (2 * conteos_v2[1.0])
)

print("Total V2:", n_v2)
print("Peso NO V2:", peso_no_v2)
print("Peso SI V2:", peso_si_v2)

In [ ]:
df_train_v2 = (
    df_train_v2
    .withColumn(
        "peso_clase",
        F.when(
            F.col("ocurre_emergencia") == 1.0,
            F.lit(peso_si_v2)
        ).otherwise(
            F.lit(peso_no_v2)
        )
    )
)

In [ ]:
train_v2_vec = assembler_mensual.transform(
    df_train_v2
)

lr_v2 = LogisticRegression(
    labelCol="ocurre_emergencia",
    featuresCol="features",
    weightCol="peso_clase",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

modelo_lr_v2 = lr_v2.fit(train_v2_vec)

print("Modelo V2 entrenado.")

In [ ]:
entrada_jun_dic_2026 = (
    df_features_mensual
    .filter(
        (F.col("anio") == 2025) &
        (F.col("mes_num").between(6, 12))
    )
)

entrada_jun_dic_vec = assembler_mensual.transform(
    entrada_jun_dic_2026
)

print(
    "Filas junio-diciembre:",
    entrada_jun_dic_vec.count()
)

In [ ]:
pred_v1_jun_dic = (
    modelo_lr_2026
    .transform(entrada_jun_dic_vec)
    .withColumn(
        "score_v1",
        vector_to_array("probability")[1]
    )
    .select(
        "ubigeo",
        "mes_num",
        "provincia",
        "distrito",
        "score_v1"
    )
)

In [ ]:
pred_v2_jun_dic = (
    modelo_lr_v2
    .transform(entrada_jun_dic_vec)
    .withColumn(
        "score_v2",
        vector_to_array("probability")[1]
    )
    .select(
        "ubigeo",
        "mes_num",
        "score_v2"
    )
)

In [ ]:
comparacion_v1_v2 = (
    pred_v1_jun_dic
    .join(
        pred_v2_jun_dic,
        on=["ubigeo", "mes_num"],
        how="inner"
    )
    .withColumn(
        "alerta_v1",
        F.when(F.col("score_v1") >= 0.50, 1).otherwise(0)
    )
    .withColumn(
        "alerta_v2",
        F.when(F.col("score_v2") >= 0.50, 1).otherwise(0)
    )
    .withColumn(
        "cambio_score",
        F.col("score_v2") - F.col("score_v1")
    )
)

print("Filas comparadas:", comparacion_v1_v2.count())

In [ ]:
(
    comparacion_v1_v2
    .groupBy("mes_num")
    .agg(
        F.sum("alerta_v1").alias("alertas_v1"),
        F.sum("alerta_v2").alias("alertas_v2"),

        F.round(
            F.avg("score_v1"), 4
        ).alias("score_medio_v1"),

        F.round(
            F.avg("score_v2"), 4
        ).alias("score_medio_v2"),

        F.round(
            F.avg("cambio_score"), 4
        ).alias("cambio_medio")
    )
    .orderBy("mes_num")
    .show()
)

In [ ]:
pred_actual_2026 = (
    pred_v2_jun_dic
    .filter(F.col("mes_num").between(10, 12))
    .withColumn(
        "alerta_2026",
        F.when(
            F.col("score_v2") >= 0.50,
            1
        ).otherwise(0)
    )
)

print("Filas oct-dic:", pred_actual_2026.count())

(
    pred_actual_2026
    .groupBy("mes_num")
    .agg(
        F.sum("alerta_2026").alias("distritos_con_alerta"),
        F.round(
            F.avg("score_v2"),
            4
        ).alias("score_medio")
    )
    .orderBy("mes_num")
    .show()
)

In [ ]:
priorizacion_oct_dic_2026 = (
    pred_actual_2026
    .groupBy(
        "ubigeo",
        "provincia",
        "distrito"
    )
    .agg(
        F.sum("alerta_2026")
            .alias("meses_con_alerta_oct_dic"),

        F.round(
            F.avg("score_v2"),
            4
        ).alias("score_promedio_oct_dic"),

        F.round(
            F.max("score_v2"),
            4
        ).alias("score_maximo_oct_dic")
    )
    .orderBy(
        F.desc("meses_con_alerta_oct_dic"),
        F.desc("score_promedio_oct_dic")
    )
)

priorizacion_oct_dic_2026.show(
    30,
    truncate=False
)

In [ ]:
pred_v2_jun_dic = (
    modelo_lr_v2
    .transform(entrada_jun_dic_vec)
    .withColumn(
        "score_v2",
        vector_to_array("probability")[1]
    )
    .select(
        "ubigeo",
        "provincia",
        "distrito",
        "mes_num",
        "score_v2"
    )
)

In [ ]:
pred_actual_2026 = (
    pred_v2_jun_dic
    .filter(F.col("mes_num").between(10, 12))
    .withColumn(
        "alerta_2026",
        F.when(
            F.col("score_v2") >= 0.50,
            1
        ).otherwise(0)
    )
)

print("Filas oct-dic:", pred_actual_2026.count())

In [ ]:
(
    pred_actual_2026
    .groupBy("mes_num")
    .agg(
        F.sum("alerta_2026")
            .alias("distritos_con_alerta"),
        F.round(
            F.avg("score_v2"),
            4
        ).alias("score_medio")
    )
    .orderBy("mes_num")
    .show()
)

In [ ]:
priorizacion_oct_dic_2026 = (
    pred_actual_2026
    .groupBy(
        "ubigeo",
        "provincia",
        "distrito"
    )
    .agg(
        F.sum("alerta_2026")
            .alias("meses_con_alerta_oct_dic"),

        F.round(
            F.avg("score_v2"),
            4
        ).alias("score_promedio_oct_dic"),

        F.round(
            F.max("score_v2"),
            4
        ).alias("score_maximo_oct_dic")
    )
    .orderBy(
        F.desc("meses_con_alerta_oct_dic"),
        F.desc("score_promedio_oct_dic")
    )
)

priorizacion_oct_dic_2026.show(
    30,
    truncate=False
)

In [ ]:
# CHIRPS 2026
ruta_chirps_2026 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/raw/chirps/2026/*.csv"
)

df_chirps_2026 = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_chirps_2026)
)

df_chirps_2026_panel = (
    df_chirps_2026
    .withColumn("mes_num", F.col("mes").cast("int"))
    .filter(F.col("mes_num").between(1, 5))
    .select(
        "ubigeo",
        "provincia",
        "distrito",
        "mes_num",
        "precipitacion_total_mm",
        "precipitacion_promedio_diaria_mm",
        "precipitacion_maxima_diaria_mm"
    )
    .withColumn("anio", F.lit(2026))
)

print("Filas CHIRPS ene-may 2026:", df_chirps_2026_panel.count())
print(
    "Distritos:",
    df_chirps_2026_panel.select("ubigeo").distinct().count()
)

In [ ]:
df_eventos_2026_mensual = (
    df_sinpad_2026
    .filter(F.col("mes_num").between(1, 5))
    .groupBy(
        "ubigeo",
        "mes_num"
    )
    .agg(
        F.count("*").alias("cantidad_emergencias_mes"),

        F.countDistinct("peligro")
            .alias("tipos_peligro_mes"),

        F.sum("afectados").alias("afectados_mes"),
        F.sum("damnificados").alias("damnificados_mes"),
        F.sum("fallecidos").alias("fallecidos_mes"),
        F.sum("desaparecidos").alias("desaparecidos_mes"),

        F.sum("viviendas_afectadas")
            .alias("viviendas_afectadas_mes"),

        F.sum("viviendas_destruidas")
            .alias("viviendas_destruidas_mes")
    )
)

In [ ]:
df_panel_2026_real = (
    df_chirps_2026_panel
    .join(
        df_eventos_2026_mensual,
        on=["ubigeo", "mes_num"],
        how="left"
    )
    .fillna({
        "cantidad_emergencias_mes": 0,
        "tipos_peligro_mes": 0,
        "afectados_mes": 0,
        "damnificados_mes": 0,
        "fallecidos_mes": 0,
        "desaparecidos_mes": 0,
        "viviendas_afectadas_mes": 0,
        "viviendas_destruidas_mes": 0
    })
)

print("Panel real 2026:", df_panel_2026_real.count())
print(
    "Distritos:",
    df_panel_2026_real.select("ubigeo").distinct().count()
)

df_panel_2026_real.groupBy("mes_num").count().orderBy("mes_num").show()

In [ ]:
columnas_panel_modelo = [
    "ubigeo",
    "provincia",
    "distrito",
    "anio",
    "mes_num",

    "precipitacion_total_mm",
    "precipitacion_promedio_diaria_mm",
    "precipitacion_maxima_diaria_mm",

    "cantidad_emergencias_mes",
    "tipos_peligro_mes",

    "afectados_mes",
    "damnificados_mes",
    "fallecidos_mes",
    "desaparecidos_mes",
    "viviendas_afectadas_mes",
    "viviendas_destruidas_mes"
]

df_panel_actualizado = (
    df_panel_mensual
    .select(*columnas_panel_modelo)
    .unionByName(
        df_panel_2026_real.select(*columnas_panel_modelo)
    )
)

print("Filas totales:", df_panel_actualizado.count())


In [ ]:
from pyspark.sql.window import Window
import math

df_panel_actualizado = (
    df_panel_actualizado
    .withColumn(
        "periodo_idx",
        F.col("anio") * 12 + F.col("mes_num")
    )
)

w = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
)

w3 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
    .rowsBetween(-3, -1)
)

w6 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
    .rowsBetween(-6, -1)
)

w12 = (
    Window
    .partitionBy("ubigeo")
    .orderBy("periodo_idx")
    .rowsBetween(-12, -1)
)

In [ ]:
df_features_actualizado = (
    df_panel_actualizado

    .withColumn(
        "emergencias_mes_anterior",
        F.lag("cantidad_emergencias_mes", 1).over(w)
    )
    .withColumn(
        "emergencias_3_meses",
        F.sum("cantidad_emergencias_mes").over(w3)
    )
    .withColumn(
        "emergencias_6_meses",
        F.sum("cantidad_emergencias_mes").over(w6)
    )
    .withColumn(
        "emergencias_12_meses",
        F.sum("cantidad_emergencias_mes").over(w12)
    )
    .withColumn(
        "emergencias_mismo_mes_anio_anterior",
        F.lag("cantidad_emergencias_mes", 12).over(w)
    )

    .withColumn(
        "afectados_12_meses",
        F.sum("afectados_mes").over(w12)
    )
    .withColumn(
        "damnificados_12_meses",
        F.sum("damnificados_mes").over(w12)
    )
    .withColumn(
        "fallecidos_12_meses",
        F.sum("fallecidos_mes").over(w12)
    )
    .withColumn(
        "viviendas_afectadas_12_meses",
        F.sum("viviendas_afectadas_mes").over(w12)
    )
    .withColumn(
        "viviendas_destruidas_12_meses",
        F.sum("viviendas_destruidas_mes").over(w12)
    )

    .withColumn(
        "lluvia_mes_anterior",
        F.lag("precipitacion_total_mm", 1).over(w)
    )
    .withColumn(
        "lluvia_3_meses",
        F.sum("precipitacion_total_mm").over(w3)
    )

    .withColumn(
        "mes_sin",
        F.sin(
            2 * F.lit(math.pi) * F.col("mes_num") / 12
        )
    )
    .withColumn(
        "mes_cos",
        F.cos(
            2 * F.lit(math.pi) * F.col("mes_num") / 12
        )
    )
)

In [ ]:
df_entrada_2027 = (
    df_features_actualizado
    .filter(
        (F.col("anio") == 2026) &
        (F.col("mes_num").between(1, 5))
    )
)

print("Filas para 2027:", df_entrada_2027.count())

df_entrada_2027.groupBy(
    "mes_num"
).count().orderBy("mes_num").show()

In [ ]:
entrada_2027_vec = assembler_mensual.transform(
    df_entrada_2027
)

pred_2027 = (
    modelo_lr_v2
    .transform(entrada_2027_vec)
    .withColumn(
        "score_2027",
        vector_to_array("probability")[1]
    )
    .withColumn(
        "alerta_2027",
        F.when(
            F.col("score_2027") >= 0.50,
            1
        ).otherwise(0)
    )
    .select(
        "ubigeo",
        "provincia",
        "distrito",
        "mes_num",
        "score_2027",
        "alerta_2027"
    )
)

print("Predicciones 2027:", pred_2027.count())

In [ ]:
(
    pred_2027
    .groupBy("mes_num")
    .agg(
        F.sum("alerta_2027")
            .alias("distritos_con_alerta"),

        F.round(
            F.avg("score_2027"),
            4
        ).alias("score_medio")
    )
    .orderBy("mes_num")
    .show()
)

In [ ]:
priorizacion_2027 = (
    pred_2027
    .groupBy(
        "ubigeo",
        "provincia",
        "distrito"
    )
    .agg(
        F.sum("alerta_2027")
            .alias("meses_con_alerta_2027"),

        F.round(
            F.avg("score_2027"),
            4
        ).alias("score_promedio_2027"),

        F.round(
            F.max("score_2027"),
            4
        ).alias("score_maximo_2027")
    )
    .orderBy(
        F.desc("meses_con_alerta_2027"),
        F.desc("score_promedio_2027")
    )
)

priorizacion_2027.show(
    30,
    truncate=False
)

In [ ]:
(
    priorizacion_2027
    .orderBy(
        F.asc("score_promedio_2027")
    )
    .show(
        10,
        truncate=False
    )
)

In [ ]:
ruta_modelo_v2 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/models/"
    "lr_distrito_mes_v2_actualizado_2026"
)

(
    modelo_lr_v2
    .write()
    .overwrite()
    .save(ruta_modelo_v2)
)

print("Modelo V2 guardado.")

In [ ]:
ruta_pred_2027 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/"
    "predicciones_ene_may_2027"
)

(
    pred_2027
    .write
    .mode("overwrite")
    .partitionBy("mes_num")
    .parquet(ruta_pred_2027)
)

print("Predicciones 2027 guardadas.")

In [ ]:
ruta_priorizacion_2027 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/"
    "priorizacion_ene_may_2027"
)

(
    priorizacion_2027
    .write
    .mode("overwrite")
    .parquet(ruta_priorizacion_2027)
)

print("Priorización 2027 guardada.")

In [ ]:
ultimo_mes_disponible = (
    df_panel_2026_real
    .agg(F.max("mes_num").alias("ultimo_mes"))
    .first()["ultimo_mes"]
)

print(
    "Último mes real disponible de 2026:",
    ultimo_mes_disponible
)

print(
    "Por tanto pueden generarse predicciones 2027 "
    f"desde enero hasta el mes {ultimo_mes_disponible}."
)

In [ ]:
tabla_consumo_2027 = (
    priorizacion_2027
    .withColumn(
        "prioridad_2027",
        F.when(
            F.col("meses_con_alerta_2027") >= 4,
            "ALTA"
        )
        .when(
            F.col("meses_con_alerta_2027") >= 2,
            "MEDIA"
        )
        .otherwise("BAJA")
    )
)

tabla_consumo_2027.groupBy(
    "prioridad_2027"
).count().show()

In [ ]:
from pyspark.sql.window import Window

w_ranking = Window.orderBy(
    F.desc("meses_con_alerta_2027"),
    F.desc("score_promedio_2027")
)

tabla_consumo_2027 = (
    tabla_consumo_2027
    .withColumn(
        "ranking_prioridad_2027",
        F.row_number().over(w_ranking)
    )
)

tabla_consumo_2027.select(
    "ranking_prioridad_2027",
    "provincia",
    "distrito",
    "prioridad_2027",
    "meses_con_alerta_2027",
    "score_promedio_2027",
    "score_maximo_2027"
).orderBy(
    "ranking_prioridad_2027"
).show(30, truncate=False)

In [ ]:
ruta_consumo_2027 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/"
    "tabla_consumo_priorizacion_2027"
)

(
    tabla_consumo_2027
    .write
    .mode("overwrite")
    .parquet(ruta_consumo_2027)
)

print("Tabla de consumo 2027 guardada.")

In [ ]:
df_impacto_distrital = (
    df_panel_mensual
    .groupBy(
        "ubigeo",
        "provincia",
        "distrito"
    )
    .agg(
        F.sum("cantidad_emergencias_mes")
            .alias("emergencias_historicas"),

        F.sum("afectados_mes")
            .alias("afectados_historicos"),

        F.sum("damnificados_mes")
            .alias("damnificados_historicos"),

        F.sum("fallecidos_mes")
            .alias("fallecidos_historicos"),

        F.sum("viviendas_destruidas_mes")
            .alias("viviendas_destruidas_historicas")
    )
)

print("Distritos impacto:", df_impacto_distrital.count())

In [ ]:
from pyspark.sql.window import Window

w_afectados = Window.orderBy("afectados_historicos")
w_damnificados = Window.orderBy("damnificados_historicos")
w_fallecidos = Window.orderBy("fallecidos_historicos")
w_viviendas = Window.orderBy("viviendas_destruidas_historicas")

df_impacto_distrital = (
    df_impacto_distrital

    .withColumn(
        "p_afectados",
        F.percent_rank().over(w_afectados)
    )

    .withColumn(
        "p_damnificados",
        F.percent_rank().over(w_damnificados)
    )

    .withColumn(
        "p_fallecidos",
        F.percent_rank().over(w_fallecidos)
    )

    .withColumn(
        "p_viviendas",
        F.percent_rank().over(w_viviendas)
    )

    .withColumn(
        "score_impacto_historico",
        (
            F.col("p_afectados")
            + F.col("p_damnificados")
            + F.col("p_fallecidos")
            + F.col("p_viviendas")
        ) / 4
    )
)

In [ ]:
(
    df_impacto_distrital
    .select(
        "provincia",
        "distrito",
        "emergencias_historicas",
        "afectados_historicos",
        "damnificados_historicos",
        "fallecidos_historicos",
        "viviendas_destruidas_historicas",
        F.round(
            "score_impacto_historico", 4
        ).alias("score_impacto")
    )
    .orderBy(
        F.desc("score_impacto_historico")
    )
    .show(20, truncate=False)
)

In [ ]:
df_poblacion_2025 = (
    df_inei
    .filter(F.col("anio") == 2025)
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .select(
        "provincia_key",
        "distrito_key",
        F.col("poblacion").alias("poblacion_2025")
    )
)

print("Distritos INEI 2025:", df_poblacion_2025.count())

In [ ]:
ruta_inei = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/raw/inei/"
    "inei_la_libertad_poblacion_distrital_2021_2025.csv"
)

df_inei = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_inei)
)

print("Filas INEI:", df_inei.count())
df_inei.show(5, truncate=False)

In [ ]:
df_poblacion_2025 = (
    df_inei
    .filter(F.col("anio") == 2025)

    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )

    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )

    .select(
        "provincia_key",
        "distrito_key",
        F.col("poblacion").alias("poblacion_2025")
    )
)

print(
    "Distritos INEI 2025:",
    df_poblacion_2025.count()
)

In [ ]:
df_prioridad_base = (
    priorizacion_2027

    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )

    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
)

df_prioridad_integral = (
    df_prioridad_base

    .join(
        df_impacto_distrital.select(
            "ubigeo",
            "score_impacto_historico"
        ),
        on="ubigeo",
        how="left"
    )

    .join(
        df_poblacion_2025,
        on=["provincia_key", "distrito_key"],
        how="left"
    )
)

print("Filas:", df_prioridad_integral.count())

print(
    "Sin población:",
    df_prioridad_integral
    .filter(F.col("poblacion_2025").isNull())
    .count()
)

print(
    "Sin impacto:",
    df_prioridad_integral
    .filter(F.col("score_impacto_historico").isNull())
    .count()
)

In [ ]:
from pyspark.sql.window import Window

w_ocurrencia = Window.orderBy("score_promedio_2027")
w_poblacion = Window.orderBy("poblacion_2025")

df_prioridad_integral = (
    df_prioridad_integral

    .withColumn(
        "p_ocurrencia",
        F.percent_rank().over(w_ocurrencia)
    )

    .withColumn(
        "p_poblacion",
        F.percent_rank().over(w_poblacion)
    )
)

In [ ]:
df_prioridad_integral = (
    df_prioridad_integral

    .withColumn(
        "score_prioridad_integral",
        (
            F.col("p_ocurrencia") * 0.50
            + F.col("score_impacto_historico") * 0.30
            + F.col("p_poblacion") * 0.20
        )
    )

    .withColumn(
        "score_prioridad_integral",
        F.round("score_prioridad_integral", 4)
    )
)

In [ ]:
w_final = Window.orderBy(
    F.desc("score_prioridad_integral")
)

df_prioridad_integral = (
    df_prioridad_integral
    .withColumn(
        "ranking_integral",
        F.row_number().over(w_final)
    )
)

In [ ]:
(
    df_prioridad_integral
    .select(
        "ranking_integral",
        "provincia",
        "distrito",
        "score_promedio_2027",
        F.round("p_ocurrencia", 4).alias("score_ocurrencia_relativo"),
        F.round("score_impacto_historico", 4).alias("score_impacto"),
        "poblacion_2025",
        F.round("p_poblacion", 4).alias("score_exposicion"),
        "score_prioridad_integral"
    )
    .orderBy("ranking_integral")
    .show(25, truncate=False)
)

In [ ]:
print("Master:", spark.sparkContext.master)
print("App ID:", spark.sparkContext.applicationId)

print(
    "Prueba Spark:",
    spark.range(10).count()
)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def normalizar_clave(columna):
    return F.translate(
        F.upper(F.trim(columna)),
        "ÁÉÍÓÚÜÑ",
        "AEIOUUN"
    )

# Ranking 2027 ya materializado
priorizacion_2027 = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/"
    "priorizacion_ene_may_2027"
)

# Panel histórico ya materializado
df_panel_mensual = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/"
    "panel_distrito_mes_2003_2025"
)

# INEI
df_inei = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(
        "hdfs://localhost:9000/"
        "proyecto_emergencias/raw/inei/"
        "inei_la_libertad_poblacion_distrital_2021_2025.csv"
    )
)

print("Ranking 2027:", priorizacion_2027.count())
print("Panel histórico:", df_panel_mensual.count())
print("INEI:", df_inei.count())

In [ ]:
# 1. Impacto histórico por distrito
df_impacto_base = (
    df_panel_mensual
    .groupBy("ubigeo")
    .agg(
        F.sum("afectados_mes").alias("afectados_historicos"),
        F.sum("damnificados_mes").alias("damnificados_historicos"),
        F.sum("fallecidos_mes").alias("fallecidos_historicos"),
        F.sum("viviendas_destruidas_mes")
            .alias("viviendas_destruidas_historicas")
    )
)

# 2. Población INEI 2025
df_poblacion_2025 = (
    df_inei
    .filter(F.col("anio") == 2025)
    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )
    .select(
        "provincia_key",
        "distrito_key",
        F.col("poblacion").alias("poblacion_2025")
    )
)

# 3. Ranking predictivo + impacto + población
df_base_84 = (
    priorizacion_2027

    .withColumn(
        "provincia_key",
        normalizar_clave(F.col("provincia"))
    )
    .withColumn(
        "distrito_key",
        normalizar_clave(F.col("distrito"))
    )

    .join(
        df_impacto_base,
        on="ubigeo",
        how="left"
    )

    .join(
        df_poblacion_2025,
        on=["provincia_key", "distrito_key"],
        how="left"
    )

    .select(
        "ubigeo",
        "provincia",
        "distrito",
        "meses_con_alerta_2027",
        "score_promedio_2027",
        "score_maximo_2027",
        "afectados_historicos",
        "damnificados_historicos",
        "fallecidos_historicos",
        "viviendas_destruidas_historicas",
        "poblacion_2025"
    )
)

print("Filas base:", df_base_84.count())

print(
    "Sin población:",
    df_base_84.filter(
        F.col("poblacion_2025").isNull()
    ).count()
)

print(
    "Sin impacto:",
    df_base_84.filter(
        F.col("afectados_historicos").isNull()
    ).count()
)

In [ ]:
ruta_base_84 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/"
    "prioridad_base_84"
)

(
    df_base_84
    .write
    .mode("overwrite")
    .parquet(ruta_base_84)
)

print("Base de 84 distritos materializada.")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ruta_base_84 = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/"
    "prioridad_base_84"
)

df84 = (
    spark.read.parquet(ruta_base_84)
    .coalesce(1)
    .cache()
)

print("Distritos:", df84.count())

In [ ]:
w_occ = Window.orderBy("score_promedio_2027")
w_afec = Window.orderBy("afectados_historicos")
w_damn = Window.orderBy("damnificados_historicos")
w_fall = Window.orderBy("fallecidos_historicos")
w_viv = Window.orderBy("viviendas_destruidas_historicas")
w_pob = Window.orderBy("poblacion_2025")

df_prioridad_final = (
    df84

    .withColumn(
        "p_ocurrencia",
        F.percent_rank().over(w_occ)
    )

    .withColumn(
        "p_afectados",
        F.percent_rank().over(w_afec)
    )

    .withColumn(
        "p_damnificados",
        F.percent_rank().over(w_damn)
    )

    .withColumn(
        "p_fallecidos",
        F.percent_rank().over(w_fall)
    )

    .withColumn(
        "p_viviendas",
        F.percent_rank().over(w_viv)
    )

    .withColumn(
        "p_poblacion",
        F.percent_rank().over(w_pob)
    )

    .withColumn(
        "score_impacto",
        (
            F.col("p_afectados") +
            F.col("p_damnificados") +
            F.col("p_fallecidos") +
            F.col("p_viviendas")
        ) / 4
    )

    .withColumn(
        "score_prioridad_integral",
        (
            F.col("p_ocurrencia") * 0.50 +
            F.col("score_impacto") * 0.30 +
            F.col("p_poblacion") * 0.20
        )
    )
)

In [ ]:
(
    df_prioridad_final
    .select(
        "provincia",
        "distrito",
        "score_promedio_2027",
        F.round("p_ocurrencia", 4).alias("ocurrencia"),
        F.round("score_impacto", 4).alias("impacto"),
        "poblacion_2025",
        F.round("p_poblacion", 4).alias("exposicion"),
        F.round(
            "score_prioridad_integral", 4
        ).alias("prioridad_integral")
    )
    .orderBy(
        F.desc("score_prioridad_integral")
    )
    .show(20, truncate=False)
)

In [ ]:
w_clasificacion = Window.orderBy(
    F.desc("score_prioridad_integral")
)

df_prioridad_final = (
    df_prioridad_final
    .withColumn(
        "cuartil_prioridad",
        F.ntile(4).over(w_clasificacion)
    )
    .withColumn(
        "prioridad",
        F.when(
            F.col("cuartil_prioridad") == 1,
            "ALTA"
        )
        .when(
            F.col("cuartil_prioridad").isin(2, 3),
            "MEDIA"
        )
        .otherwise("BAJA")
    )
)

In [ ]:
df_prioridad_final.groupBy(
    "prioridad"
).count().show()

In [ ]:
tabla_consumo_final = (
    df_prioridad_final
    .select(
        "ubigeo",
        "provincia",
        "distrito",
        "prioridad",
        "meses_con_alerta_2027",

        F.round(
            "score_promedio_2027", 4
        ).alias("score_ocurrencia_modelo"),

        F.round(
            "score_impacto", 4
        ).alias("score_impacto_historico"),

        "poblacion_2025",

        F.round(
            "score_prioridad_integral", 4
        ).alias("score_prioridad_integral")
    )
    .orderBy(
        F.desc("score_prioridad_integral")
    )
)

tabla_consumo_final.show(
    30,
    truncate=False
)

In [ ]:
print("Distribución final:")
tabla_consumo_final.groupBy("prioridad").count().show()

In [ ]:
ruta_tabla_final = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/"
    "priorizacion_integral_2027"
)

(
    tabla_consumo_final
    .write
    .mode("overwrite")
    .parquet(ruta_tabla_final)
)

print("Priorización integral 2027 guardada.")

In [ ]:
df_verificacion_final = spark.read.parquet(ruta_tabla_final)

print("Filas:", df_verificacion_final.count())
df_verificacion_final.groupBy("prioridad").count().show()

In [ ]:
pred_2027_consumo = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/"
    "predicciones_ene_may_2027"
)

prioridad_integral = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/"
    "priorizacion_integral_2027"
)

print("Predicciones mensuales:", pred_2027_consumo.count())
print("Distritos priorizados:", prioridad_integral.count())

In [ ]:
dim_distrito = (
    prioridad_integral
    .select(
        "ubigeo",
        "provincia",
        "distrito",
        "poblacion_2025"
    )
    .distinct()
)

dim_tiempo = (
    pred_2027_consumo
    .select("mes_num")
    .distinct()
    .withColumn("anio", F.lit(2027))
    .withColumn(
        "nombre_mes",
        F.when(F.col("mes_num") == 1, "ENERO")
         .when(F.col("mes_num") == 2, "FEBRERO")
         .when(F.col("mes_num") == 3, "MARZO")
         .when(F.col("mes_num") == 4, "ABRIL")
         .when(F.col("mes_num") == 5, "MAYO")
    )
)

dim_prioridad = spark.createDataFrame(
    [
        ("ALTA", 1),
        ("MEDIA", 2),
        ("BAJA", 3)
    ],
    ["prioridad", "orden_prioridad"]
)

In [ ]:
hecho_priorizacion = (
    pred_2027_consumo
    .join(
        prioridad_integral.select(
            "ubigeo",
            "prioridad",
            "score_impacto_historico",
            "score_prioridad_integral"
        ),
        on="ubigeo",
        how="left"
    )
    .select(
        "ubigeo",
        F.lit(2027).alias("anio"),
        "mes_num",

        F.col("score_2027")
            .alias("score_ocurrencia"),

        F.col("alerta_2027")
            .alias("alerta"),

        "prioridad",
        "score_impacto_historico",
        "score_prioridad_integral"
    )
)

print("Hechos:", hecho_priorizacion.count())

In [ ]:
print("DIM DISTRITO:", dim_distrito.count())
print("DIM TIEMPO:", dim_tiempo.count())
print("DIM PRIORIDAD:", dim_prioridad.count())
print("HECHO:", hecho_priorizacion.count())

In [ ]:
base_dimensional = (
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/modelo_dimensional"
)

(
    dim_distrito
    .write
    .mode("overwrite")
    .parquet(f"{base_dimensional}/dim_distrito")
)

(
    dim_tiempo
    .write
    .mode("overwrite")
    .parquet(f"{base_dimensional}/dim_tiempo")
)

(
    dim_prioridad
    .write
    .mode("overwrite")
    .parquet(f"{base_dimensional}/dim_prioridad")
)

(
    hecho_priorizacion
    .write
    .mode("overwrite")
    .partitionBy("anio", "mes_num")
    .parquet(f"{base_dimensional}/hecho_priorizacion")
)

print("Modelo dimensional guardado.")

In [ ]:
print(
    "DIM DISTRITO:",
    spark.read.parquet(
        f"{base_dimensional}/dim_distrito"
    ).count()
)

print(
    "DIM TIEMPO:",
    spark.read.parquet(
        f"{base_dimensional}/dim_tiempo"
    ).count()
)

print(
    "DIM PRIORIDAD:",
    spark.read.parquet(
        f"{base_dimensional}/dim_prioridad"
    ).count()
)

print(
    "HECHO:",
    spark.read.parquet(
        f"{base_dimensional}/hecho_priorizacion"
    ).count()
)

In [ ]:
dim_distrito_sql = spark.read.parquet(
    f"{base_dimensional}/dim_distrito"
)

dim_tiempo_sql = spark.read.parquet(
    f"{base_dimensional}/dim_tiempo"
)

dim_prioridad_sql = spark.read.parquet(
    f"{base_dimensional}/dim_prioridad"
)

hecho_sql = spark.read.parquet(
    f"{base_dimensional}/hecho_priorizacion"
)

dim_distrito_sql.createOrReplaceTempView("dim_distrito")
dim_tiempo_sql.createOrReplaceTempView("dim_tiempo")
dim_prioridad_sql.createOrReplaceTempView("dim_prioridad")
hecho_sql.createOrReplaceTempView("hecho_priorizacion")

print("Vistas Spark SQL creadas.")

In [ ]:
spark.sql("""
SELECT
    prioridad,
    COUNT(DISTINCT ubigeo) AS distritos
FROM hecho_priorizacion
GROUP BY prioridad
ORDER BY
    CASE prioridad
        WHEN 'ALTA' THEN 1
        WHEN 'MEDIA' THEN 2
        WHEN 'BAJA' THEN 3
    END
""").show()

In [ ]:
spark.sql("""
SELECT
    t.mes_num,
    t.nombre_mes,
    SUM(h.alerta) AS distritos_con_alerta,
    ROUND(AVG(h.score_ocurrencia), 4) AS score_medio
FROM hecho_priorizacion h
JOIN dim_tiempo t
    ON h.anio = t.anio
    AND h.mes_num = t.mes_num
GROUP BY
    t.mes_num,
    t.nombre_mes
ORDER BY t.mes_num
""").show()

In [ ]:
spark.sql("""
SELECT DISTINCT
    d.provincia,
    d.distrito,
    h.prioridad,
    ROUND(h.score_prioridad_integral, 4)
        AS score_prioridad_integral,
    d.poblacion_2025
FROM hecho_priorizacion h
JOIN dim_distrito d
    ON h.ubigeo = d.ubigeo
ORDER BY score_prioridad_integral DESC
LIMIT 10
""").show(truncate=False)

In [ ]:
spark.sql("""
SELECT
    d.provincia,
    d.distrito,
    SUM(h.alerta) AS meses_con_alerta,
    ROUND(AVG(h.score_ocurrencia), 4)
        AS score_ocurrencia_promedio
FROM hecho_priorizacion h
JOIN dim_distrito d
    ON h.ubigeo = d.ubigeo
GROUP BY
    d.provincia,
    d.distrito
ORDER BY
    meses_con_alerta DESC,
    score_ocurrencia_promedio DESC
LIMIT 20
""").show(truncate=False)

In [ ]:
# SINPAD histórico procesado 2003-2025
df_sinpad_historico = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/"
    "dataset_maestro_2003_2025"
)

# SINPAD 2026 procesado
df_sinpad_2026_audit = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/"
    "sinpad_la_libertad_2026"
)

print("Histórico:", len(df_sinpad_historico.columns), "columnas")
print("2026:", len(df_sinpad_2026_audit.columns), "columnas")

In [ ]:
columnas_auditar = (
    df_sinpad_historico.columns
    + df_sinpad_2026_audit.columns
)

terminos_sensibles = [
    "dni",
    "nombre",
    "apellido",
    "telefono",
    "celular",
    "correo",
    "email",
    "direccion",
    "documento",
    "persona"
]

columnas_detectadas = sorted({
    c for c in columnas_auditar
    if any(
        termino in c.lower()
        for termino in terminos_sensibles
    )
})

print("Columnas potencialmente sensibles:")
print(columnas_detectadas)

In [ ]:
datos_linaje = [
    (
        "SINPAD 2003-2025",
        "INDECI",
        "RAW",
        "CSV",
        "Limpieza de nombres, fechas, tipos y filtrado de La Libertad",
        "dataset_maestro_2003_2025"
    ),
    (
        "GeoSINPAD 2026",
        "INDECI",
        "RAW",
        "GeoJSON",
        "Aplanado, conversión de fecha, normalización geográfica",
        "sinpad_la_libertad_2026"
    ),
    (
        "COEN",
        "INDECI",
        "RAW",
        "PDF / CSV metadatos",
        "Vinculación mediante código SINPAD y agregación de reportes",
        "emergencias_integradas"
    ),
    (
        "INEI",
        "INEI",
        "RAW",
        "CSV",
        "Normalización de provincia/distrito y selección población 2025",
        "dim_distrito"
    ),
    (
        "CHIRPS",
        "UCSB-CHC",
        "RAW",
        "CSV mensual",
        "Integración por distrito, año y mes",
        "panel_distrito_mes"
    ),
    (
        "Panel distrito-mes",
        "Pipeline Spark",
        "PROCESSED",
        "Parquet",
        "Lags, ventanas 3/6/12 meses, estacionalidad e impacto histórico",
        "features_modelo"
    ),
    (
        "Modelo LR V2",
        "Spark ML",
        "ANALYTICS",
        "Modelo",
        "Scoring, ranking e integración con impacto y población",
        "priorizacion_integral_2027"
    ),
    (
        "Modelo dimensional",
        "Spark SQL",
        "ANALYTICS",
        "Parquet",
        "Hecho de priorización + dimensiones distrito, tiempo y prioridad",
        "capa_consumo"
    )
]

df_linaje = spark.createDataFrame(
    datos_linaje,
    [
        "fuente",
        "responsable_fuente",
        "zona",
        "formato",
        "transformacion_principal",
        "salida"
    ]
)

df_linaje.show(truncate=False)

In [ ]:
(
    df_linaje
    .write
    .mode("overwrite")
    .parquet(
        "hdfs://localhost:9000/"
        "proyecto_emergencias/analytics/linaje_datos"
    )
)

print("Linaje guardado.")

In [ ]:
sesgos = [
    (
        "Cambio temporal en el registro",
        "El número de registros aumenta fuertemente desde 2023.",
        "El modelo puede interpretar mayor capacidad de registro como mayor riesgo real.",
        "Comparar periodos, monitorear drift y reentrenar con datos recientes."
    ),
    (
        "Desbalance de clases",
        "Históricamente los distrito-mes con emergencia son minoritarios.",
        "El modelo podría favorecer la clase sin emergencia.",
        "Pesos por clase y evaluación con Recall, F1, F2 y PR-AUC."
    ),
    (
        "Drift temporal",
        "La distribución observada en 2026 difiere del histórico.",
        "Puede degradarse el desempeño conforme cambian los patrones.",
        "Actualización incremental y validación con datos posteriores."
    ),
    (
        "Subregistro geográfico",
        "La capacidad de reporte puede variar entre distritos.",
        "Un distrito con pocos registros podría parecer artificialmente seguro.",
        "No interpretar ausencia de registro como ausencia absoluta de riesgo."
    ),
    (
        "Dominio de eventos hidrometeorológicos",
        "Lluvia intensa es el peligro más frecuente del histórico.",
        "Las variables climáticas pueden ser menos útiles para otros peligros.",
        "Mantener CHIRPS como variable contextual y no como causa universal."
    ),
    (
        "Corte parcial 2026",
        "Los registros disponibles de GeoSINPAD cubren enero-mayo.",
        "No representa el comportamiento completo del año 2026.",
        "Actualizar el pipeline conforme INDECI publique nuevos meses."
    )
]

df_sesgos = spark.createDataFrame(
    sesgos,
    [
        "sesgo",
        "evidencia",
        "consecuencia",
        "salvaguarda"
    ]
)

df_sesgos.show(truncate=False)

In [ ]:
(
    df_sesgos
    .write
    .mode("overwrite")
    .parquet(
        "hdfs://localhost:9000/"
        "proyecto_emergencias/analytics/"
        "sesgos_y_salvaguardas"
    )
)

print("Sesgos y salvaguardas guardados.")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ProyectoEmergenciasLaLibertad")
    .master("yarn")
    .getOrCreate()
)

print("Spark master:", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)

In [ ]:
import matplotlib.pyplot as plt

anios = list(range(2003, 2026))

emergencias_anuales = [
    32, 43, 71, 74, 70, 18, 40, 40, 68, 46, 46,
    31, 26, 40, 357, 64, 48, 39, 94, 93, 360, 313, 840
]

plt.figure(figsize=(14, 6))

plt.plot(
    anios,
    emergencias_anuales,
    marker="o",
    linewidth=2
)

plt.title(
    "Evolución anual de emergencias registradas en La Libertad (2003–2025)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Año")
plt.ylabel("Número de emergencias")

plt.xticks(anios, rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.4)

for x, y in zip(anios, emergencias_anuales):
    plt.text(
        x,
        y + 10,
        str(y),
        ha="center",
        fontsize=8
    )

plt.tight_layout()

plt.savefig(
    "grafico_01_emergencias_por_anio.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

peligros = [
    "LLUVIA INTENSA",
    "VIENTOS FUERTES",
    "INCENDIO URBANO E INDUSTRIAL",
    "DESLIZAMIENTO",
    "BAJAS TEMPERATURAS",
    "INUNDACION",
    "HUAYCO",
    "EROSION",
    "DERRUMBE",
    "INCENDIO FORESTAL"
]

cantidades = [
    1373,
    304,
    290,
    252,
    125,
    102,
    89,
    63,
    57,
    51
]

plt.figure(figsize=(11, 6))

plt.barh(
    peligros[::-1],
    cantidades[::-1]
)

plt.title(
    "Top 10 tipos de peligro registrados en La Libertad (2003–2025)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Número de emergencias")
plt.ylabel("Tipo de peligro")

plt.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

for i, valor in enumerate(cantidades[::-1]):
    plt.text(
        valor + 10,
        i,
        str(valor),
        va="center",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    "grafico_02_top_peligros.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

meses = [
    "Ene", "Feb", "Mar", "Abr", "May", "Jun",
    "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"
]

emergencias_mensuales = [
    189, 423, 810, 374, 159, 95,
    119, 152, 171, 134, 127, 100
]

plt.figure(figsize=(11, 6))

plt.bar(
    meses,
    emergencias_mensuales
)

plt.title(
    "Distribución mensual de emergencias registradas en La Libertad (2003–2025)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Mes")
plt.ylabel("Número de emergencias")

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

for i, valor in enumerate(emergencias_mensuales):
    plt.text(
        i,
        valor + 10,
        str(valor),
        ha="center",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    "grafico_03_emergencias_por_mes.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

meses = ["Ene", "Feb", "Mar", "Abr", "May"]

# Totales históricos 2003-2025 / 23 años
promedio_historico = [
    189 / 23,
    423 / 23,
    810 / 23,
    374 / 23,
    159 / 23
]

# GeoSINPAD real 2026
emergencias_2026 = [
    98,
    185,
    258,
    57,
    26
]

x = np.arange(len(meses))
ancho = 0.36

plt.figure(figsize=(10, 6))

plt.bar(
    x - ancho/2,
    promedio_historico,
    ancho,
    label="Promedio histórico 2003–2025"
)

plt.bar(
    x + ancho/2,
    emergencias_2026,
    ancho,
    label="2026 observado"
)

plt.title(
    "Emergencias observadas en 2026 frente al promedio histórico",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Mes")
plt.ylabel("Número de emergencias")

plt.xticks(x, meses)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

plt.legend()

for i, valor in enumerate(promedio_historico):
    plt.text(
        i - ancho/2,
        valor + 4,
        f"{valor:.1f}",
        ha="center",
        fontsize=9
    )

for i, valor in enumerate(emergencias_2026):
    plt.text(
        i + ancho/2,
        valor + 4,
        str(valor),
        ha="center",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    "grafico_04_2026_vs_historico.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

pred_2027 = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/"
    "predicciones_ene_may_2027"
)

resumen_2027 = (
    pred_2027
    .groupBy("mes_num")
    .agg(
        F.sum("alerta_2027").alias("distritos_alerta")
    )
    .orderBy("mes_num")
    .collect()
)

nombres_meses = {
    1: "Ene",
    2: "Feb",
    3: "Mar",
    4: "Abr",
    5: "May"
}

meses = [
    nombres_meses[r["mes_num"]]
    for r in resumen_2027
]

alertas = [
    int(r["distritos_alerta"])
    for r in resumen_2027
]

plt.figure(figsize=(10, 6))

plt.bar(
    meses,
    alertas
)

plt.title(
    "Distritos con alerta prevista por mes para 2027",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Mes")
plt.ylabel("Distritos con alerta")

plt.ylim(0, 84)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

for i, valor in enumerate(alertas):
    plt.text(
        i,
        valor + 1,
        str(valor),
        ha="center",
        fontsize=10
    )

plt.tight_layout()

plt.savefig(
    "grafico_05_alertas_2027.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

prioridad_2027 = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/analytics/"
    "priorizacion_integral_2027"
)

top10_2027 = (
    prioridad_2027
    .orderBy(F.desc("score_prioridad_integral"))
    .limit(10)
    .select(
        "distrito",
        "provincia",
        "score_prioridad_integral"
    )
    .collect()
)

# Se invierte para que el distrito de mayor prioridad
# aparezca arriba en el gráfico horizontal
top10_2027 = top10_2027[::-1]

distritos = [
    f'{r["distrito"]} ({r["provincia"]})'
    for r in top10_2027
]

scores = [
    float(r["score_prioridad_integral"])
    for r in top10_2027
]

plt.figure(figsize=(11, 7))

plt.barh(
    distritos,
    scores
)

plt.title(
    "Top 10 distritos con mayor prioridad proyectada para 2027",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Score de prioridad integral")
plt.ylabel("Distrito")

plt.xlim(0, 1)

plt.grid(
    axis="x",
    linestyle="--",
    alpha=0.4
)

for i, valor in enumerate(scores):
    plt.text(
        valor + 0.01,
        i,
        f"{valor:.3f}",
        va="center",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    "grafico_05_top_distritos_2027.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from pyspark.sql import functions as F

df_panel = spark.read.parquet(
    "hdfs://localhost:9000/"
    "proyecto_emergencias/processed/"
    "panel_distrito_mes_2003_2025"
)

print("Filas:", df_panel.count())
print("\nColumnas:")
print(df_panel.columns)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = (
    Window
    .partitionBy("ubigeo")
    .orderBy("anio", "mes_num")
)

df_base_modelo = (
    df_panel

    # Emergencias del mes anterior
    .withColumn(
        "emergencias_mes_anterior",
        F.lag("cantidad_emergencias_mes", 1).over(w)
    )

    # Mismo mes del año anterior
    .withColumn(
        "emergencias_mismo_mes_anio_anterior",
        F.lag("cantidad_emergencias_mes", 12).over(w)
    )

    # Lo que ocurre 12 meses después
    .withColumn(
        "emergencias_12m_despues",
        F.lead("cantidad_emergencias_mes", 12).over(w)
    )

    # Estacionalidad
    .withColumn(
        "mes_sin",
        F.sin(
            2 * F.lit(3.141592653589793)
            * F.col("mes_num") / 12
        )
    )

    .withColumn(
        "mes_cos",
        F.cos(
            2 * F.lit(3.141592653589793)
            * F.col("mes_num") / 12
        )
    )

    # Variable objetivo binaria
    .withColumn(
        "target_emergencia_12m",
        F.when(
            F.col("emergencias_12m_despues").isNull(),
            F.lit(None)
        )
        .when(
            F.col("emergencias_12m_despues") > 0,
            F.lit(1.0)
        )
        .otherwise(F.lit(0.0))
    )

    # Necesitamos al menos un año anterior
    .filter(
        (F.col("anio") >= 2004) &
        F.col("target_emergencia_12m").isNotNull()
    )
)

print("Filas modelables:", df_base_modelo.count())

df_base_modelo.groupBy(
    "target_emergencia_12m"
).count().orderBy(
    "target_emergencia_12m"
).show()

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

features_base = [
    "cantidad_emergencias_mes",
    "emergencias_mes_anterior",
    "emergencias_mismo_mes_anio_anterior",
    "precipitacion_total_mm",
    "mes_sin",
    "mes_cos"
]

assembler_base = VectorAssembler(
    inputCols=features_base,
    outputCol="features_base",
    handleInvalid="skip"
)

df_ml_base = assembler_base.transform(df_base_modelo)

# Entrenamiento: 2004–2023
train_base = df_ml_base.filter(
    (F.col("anio") >= 2004) &
    (F.col("anio") <= 2023)
)

# Prueba: variables de 2024 -> objetivo en 2025
test_base = df_ml_base.filter(
    F.col("anio") == 2024
)

print("Train:", train_base.count())
print("Test:", test_base.count())

print("\nClases train:")
train_base.groupBy(
    "target_emergencia_12m"
).count().orderBy(
    "target_emergencia_12m"
).show()

print("Clases test:")
test_base.groupBy(
    "target_emergencia_12m"
).count().orderBy(
    "target_emergencia_12m"
).show()

In [ ]:
lr_base = LogisticRegression(
    featuresCol="features_base",
    labelCol="target_emergencia_12m",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

modelo_base = lr_base.fit(train_base)

pred_base = modelo_base.transform(test_base)

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F

confusion = (
    pred_base
    .agg(
        F.sum(
            F.when(
                (F.col("prediction") == 1.0) &
                (F.col("target_emergencia_12m") == 1.0),
                1
            ).otherwise(0)
        ).alias("TP"),

        F.sum(
            F.when(
                (F.col("prediction") == 0.0) &
                (F.col("target_emergencia_12m") == 1.0),
                1
            ).otherwise(0)
        ).alias("FN"),

        F.sum(
            F.when(
                (F.col("prediction") == 1.0) &
                (F.col("target_emergencia_12m") == 0.0),
                1
            ).otherwise(0)
        ).alias("FP"),

        F.sum(
            F.when(
                (F.col("prediction") == 0.0) &
                (F.col("target_emergencia_12m") == 0.0),
                1
            ).otherwise(0)
        ).alias("TN")
    )
    .first()
)

TP = confusion["TP"]
FN = confusion["FN"]
FP = confusion["FP"]
TN = confusion["TN"]

accuracy = (TP + TN) / (TP + TN + FP + FN)

precision = (
    TP / (TP + FP)
    if (TP + FP) > 0 else 0
)

recall = (
    TP / (TP + FN)
    if (TP + FN) > 0 else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0 else 0
)

f2 = (
    5 * precision * recall / (4 * precision + recall)
    if (4 * precision + recall) > 0 else 0
)

pr_auc = BinaryClassificationEvaluator(
    labelCol="target_emergencia_12m",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
).evaluate(pred_base)

print("=== MATRIZ DE CONFUSIÓN ===")
print("TP:", TP)
print("FN:", FN)
print("FP:", FP)
print("TN:", TN)

print("\n=== MÉTRICAS MODELO BASE ===")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1       :", round(f1, 4))
print("F2       :", round(f2, 4))
print("PR-AUC   :", round(pr_auc, 4))